In [ ]:
!pip install pypdf
Chroma.from_documents

<bound method Chroma.from_documents of <class 'langchain_chroma.vectorstores.Chroma'>>

In [ ]:
import os
import re
import json
from pypdf import PdfReader
from google.colab import files

pdf_file = "Luật Các tổ chức tín dụng số 32-2024-QH15.pdf"
output_file = "chunks_luat_tctd_32_2024.json"

if not os.path.exists(pdf_file):
    print(f"❌ Chưa tìm thấy file '{pdf_file}'. Hãy kéo thả file vào thanh Files bên trái Colab!")
else:
    print(f"⏳ Đang đọc file PDF: {pdf_file}...")
    reader = PdfReader(pdf_file)
    print(f"✅ Đã tải {len(reader.pages)} trang.")

    # 1. Trích xuất text kèm đánh dấu mốc trang
    full_text = ""
    for idx, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        cleaned = re.sub(r'\r\n', '\n', text)
        full_text += f"\n[__PAGE_START_{idx + 1}__]\n" + cleaned

    # 2. Phân tích cấu trúc Chương & Điều
    chapter_regex = re.compile(r'(Chương\s+[IVXLCDM\d]+[^\n]*)', re.IGNORECASE)
    article_regex = re.compile(r'(Điều\s+(\d+)[\.\:]?\s*([^\n]*))', re.IGNORECASE)

    current_chapter = "Mở đầu / Căn cứ pháp lý"
    current_art_num = ""
    current_art_title = ""
    current_page = 1
    art_start_page = 1

    articles = []
    current_lines = []

    for line in full_text.split('\n'):
        stripped = line.strip()
        if not stripped:
            continue

        page_m = re.match(r'\[__PAGE_START_(\d+)__\]', stripped)
        if page_m:
            current_page = int(page_m.group(1))
            continue

        chap_m = chapter_regex.match(stripped)
        if chap_m:
            current_chapter = chap_m.group(1).strip()

        art_m = article_regex.match(stripped)
        if art_m:
            if current_lines:
                articles.append({
                    "chapter": current_chapter,
                    "article_num": current_art_num,
                    "article_title": current_art_title,
                    "text": "\n".join(current_lines),
                    "page_start": art_start_page,
                    "page_end": current_page
                })
                current_lines = []
            current_art_num = f"Điều {art_m.group(2)}"
            current_art_title = art_m.group(3).strip()
            art_start_page = current_page
            current_lines.append(stripped)
        else:
            current_lines.append(stripped)

    if current_lines:
        articles.append({
            "chapter": current_chapter,
            "article_num": current_art_num or "Lời nói đầu",
            "article_title": current_art_title or "",
            "text": "\n".join(current_lines),
            "page_start": art_start_page,
            "page_end": current_page
        })

    # 3. Cắt nhỏ văn bản (500 - 700 từ có overlap) và đính kèm Metadata
    final_chunks = []
    chunk_counter = 1

    def split_words(txt, max_w=700, over_w=100):
        words = txt.split()
        if len(words) <= max_w:
            return [txt]
        res, s = [], 0
        while s < len(words):
            e = min(s + max_w, len(words))
            res.append(" ".join(words[s:e]))
            if e >= len(words):
                break
            s += (max_w - over_w)
        return res

    for art in articles:
        sub_texts = split_words(art["text"])
        for idx, sub_txt in enumerate(sub_texts):
            chunk_data = {
                "id": f"TCTD_{art['article_num'].replace(' ', '_')}_part_{idx + 1}" if art['article_num'] else f"TCTD_chunk_{chunk_counter}",
                "content": f"Văn bản: Luật Các tổ chức tín dụng số 32/2024/QH15\n{art['chapter']}\n{art['article_num']}: {art['article_title']}\n\nNội dung:\n{sub_txt}",
                "metadata": {
                    "source": "Luật Các tổ chức tín dụng số 32/2024/QH15",
                    "doc_title": "Luật Các tổ chức tín dụng 2024",
                    "chapter": art["chapter"],
                    "article": art["article_num"],
                    "article_title": art["article_title"],
                    "page_start": art["page_start"],
                    "page_end": art["page_end"],
                    "part": idx + 1,
                    "total_parts": len(sub_texts)
                }
            }
            final_chunks.append(chunk_data)
            chunk_counter += 1

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(final_chunks, f, ensure_ascii=False, indent=2)

    print(f"\n🎉 XONG! Đã tạo thành công {len(final_chunks)} chunks kèm Metadata chuẩn.")
    print("📥 Đang tải file kết quả về máy tính...")
    files.download(output_file)

❌ Chưa tìm thấy file 'Luật Các tổ chức tín dụng số 32-2024-QH15.pdf'. Hãy kéo thả file vào thanh Files bên trái Colab!


In [ ]:
import os
import re
import json
import urllib.request
from pypdf import PdfReader
from google.colab import files

output_file = "chunks_luat_tctd_32_2024.json"

# 1. Tải bản PDF số hóa có sẵn text layer
pdf_url = "https://thanhtra.laichau.gov.vn/Files/ttt/Users/140/2024/Thang6/Luat-TCTD-so-32.pdf"
pdf_file = "Luat_TCTD_2024_searchable.pdf"

print("⏳ Đang tải bản PDF số hóa có sẵn văn bản...")
try:
    # Thiết lập User-Agent giả lập trình duyệt để tránh bị chặn chặn truy cập
    opener = urllib.request.build_opener()
    opener.addheaders = [('User-Agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')]
    urllib.request.install_opener(opener)
    urllib.request.urlretrieve(pdf_url, pdf_file)
    print("✅ Tải PDF thành công từ cổng thông tin!")
except Exception as e:
    print(f"❌ Không thể tự động tải xuống do lỗi kết nối máy chủ: {e}")
    print("💡 HƯỚNG DẪN: Hãy tải file PDF 'Luật Các tổ chức tín dụng 2024' về máy tính của bạn trước, sau đó kéo thả file đó vào mục 'Files' ở thanh bên trái của Google Colab và đổi tên file thành 'Luat_TCTD_2024_searchable.pdf' rồi chạy lại cell này.")

# 2. Đọc và kiểm tra văn bản
if not os.path.exists(pdf_file):
    print(f"⚠️ Hiện chưa có file '{pdf_file}' để xử lý. Vui lòng tải file lên Colab theo hướng dẫn phía trên.")
else:
    reader = PdfReader(pdf_file)
    print(f"📖 Tổng số trang: {len(reader.pages)}")

    full_text = ""
    for idx, page in enumerate(reader.pages):
        txt = page.extract_text() or ""
        full_text += f"\n[__PAGE_START_{idx + 1}__]\n" + txt

    sample_len = len(full_text.strip())
    print(f"📊 Tổng số ký tự trích xuất được: {sample_len} ký tự.")

    if sample_len < 1000:
        print("❌ Lỗi: File vẫn chưa có text layer (file scan không có chữ).")
    else:
        # 3. Phân tích cấu trúc Chương & Điều
        chapter_regex = re.compile(r'(Chương\s+[IVXLCDM\d]+[^\n]*)', re.IGNORECASE)
        article_regex = re.compile(r'(Điều\s+(\d+)[\.\:]?\s*([^\n]*))', re.IGNORECASE)

        current_chapter = "Mở đầu / Căn cứ pháp lý"
        current_art_num = ""
        current_art_title = ""
        current_page = 1
        art_start_page = 1

        articles = []
        current_lines = []

        for line in full_text.split('\n'):
            stripped = line.strip()
            if not stripped:
                continue

            page_m = re.match(r'\[__PAGE_START_(\d+)__\]', stripped)
            if page_m:
                current_page = int(page_m.group(1))
                continue

            chap_m = chapter_regex.match(stripped)
            if chap_m:
                current_chapter = chap_m.group(1).strip()

            art_m = article_regex.match(stripped)
            if art_m:
                if current_lines:
                    articles.append({
                        "chapter": current_chapter,
                        "article_num": current_art_num,
                        "article_title": current_art_title,
                        "text": "\n".join(current_lines),
                        "page_start": art_start_page,
                        "page_end": current_page
                    })
                    current_lines = []
                current_art_num = f"Điều {art_m.group(2)}"
                current_art_title = art_m.group(3).strip()
                art_start_page = current_page
                current_lines.append(stripped)
            else:
                current_lines.append(stripped)

        if current_lines:
            articles.append({
                "chapter": current_chapter,
                "article_num": current_art_num or "Lời nói đầu",
                "article_title": current_art_title or "",
                "text": "\n".join(current_lines),
                "page_start": art_start_page,
                "page_end": current_page
            })

        # 4. Cắt nhỏ văn bản (500 - 700 từ) có overlap và gắn Metadata
        def split_words(txt, max_w=700, over_w=100):
            words = txt.split()
            if len(words) <= max_w:
                return [txt]
            res, s = [], 0
            while s < len(words):
                e = min(s + max_w, len(words))
                res.append(" ".join(words[s:e]))
                if e >= len(words):
                    break
                s += (max_w - over_w)
            return res

        final_chunks = []
        chunk_counter = 1

        for art in articles:
            sub_texts = split_words(art["text"])
            for idx, sub_txt in enumerate(sub_texts):
                chunk_data = {
                    "id": f"TCTD_{art['article_num'].replace(' ', '_')}_part_{idx + 1}" if art['article_num'] else f"TCTD_chunk_{chunk_counter}",
                    "content": f"Văn bản: Luật Các tổ chức tín dụng số 32/2024/QH15\n{art['chapter']}\n{art['article_num']}: {art['article_title']}\n\nNội dung:\n{sub_txt}",
                    "metadata": {
                        "source": "Luật Các tổ chức tín dụng số 32/2024/QH15",
                        "doc_title": "Luật Các tổ chức tín dụng 2024",
                        "chapter": art["chapter"],
                        "article": art["article_num"],
                        "article_title": art["article_title"],
                        "page_start": art["page_start"],
                        "page_end": art["page_end"],
                        "part": idx + 1,
                        "total_parts": len(sub_texts)
                    }
                }
                final_chunks.append(chunk_data)
                chunk_counter += 1

        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(final_chunks, f, ensure_ascii=False, indent=2)

        print(f"\n🎉 THÀNH CÔNG! Đã tạo được {len(final_chunks)} đoạn văn bản luật kèm Metadata.")
        print("📥 Đang tải file JSON đầy đủ nội dung về máy tính của bạn...")
        files.download(output_file)

⏳ Đang tải bản PDF số hóa có sẵn văn bản...
✅ Tải PDF thành công từ cổng thông tin!
📖 Tổng số trang: 131
📊 Tổng số ký tự trích xuất được: 300895 ký tự.

🎉 THÀNH CÔNG! Đã tạo được 244 đoạn văn bản luật kèm Metadata.
📥 Đang tải file JSON đầy đủ nội dung về máy tính của bạn...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
"""Dữ liệu các chunk văn bản pháp lý (chuyển từ chunks_member_b.json).

Cách dùng:
    from chunks_member_b import CHUNKS
    for c in CHUNKS:
        print(c["id"], c["metadata"]["article"], c["content"][:80])
"""

CHUNKS = [{'id': 1,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Mở đầu',
               'article_title': 'Căn cứ ban hành',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Phần mở đầu / Phụ lục',
               'topics': 'Hạn mức & Giao dịch, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 1,
               'chunk_part': '1/1'},
  'content': 'VG P>: ký: CÔNG THÔNG TIN ĐIỆN TỬ CHÍNH PHỦ \n'
             'Email: thongtinchinhphuG) chinhphu.vn \n'
             'Cơ quan: VĂN PHÒNG CHÍNH PHÙ GHINHPHIIL VNThời gian ký: 20.05.2024 16:30:29 +07:00 '
             '/# CHINH PHÙ CỘNG HOÀ XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc \n'
             'Số: 52/2024/NĐ-CP Hà Nội, ngày 15 tháng 5 năm 2024 \n'
             'CỔNG THÔNG TTN ĐIỆN TỬ CHÍNH PHÙ \n'
             'NGHỊ ĐỊNH \n'
             'Căn cứ Luật Tổ chức Chính phủ ngày 19 tháng 6 năm 2015; Luật sửa đổi, bố sung một số '
             'điều của Luật Tổ chức Chính phủ và Luật Tổ chức chính quyên địa phương ngày 22 tháng '
             '11 năm 2010; \n'
             'Căn cứ Luật Ngân hàng Nhà nước Việt Nam ngày 16 tháng 6 năm 2010; Căn cứ Luật Các tổ '
             'chức tín dụng ngày 18 tháng 01 năm 2024, \n'
             'Căn cứ Luật Đầu tư ngày 17 tháng 6 năm 2020; \n'
             'Căn cứ Luật Ngân sách nhà nước ngày 25 tháng 6 năm 2015; \n'
             'Căn cứ Luật An ninh mạng ngày 12 tháng 6 năm 2016; \n'
             'Căn cứ Luật Phòng, chống rửa tiền ngày 15 tháng 11 năm 2022; \n'
             'Căn cứ Luật Phòng, chống khủng bó ngày 12 tháng 6 năm 2013; \n'
             'Căn cứ Luật Bưu chỉnh ngày 17 tháng 6 năm 2010; \n'
             'Theo đề nghị của Thống đốc Ngân hàng Nhà nước Việt Nam; \n'
             'Chính phủ ban hành Nghị định quy định về thanh toán không dùng tiền mặt.\n'
             '\n'
             'Chương Ï QUY ĐỊNH CHUNG'},
 {'id': 2,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 1',
               'article_title': 'Phạm vi điều chính',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 1. Phạm vi điều chính',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 1,
               'chunk_part': '1/1'},
  'content': 'Điều 1. Phạm vi điều chính \n'
             'Nghị định này quy định về hoạt động thanh toán không dùng tiền mặt, bao gôm: mở và '
             'sử dụng tài khoản thanh toán; dịch vụ thanh toán không dùng tiền mặt; dịch vụ trung '
             'gian thanh toán; tô chức, quản lý và giám sát các hệ thống thanh toán.'},
 {'id': 3,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 2',
               'article_title': 'Đối tượng áp dụng',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 2. Đối tượng áp dụng',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 2,
               'chunk_part': '1/1'},
  'content': 'Điều 2. Đối tượng áp dụng\n'
             '\n'
             '1. Tổ chức cung ứng dịch vụ thanh toán không dùng tiền mặt. \n'
             '\n'
             '2. Tổ chức cung ứng dịch vụ trung gian thanh toán.\n'
             '\n'
             '3. Tổ chức, cá nhân có liên quan đến hoạt động cung ứng dịch vụ thanh toán không '
             'dùng tiên mặt, dịch vụ trung gian thanh toán.\n'
             '\n'
             '4. Tổ chức, cá nhân sử dụng địch vụ thanh toán không dùng tiền mặt, địch vụ trung '
             'gian thanh toán (sau đây gọi là khách hàng).'},
 {'id': 4,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 3',
               'article_title': 'Giải thích từ ngữ Trong Nghị định này, các từ ngữ dưới đây được '
                                'hiểu như sau:',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 3. Giải thích từ ngữ Trong Nghị '
                                  'định này, các từ ngữ dưới đây được hiểu như sau:',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 3,
               'chunk_part': '1/7'},
  'content': 'Điều 3. Giải thích từ ngữ Trong Nghị định này, các từ ngữ dưới đây được hiểu như '
             'sau:'},
 {'id': 5,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 3',
               'article_title': 'Giải thích từ ngữ Trong Nghị định này, các từ ngữ dưới đây được '
                                'hiểu như sau:',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 3. Giải thích từ ngữ Trong Nghị '
                                  'định này, các từ ngữ dưới đây được hiểu như sau:',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 3,
               'chunk_part': '2/7'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 3. Giải thích từ ngữ Trong Nghị định này, các từ ngữ '
             'dưới đây được hiểu như sau: (tiếp theo)]\n'
             '1. Dịch vụ thanh toán không dùng tiền mặt (sau đây gọi là dịch vụ thanh toán) bao '
             'gồm: dịch vụ thanh toán qua tài khoản thanh toán của khách hàng và dịch vụ thanh '
             'toán không qua tài khoản thanh toán của khách hàng.'},
 {'id': 6,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 3',
               'article_title': 'Giải thích từ ngữ Trong Nghị định này, các từ ngữ dưới đây được '
                                'hiểu như sau:',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 3. Giải thích từ ngữ Trong Nghị '
                                  'định này, các từ ngữ dưới đây được hiểu như sau:',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 3,
               'chunk_part': '3/7'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 3. Giải thích từ ngữ Trong Nghị định này, các từ ngữ '
             'dưới đây được hiểu như sau: (tiếp theo)]\n'
             '2. Dịch vụ thanh toán qua tài khoản thanh toán của khách hàng là việc cung ứng '
             'phương tiện thanh toán; thực hiện dịch vụ thanh toán séc, lệnh chỉ, ủy nhiệm chỉ, '
             'nhờ thu, ủy nhiệm thu, thẻ ngân hàng, chuyển tiền, thu hộ, chỉ hộ và các dịch vụ '
             'thanh toán khác cho khách hàng thông qua tài khoản thanh toán của khách hàng.'},
 {'id': 7,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 3',
               'article_title': 'Giải thích từ ngữ Trong Nghị định này, các từ ngữ dưới đây được '
                                'hiểu như sau:',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 3. Giải thích từ ngữ Trong Nghị '
                                  'định này, các từ ngữ dưới đây được hiểu như sau:',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 3,
               'chunk_part': '4/7'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 3. Giải thích từ ngữ Trong Nghị định này, các từ ngữ '
             'dưới đây được hiểu như sau: (tiếp theo)]\n'
             '3. Dịch vụ thanh toán không qua tài khoản thanh toán của khách hàng là việc cung ứng '
             'dịch vụ thanh toán, thực hiện giao dịch thanh toán không thông qua tài khoản thanh '
             'toán của khách hàng.'},
 {'id': 8,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 3',
               'article_title': 'Giải thích từ ngữ Trong Nghị định này, các từ ngữ dưới đây được '
                                'hiểu như sau:',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 3. Giải thích từ ngữ Trong Nghị '
                                  'định này, các từ ngữ dưới đây được hiểu như sau:',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 3,
               'chunk_part': '5/7'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 3. Giải thích từ ngữ Trong Nghị định này, các từ ngữ '
             'dưới đây được hiểu như sau: (tiếp theo)]\n'
             '4. Tổ chức cung ứng dịch vụ thanh toán không dùng tiền mặt (sau đây gọi là tổ chức '
             'cung ứng dịch vụ thanh toán) là tổ chức được cung ứng một hoặc một số dịch vụ thanh '
             'toán theo quy định tại Nghị định này, bao gồm: Ngân hàng Nhà nước Việt Nam (sau đây '
             'gọi là Ngân hàng Nhà nước), ngân hàng, chỉ nhánh ngân hàng nước ngoài, quỹ tín dụng '
             'nhân dân, tổ chức tài chính vi mô và doanh nghiệp cung ứng dịch vụ bưu chính công '
             'ích.'},
 {'id': 9,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 3',
               'article_title': 'Giải thích từ ngữ Trong Nghị định này, các từ ngữ dưới đây được '
                                'hiểu như sau:',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 3. Giải thích từ ngữ Trong Nghị '
                                  'định này, các từ ngữ dưới đây được hiểu như sau:',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 3,
               'chunk_part': '6/7'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 3. Giải thích từ ngữ Trong Nghị định này, các từ ngữ '
             'dưới đây được hiểu như sau: (tiếp theo)]\n'
             '5. Tổ chức cung ứng dịch vụ trung gian thanh toán là tổ chức không phải là ngân '
             'hàng, chỉ nhánh ngân hàng nước ngoài được Ngân hàng Nhà nước câp Giây phép hoạt động '
             'cung ứng địch vụ trung gian thanh toán.\n'
             '\n'
             '6. Doanh nghiệp cung ứng dịch vụ bưu chính công ích là doanh nghiệp được chỉ định '
             'theo quy định của Luật Bưu chính.\n'
             '\n'
             '7. Giao dịch thanh toán không dùng tiền mặt (sau đây gọi là giao dịch thanh toán) là '
             'việc sử dụng dịch vụ thanh toán để thực hiện trả tiền hoặc chuyển tiền của tổ chức, '
             'cá nhân.\n'
             '\n'
             '8. Thanh toán quốc tế là giao dịch thanh toán được thực hiện cho một \n'
             'bên liên quan là tổ chức hoặc cá nhân có tài khoản thanh toán hoặc phương tiện thanh '
             'toán phát hành ở ngoài lãnh thô Việt Nam. \n'
             '\n'
             '9. Chủ tài khoản thanh toán là cá nhân đứng tên mở tài khoản đối với tài khoản của '
             'cá nhân hoặc là tổ chức mở tài khoản đối với tài khoản của tổ chức.\n'
             '\n'
             '10. Phương tiện thanh toán không dùng tiền mặt (sau đây gọi là phương tiện thanh '
             'toán) là phương tiện do tổ chức cung ứng dịch vụ thanh toán, công, ty tài chính được '
             'phép phát hành thẻ tín dụng, tổ chức cung ứng dịch vụ trung gian thanh toán cung ứng '
             'dịch vụ ví điện tử phát hành và được khách hàng sử dụng nhằm thực hiện giao địch '
             'thanh toán, bao gồm: séc, lệnh chi, ủy nhiệm chi, nhờ thu, ủy nhiệm thu, thẻ ngân '
             'hàng (bao gôm: thẻ ghi nợ, thẻ tín dụng, thẻ trả trước), ví điện tử và các phương '
             'tiện thanh toán khác theo quy định của Ngân hàng Nhà nước.\n'
             '\n'
             '11. Phương tiện thanh toán không hợp pháp là các phương tiện thanh toán không thuộc '
             'quy định tại khoản 10 Điều này.\n'
             '\n'
             '12. Tiền điện tử là giá trị tiền Việt Nam đồng lưu trữ trên các phương tiện điện tử '
             'được cung ứng trên cơ sở đối ứng với số tin được khách hàng trả trước cho ngân hàng, '
             'chỉ nhánh ngân hàng nước ngoài, tổ chức cung ứng dịch vụ trung gian thanh toán cung '
             'ứng dịch vụ ví điện tử.\n'
             '\n'
             '13. Dịch vụ chuyển mạch tài chính là dịch vụ cung ứng hạ tầng kỹ thuật kết nối, '
             'truyền dẫn và xử lý đữ liệu điện tử các giao dịch thanh toán trong, nước giữa các tổ '
             'chức cung ứng địch vụ thanh toán, công ty tài chính được phép phát hành thẻ tín '
             'dụng, tổ chức cung ứng dịch vụ trung gian thanh toán.\n'
             '\n'
             '14. Dịch vụ chuyển mạch tài chính quốc tế là việc kết nối với hệ thống thanh toán '
             'quốc tế để truyền dẫn và xử lý dữ liệu điện tử các giao dịch thanh toán quốc tê.\n'
             '\n'
             '15. Dịch vụ bù trừ điện tử là dịch vụ cung ứng hạ tằng kỹ thuật, nghiệp vụ để thực '
             'hiện việc tiếp nhận, đôi chiêu dữ liệu thanh toán và tính toán kết quả số tiền phải '
             'thu, phải trả sau khi bù trừ giữa các bên thành viên tham gia để thực hiện việc '
             'quyết toán cho các bên có liên quan.\n'
             '\n'
             '16. Dịch vụ ví điện tử là dịch vụ do ngân hàng, chi nhánh ngân hàng nước ngoài, tổ '
             'chức cung ứng dịch vụ trung gian thanh toán cung ứng cho khách hàng để nạp tiền vào '
             'ví điện tử, rút tiền ra khỏi ví điện tử và thực hiện giao dịch thanh toán.'},
 {'id': 10,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 3',
               'article_title': 'Giải thích từ ngữ Trong Nghị định này, các từ ngữ dưới đây được '
                                'hiểu như sau:',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 3. Giải thích từ ngữ Trong Nghị '
                                  'định này, các từ ngữ dưới đây được hiểu như sau:',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 3,
               'chunk_part': '7/7'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 3. Giải thích từ ngữ Trong Nghị định này, các từ ngữ '
             'dưới đây được hiểu như sau: (tiếp theo)]\n'
             '16. Dịch vụ ví điện tử là dịch vụ do ngân hàng, chi nhánh ngân hàng nước ngoài, tổ '
             'chức cung ứng dịch vụ trung gian thanh toán cung ứng cho khách hàng để nạp tiền vào '
             'ví điện tử, rút tiền ra khỏi ví điện tử và thực hiện giao dịch thanh toán.\n'
             '\n'
             '17. Dịch vụ hỗ trợ thu hộ, chỉ hộ là việc tiếp nhận, xử lý dữ liệu điện tử, tính '
             'toán kết quả thu hộ, chỉ hộ, hủy việc thu hộ, chị hộ cho khách hàng có tài khoản '
             'thanh toán, thẻ ngân hàng và thực hiện thanh toán cho các bên có liên quan.\n'
             '\n'
             '18. Dịch vụ cổng thanh toán điện tử là dịch vụ cung ứng hạ tầng kỹ thuật đề kết nối, '
             'truyền dẫn và xử lý dữ liệu điện tử các giao dịch thanh toán được thực hiện bằng '
             'phương tiện thanh toán giữa khách hàng, đơn vị chấp nhận thanh toán với ngân hàng, '
             'chi nhánh ngân hàng nước ngoài, công ty tài chính được phép phát hành thẻ tín dụng, '
             'tổ chức cung ứng dịch vụ trung gian thanh toán. \n'
             '\n'
             '19. Hệ thống thanh toán là hệ thống bao gồm các quy định, phương tiện, quy trình, '
             'thủ tục, cơ sở hạ tằng kỹ thuật để xử lý, chuyên mạch, bù trừ, quyết toán các giao '
             'dịch thanh toán.\n'
             '\n'
             '20. Hệ thống thanh toán quốc tế là hệ thống thanh toán được thành lập ở nước ngoài '
             'và cho phép thực hiện các giao dịch thanh toán quốc tế.\n'
             '\n'
             '21. Hệ thống thanh toán quan trọng là hệ thống thanh toán có vai trò chủ đạo trong '
             'việc phục vụ nhu câu thanh toán của các chủ thể trong nền kinh tế, có khả năng phát '
             'sinh rủi ro hệ thống, đáp ứng ít nhất một trong các tiêu chí sau:\n'
             '\n'
             'a) Là hệ thống thanh toán duy nhất hoặc chiếm tỷ trọng lớn so với tổng giá trị thanh '
             'toán của các hệ thống thanh toán cùng loại; hoặc\n'
             '\n'
             'b) Là hệ thống xử lý các giao dịch thanh toán giá trị cao; hoặc\n'
             '\n'
             'c) Là hệ thống được sử dụng để quyết toán cho các hệ thông thanh toán khác hoặc cho '
             'các giao dịch trên thị trường tài chính.\n'
             '\n'
             '22. Rủi ro hệ thống là rủi ro mà một thành viên tham gia hệ thống thanh toán không '
             'có khả năng thực hiện các nghĩa vụ tài chính trong hệ thống thanh toán khi đến hạn '
             'dẫn đến việc các thành viên tham gia khác cũng không thể thực hiện nghĩa vụ tài '
             'chính khi đến hạn, có lan truyền rủi ro đến các hệ thống thanh toán khác.\n'
             '\n'
             '23. Tài khoản thanh toán chung là tài khoản thanh toán cớ ít nhất hai chủ thê trở '
             'lên cùng đứng tên mở tài khoản.'},
 {'id': 11,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 4',
               'article_title': 'Trách nhiệm quản lý nhà nước của Ngân hàng Nhà nước về hoạt động '
                                'thanh toán không đùng tiên mặt',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 4. Trách nhiệm quản lý nhà nước '
                                  'của Ngân hàng Nhà nước về hoạt động thanh toán không đùng tiên '
                                  'mặt',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 5,
               'chunk_part': '1/1'},
  'content': 'Điều 4. Trách nhiệm quản lý nhà nước của Ngân hàng Nhà nước về hoạt động thanh toán '
             'không đùng tiên mặt\n'
             '\n'
             '1. Ban hành theo thâm quyền hoặc trình cơ quan có thâm quyền ban hành các văn bản '
             'quy phạm pháp luật về hoạt động thanh toán không dùng tiền mặt; quy định việc quản '
             'lý, kết nối, chia sẻ đữ liệu phục vụ hoạt động thanh toán không dùng tiền mặt.\n'
             '\n'
             '2. Tổ chức, quản lý, vận hành, giám sát hệ thống thanh toán quốc gia; tham gia tổ '
             'chức, giám sát sự vận hành của các hệ thống thanh toán quan trọng khác trong nền '
             'kinh tế; giám sát hoạt động cung ứng dịch vụ thanh toán và dịch vụ trung gian thanh '
             'toán.\n'
             '\n'
             '3. Chấp thuận bằng văn bản việc tham gia hệ thông thanh toán quốc tế của ngân hàng '
             'thương mại, chỉ nhánh ngân hàng nước ngoài.\n'
             '\n'
             '4. Cấp, sửa đối, bỗ sung và thu hồi Giấy phép hoạt động cung ứng dịch vụ trung gian '
             'thanh toán của tổ chức cung ứng địch vụ trung gian thanh toán. \n'
             '\n'
             '5. Chấp thuận và thu hồi văn bản hoạt động cung ứng dịch vụ thanh toán không qua tài '
             'khoản thanh toán của khách hàng của doanh nghiệp cung ứng địch vụ bưu chính công '
             'ích.\n'
             '\n'
             '6. Kiểm tra, thanh tra và xử lý theo thâm quyền đối với các hành vị vi phạm pháp '
             'luật hoạt động thanh toán không dùng tiên mặt của các tổ chức, cá nhân. \n'
             '._7. Quản lý hoạt động hợp tác quốc tế trong lĩnh vực thanh toán; chủ trì, phôi hợp '
             'với các cơ quan chức năng trong việc quản lý hoạt động thanh toán quôc tê.'},
 {'id': 12,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 5',
               'article_title': 'Thanh toán bằng ngoại tệ và thanh toán quốc tế',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 5. Thanh toán bằng ngoại tệ và '
                                  'thanh toán quốc tế',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '1/1'},
  'content': 'Điều 5. Thanh toán bằng ngoại tệ và thanh toán quốc tế\n'
             '\n'
             '1. Thanh toán bằng ngoại tệ và thanh toán quốc tế phải tuân theo các quy định của '
             'Nghị định này, pháp luật về quản lý ngoại hối, bảo vệ dữ liệu người dùng, an ninh '
             'mạng, quản lý thuế, pháp luật về phòng, chống rửa tiền, tài trợ khủng bố, tài trợ '
             'phổ. biến vũ khí hủy điệt hàng loạt và các điều ước quốc tế, thỏa thuận quốc. tế về '
             'thanh toán mà Việt Nam tham gia. Việc áp dụng tập quán thương mại thực hiện theo '
             'Điều 3 Luật Các tổ chức tín dụng.\n'
             '\n'
             '2. Ngân hàng thương mại, chỉ nhánh ngân hàng nước ngoài được tham gia hệ thống thanh '
             'toán quốc tế sau khi đáp ứng các điều kiện quy định tại\n'
             'Điều 21 Nghị định này. \n'
             '3, Tổ chức nước ngoài cung ứng dịch vụ thanh toán, dịch vụ trung gian thanh toán cho '
             'khách hàng là người không cư trú và người nước ngoài cư trú tại Việt Nam để thực '
             'hiện giao dịch thanh toán bàng hóa, dịch vụ tại Việt Nam phải thực hiện thông qua '
             'ngân hàng thương mại, chỉ nhánh ngân hàng nước ngoài đã được Ngân hàng Nhà nước chấp '
             'thuận tham gia hệ thông thanh toán quốc tế của tô chức nước ngoài đó.\n'
             '\n'
             '4. Tổ chức cung ứng dịch vụ chuyển mạch tài chính được kết nối với hệ thống thanh '
             'toán quôc. tế để thực hiện dịch vụ chuyển mạch tài chính quốc tế sau khi đáp ứng các '
             'điều kiện quy định tại Điều 22 Nghị định này.\n'
             '\n'
             '5. Tổ chức cung ứng dịch vụ trung gian thanh toán (trừ tổ chức cung ứng dịch vụ '
             'chuyển mạch tài chính) được cung ứng dịch vụ trung gian thanh toán cho khách hàng để '
             'thực hiện giao dịch thanh toán cho hàng hóa, dịch vụ nước ngoài; việc thực hiện '
             'thanh toán, quyết toán cho các giao dịch thanh toán quốc tế đó phải được thực hiện '
             'thông qua ngân hàng thương mại, chỉ nhánh ngân hàng nước ngoài đã được Ngân hàng Nhà '
             'nước chấp thuận hoạt động ngoại hối trên thị trường quốc tế.\n'
             '\n'
             '6. Các bên liên quan đến hoạt động thanh toán quốc tế có trách nhiệm cung cấp thông '
             'tin đầy đủ, chính xác, kịp thời và đáp ứng các yêu cầu của cơ quan quản lý nhà nước '
             'theo quy định của pháp luật Việt Nam.'},
 {'id': 13,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 6',
               'article_title': 'Ví điện tử, thẻ trả trước',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 6. Ví điện tử, thẻ trả trước',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '1/1'},
  'content': 'Điều 6. Ví điện tử, thẻ trả trước\n'
             '1. Ví điện tử, thẻ trả trước là phương tiện lưu trữ tiền điện tử.\n'
             '\n'
             '2. Ngân hàng, chỉ nhánh ngân hàng nước ngoài được phát hành, cung ứng ví điện tử, '
             'thẻ trả trước. Việc cung ứng, phát hành và sử dụng ví điện tử, thẻ trả trước thực '
             'hiện theo quy định của Ngân hàng Nhà nước.\n'
             '\n'
             '3. Tổ chức cung ứng dịch vụ trung gian thanh toán cung ứng dịch vụ ví điện tử phải '
             'đảm bảo duy trì tổng số dư trên tất cả các tài khoán đảm bảo thanh toán cho dịch vụ '
             'ví điện tử mở tại ngân hàng, chi nhánh ngân hàng nước ngoài không thấp hơn tổng số '
             'dự tất cả các ví điện tử đã phát hành cho khách hàng: chỉ cho phép sử dụng dịch vụ '
             'đối với các ví điện tử có liên kết với tài khoản thanh toán, thẻ ghi nợ của chính '
             'khách hàng.'},
 {'id': 14,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 7',
               'article_title': 'Tổ chức, quản lý và vận hành hệ thống thanh toán quốc gia',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 7. Tổ chức, quản lý và vận hành '
                                  'hệ thống thanh toán quốc gia',
               'topics': 'Hạn mức & Giao dịch, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '1/1'},
  'content': 'Điều 7. Tổ chức, quản lý và vận hành hệ thống thanh toán quốc gia\n'
             '\n'
             '1. Ngân hàng Nhà nước tổ chức, quản lý và vận hành hệ thống thanh toán quốc gia để '
             'cung ứng dịch vụ thanh toán cho các thành viên tham gia hệ thống là Ngân hàng Nhà '
             'nước, các tổ chức tín dụng, chi nhánh ngân hàng nước ngoài, Kho bạc Nhà nước; thực '
             'hiện quyết toán kết quả bù trừ cho các hệ thống thanh toán khác.\n'
             '\n'
             '2. Ngân hàng Nhà nước quy định việc quản lý, vận hành, đảm bảo an toàn hoạt động của '
             'hệ thống thanh toán quốc gia.'},
 {'id': 15,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 8',
               'article_title': 'Các hành vi bị cắm',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 8. Các hành vi bị cắm',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 7,
               'chunk_part': '1/1'},
  'content': 'Điều 8. Các hành vi bị cắm\n'
             '\n'
             '1. Sửa chữa, tây xóa phương tiện thanh toán, chứng từ thanh toán không đúng quy định '
             'pháp luật; làm giả phương tiện thanh toán, chứng từ thanh toán; lưu giữ, lưu hành, '
             'chuyền nhượng, sử dụng phương tiện thanh toán giả.\n'
             '\n'
             '2. Xâm nhập hoặc tìm cách xâm nhập, đánh cắp dữ liệu, phá hoại, làm thay đôi trái '
             'phép chương trình phần mêm, dữ liệu điện tử sử dụng trong thanh toán; lợi dụng lỗi '
             'hệ thông mạng máy tính đê trục lợi.\n'
             '\n'
             '3. Cung cấp không trung thực thông tin có liên quan đến việc cung ứng hoặc sử dụng '
             'dịch vụ thanh toán, dịch vụ trung gian thanh toán.\n'
             '\n'
             '4. Tiết lộ, cung cấp thông tin về số dư trên tài khoản thanh toán, số dự thẻ ngân '
             'hàng, số dư ví điện tử và các giao dịch thanh toán của khách hàng tại tổ chức cung '
             'ứng dịch vụ thanh toán, tổ chức cung ứng dịch vụ trung gian thanh toán không đúng '
             'theo quy định của pháp luật có liên quan.\n'
             '\n'
             '5. Mở hoặc duy trì tài khoản thanh toán, ví điện tử nặc danh, mạo danh; mua, bán, '
             'thuê, cho thuê, mượn, cho mượn tài khoản thanh toán, ví điện tử; thuê, cho thuê, '
             'mua, bán, mở hộ thẻ ngân hàng (trừ trường hợp thẻ trả trước vô danh); lấy cắp, thông '
             'đồng để lấy cắp, mua, bán thông tin tài khoản thanh toán, thông tin thẻ ngân hàng, '
             'thông tin ví điện tử. \n'
             '\n'
             '6. Phát hành, cung ứng và sử dụng các phương tiện thanh toán không hợp pháp.\n'
             '\n'
             '7. Thực hiện cung ứng dịch vụ trung gian thanh toán khi chưa được Ngân hàng Nhà nước '
             'cấp Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán. Thực hiện cung ứng '
             'dịch vụ thanh toán mà không phải là tổ chức cung ứng dịch vụ thanh toán. \n'
             '§. Thực hiện, tổ chức thực hiện hoặc tạo điều kiện thực hiện các hành vi: sử dụng, '
             'lợi dụng tài khoản thanh toán, phương tiện thanh toán, dịch vụ thanh toán, dịch vụ '
             'trung gian thanh toán để đánh bạc, tổ chức đánh bạc, gian lận, lừa đảo, kinh doanh '
             'trái pháp luật và thực hiện các hành vi vi phạm pháp luật khác. \n'
             '9, Tẩy xóa, thay đổi nội dung, mua, bán, chuyển nhượng, cho thuê, cho mượn, làm giả '
             'Giây phép hoạt động cung ứng dịch vụ trung gian thanh toán.\n'
             '\n'
             '10. Ủy thác, giao đại lý cho tổ chức, cá nhân khác thực hiện hoạt động được phép '
             'theo Giây phép hoạt động cung ứng dịch vụ trung gian thanh toán.\n'
             '\n'
             '11. Gian lận, giả mạo các giấy tờ chứng minh đủ điều kiện để được cấp Giấy phép hoạt '
             'động cung ứng dịch vụ trung gian thanh toán trong hồ sơ đề nghị cấp Giấy phép.\n'
             '\n'
             '12. Hoạt động không đúng nội dung quy định trong Giấy phép hoạt động cung ứng dịch '
             'vụ trung gian thanh toán.\n'
             '\n'
             '13. Chủ tài khoản thanh toán có tài khoản thanh toán tại tổ chức cung ứng dịch vụ '
             'thanh toán nhưng cung cấp thông tin hoặc cam kết không có tài khoản thanh toán tại '
             'tổ chức cung ứng dịch vụ thanh toán cho các bên có quyền, nghĩa vụ liên quan theo '
             'quy định của pháp luật về giải ngân vốn cho vay của tổ chức tín dụng, chỉ nhánh ngân '
             'hàng nước ngoài. \n'
             '¬ Chương H - „ MỞ VÀ SỬ DỰNG TÀI KHOẢN THANH TOÁN\n'
             '\n'
             'Mục 1 QUY ĐỊNH CHUNG'},
 {'id': 16,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 9',
               'article_title': 'Mỡ và sử dụng tài khoản thanh toán',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 9. Mỡ và sử dụng tài khoản thanh '
                                  'toán',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 7,
               'chunk_part': '1/1'},
  'content': 'Điều 9. Mỡ và sử dụng tài khoản thanh toán \n'
             'Việc mở và sử dụng tài khoản thanh toán của khách hàng tại tổ chức cung ứng dịch vụ '
             'thanh toán thực hiện theo quy định của Ngân hàng Nhà nước và quy định của pháp luật '
             'liên quan.'},
 {'id': 17,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 10',
               'article_title': 'Sử dụng và ủy quyền sử dụng tài khoản thanh toán',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 10. Sử dụng và ủy quyền sử dụng '
                                  'tài khoản thanh toán',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 8,
               'chunk_part': '1/1'},
  'content': 'Điều 10. Sử dụng và ủy quyền sử dụng tài khoản thanh toán\n'
             '\n'
             '1. Chủ tài khoản thanh toán được sử dụng tài khoản thanh toán của mình đề nộp, rút '
             'tiền mặt và yêu cầu tổ chức cung ứng dịch vụ thanh toán thực hiện các giao dịch '
             'thanh toán hợp lệ. Chủ tài khoản thanh toán có quyền yêu cầu tổ \n'
             ' chức cung ứng dịch vụ thanh toán cung cấp thông tin về giao dịch và số dự trên tài '
             'khoản thanh toán của mình theo thỏa thuận với tổ chức cung ứng dịch vụ thanh toán '
             'nơi mở tài khoản thanh toán.\n'
             '\n'
             '2. Chủ tài khoản thanh toán được ủy quyền trong sử dụng tài khoản thanh toán. "Việc '
             'ủy quyền phải thực hiện bằng văn bản, phù hợp với quy định pháp luật về ủy quyền.\n'
             '\n'
             '3. Chủ tài khoản thanh toán có nghĩa vụ cung cấp thông tin đầy đủ, trung thực và '
             'tuân thủ các quy định về mở, sử dụng, ủy quyền trong sử dụng tài khoản thanh toán '
             'của tổ chức cung ứng dịch vụ thanh toán và phải đảm bảo có đủ tiền (số dư Có) trên '
             'tài khoản thanh toán để thực hiện lệnh thanh toán đã lập trừ trường hợp có thỏa '
             'thuận cho vay thấu chỉ với tổ chức cung ứng dịch vụ thanh toán.\n'
             '\n'
             '4. Tổ chức cung ứng dịch vụ thanh toán có nghĩa vụ thực hiện đầy đủ, kịp thời lệnh '
             'thanh toán hợp lệ của chủ tài khoản thanh toán.\n'
             '\n'
             '5. Tổ chức cung ứng dịch vụ thanh toán có quyền từ chối thực hiện lệnh thanh toán '
             'của chủ tài khoản thanh toán khi lệnh thanh toán không hợp lệ hoặc có cơ sở pháp lý '
             'để xác định chủ tài khoản vi phạm các hành vi bị cấm theo quy định tại Điều 8 Nghị '
             'định này hoặc khi tài khoản thanh toán không đủ tiền trừ trường hợp có thỏa thuận '
             'khác. Trường hợp từ chối thực hiện lệnh thanh toán của chủ tài khoản thanh toán, tổ '
             'chức cung ứng dịch vụ thanh toán phải thông báo lý do từ chối cho chủ tài khoản '
             'thanh toán.'},
 {'id': 18,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 11',
               'article_title': 'Phong tôa tài khoản thanh toán',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 11. Phong tôa tài khoản thanh '
                                  'toán',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 9,
               'chunk_part': '1/1'},
  'content': 'Điều 11. Phong tôa tài khoản thanh toán\n'
             '\n'
             '1. Tài khoản thanh toán bị phong toả một phần hoặc toàn bộ số dư trên tài khoản '
             'thanh toán trong các trường hợp sau:\n'
             '\n'
             'a) Theo thỏa thuận trước giữa chủ tài khoản thanh toán và tổ chức cung ứng dịch vụ '
             'thanh toán hoặc theo yêu câu của chủ tài khoản;\n'
             '\n'
             'b) Khi có quyết định hoặc yêu cầu bằng văn bản của cơ quan có thâm quyền theo quy '
             'định của pháp luật;\n'
             '\n'
             'c) Khi tổ chức cung ứng dịch vụ thanh toán phát hiện có nhằm lẫn, sai sót khi ph! Có '
             'nhằm vào tài khoản thanh toán của khách hàng hoặc thực hiện theo yêu cầu hoàn trả '
             'lại tiền của tổ chức cung ứng dịch vụ thanh toán chuyển tiền do có nhằm lẫn, sai sót '
             'so với lệnh thanh toán của bên chuyển tiền sau khi phi Có vào tài khoản thanh toán '
             'của khách hàng. Số tiền bị phong tỏa trên tài khoản thanh toán không được vượt quá '
             'số tiền bị nhằm lẫn, sai sót; \n'
             '\n'
             'd) Khi có yêu cầu phong tỏa của một trong các chủ tài khoản thanh toán chung trừ '
             'trường hợp có thỏa thuận trước bằng văn bản giữa tổ chức cung ứng dịch vụ thanh toán '
             'và các chủ tài khoản thanh toán chung.\n'
             '\n'
             '2. Việc chấm dứt phong toả tài khoản thanh toán được thực hiện: :\n'
             '\n'
             'a) Theo thỏa thuận bằng văn bản giữa chủ tài khoản thanh toán và tổ chức cung ứng '
             'dịch vụ thanh toán;\n'
             '\n'
             'b) Khi có quyết định chấm dứt phong tỏa của cơ quan có thẩm quyền theo quy định của '
             'pháp luật; \n'
             '©) Đã xử lý xong sai sót, nhằm lẫn trong thanh toán chuyển tiền quy định tại điểm c '
             'khoản 1 Điều này;\n'
             '\n'
             'd) Khi có yêu cầu chấm dứt phong tỏa của tất cả các chủ tài khoản thanh toán chung '
             'hoặc theo thỏa thuận trước bằng văn bản giữa tổ chức cung ứng địch vụ thanh toán và '
             'các chủ tải khoản thanh toán chung.\n'
             '\n'
             '3. Tổ chức cung ứng dịch vụ thanh toán, chủ tài khoản thanh toán và cơ quan có thâm '
             'quyền nêu thực hiện hoặc yêu cầu thực hiện phong tỏa tài khoản thanh toán trái pháp '
             'luật gây thiệt hại cho chủ tài khoản thanh toán thì chịu trách nhiệm bồi thường theo '
             'quy định của pháp luật.'},
 {'id': 19,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 12',
               'article_title': 'Đóng tài khoản thanh toán',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 12. Đóng tài khoản thanh toán',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 10,
               'chunk_part': '1/1'},
  'content': 'Điều 12. Đóng tài khoản thanh toán\n'
             '1. Việc đóng tài khoản thanh toán được thực hiện khi:\n'
             '\n'
             'a) Chủ tài khoản thanh toán có yêu cầu và đã thực hiện đầy đủ các nghĩa vụ liên quan '
             'đến tài khoản thanh toán;\n'
             '\n'
             'b) Chủ tài khoản thanh toán là cá nhân bị chết, bị tuyên bố đã chết;\n'
             '\n'
             'c) Tổ chức có tài khoản thanh toán chấm dứt hoạt động theo quy định của pháp luật;\n'
             '\n'
             'd) Chủ tài khoản thanh toán vi phạm hành vi bị cấm về tài khoản thanh toán quy định '
             'tại khoản 5, khoản 8§ Điều 8 Nghị định này. \n'
             'đ) Các trường hợp theo thỏa thuận trước bằng văn bản giữa chủ tài khoản thanh toán '
             'với tô chức cung ứng dịch vụ thanh toán;\n'
             '\n'
             'e) Các trường hợp khác theo quy định của pháp luật.\n'
             '2. Xử lý số dư khi đóng tài khoản thanh toán:\n'
             '\n'
             'a) Chị trả theo yêu cầu của chủ tài khoản thanh toán hoặc được thực hiện theo thỏa '
             'thuận trước giữa chủ tài khoản thanh toán và tổ chức cung ứng dịch vụ thanh toán; '
             'trường hợp chủ tài khoản là người mất năng lực hành vi đân \n'
             ' sự, người có khó khăn trong nhận thức, làm chủ hành vi, người bị hạn chế năng lực '
             'hành vi dân sự, việc chỉ trả thực hiện theo yêu câu người đại diện theo pháp luật, '
             'người giám hộ phù hợp với quy định pháp luật dân sự; hoặc chi trả cho người thừa kế, '
             'đại diện thừa kế hợp pháp trong trường hợp chủ tải khoản thanh toán là cá nhân bị '
             'chết, bị tuyên bố đã chết;\n'
             '\n'
             'b) Chỉ trả theo quyết định của cơ quan có thấm quyền theo quy định của pháp luật;\n'
             '\n'
             'c) Xử lý theo quy định của pháp luật đối với trường hợp người thụ hưởng hợp pháp số '
             'dư trên tài khoản thanh toán đã được thông báo mà không đên nhận.\n'
             '\n'
             'Mục 2 MỞ VÀ SỬ DỤNG TÀI KHOẢN THANH TOÁN CỦA NGÂN HÀNG NHÀ NƯỚC'},
 {'id': 20,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 13',
               'article_title': 'Mỡ và sử dụng tài khoản thanh toán của Ngân hàng Nhà nước',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 13. Mỡ và sử dụng tài khoản '
                                  'thanh toán của Ngân hàng Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 10,
               'chunk_part': '1/1'},
  'content': 'Điều 13. Mỡ và sử dụng tài khoản thanh toán của Ngân hàng Nhà nước\n'
             '\n'
             '1. Ngân hàng Nhà nước mở tài khoản thanh toán cho Kho bạc Nhà nước, các tô chức tín '
             'dụng, chỉ nhánh ngân hàng nước ngoài theo quy định tại Luật Ngân hàng Nhà nước và '
             'Luật Các tô chức tín dụng.\n'
             '\n'
             '2. Ngân hàng Nhà nước mở tài khoản cho ngân hàng trung ương các nước, ngân hàng nước '
             'ngoài, tổ chức tiền tệ quốc tế, ngân hàng quốc tế theo các điều ước và thỏa thuận '
             'quốc tế mà Việt Nam là thành viên tham gia. Trong trường hợp Việt Nam chưa phải là '
             'thành viên tham gia, việc mở tài khoản thanh toán thực hiện theo quyết định của Thủ '
             'tướng Chính phủ.\n'
             '\n'
             '3. Ngân hàng Nhà nước mở tài khoản thanh toán và thực hiện giao dịch trên tài khoản, '
             'thanh toán ở ngân hàng nước ngoài, tổ chức tiền tệ quốc tế, ngân hàng quốc tế. Ngân '
             'hàng Nhà nước mở tài khoản thanh toán tại ngân hàng trung ương các nước, mở tài '
             'khoản thanh toán và thực hiện giao dịch thanh toán ở nước ngoài theo các điều ước và '
             'thỏa thuận quốc tế mà Việt Nam là thành viên tham gia.'},
 {'id': 21,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 14',
               'article_title': 'Hồ sơ và trình tự thủ tục mở, đóng tài khoản thanh toán tại Ngân '
                                'hàng Nhà nước của tô chức tín dụng, chỉ nhánh ngân hàng nước '
                                'ngoài, Kho bạc Nhà nước',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 14. Hồ sơ và trình tự thủ tục '
                                  'mở, đóng tài khoản thanh toán tại Ngân hàng Nhà nước của tô '
                                  'chức tín dụng, chỉ nhánh ngân hàng nước ngoài, Kho bạc Nhà nước',
               'topics': 'eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 11,
               'chunk_part': '1/5'},
  'content': 'Điều 14. Hồ sơ và trình tự thủ tục mở, đóng tài khoản thanh toán tại Ngân hàng Nhà '
             'nước của tô chức tín dụng, chỉ nhánh ngân hàng nước ngoài, Kho bạc Nhà nước'},
 {'id': 22,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 14',
               'article_title': 'Hồ sơ và trình tự thủ tục mở, đóng tài khoản thanh toán tại Ngân '
                                'hàng Nhà nước của tô chức tín dụng, chỉ nhánh ngân hàng nước '
                                'ngoài, Kho bạc Nhà nước',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 14. Hồ sơ và trình tự thủ tục '
                                  'mở, đóng tài khoản thanh toán tại Ngân hàng Nhà nước của tô '
                                  'chức tín dụng, chỉ nhánh ngân hàng nước ngoài, Kho bạc Nhà nước',
               'topics': 'eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 11,
               'chunk_part': '2/5'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 14. Hồ sơ và trình tự thủ tục mở, đóng tài khoản '
             'thanh toán tại Ngân hàng Nhà nước của tô chức tín dụng, chỉ nhánh ngân hàng nước '
             'ngoài, Kho bạc Nhà nước (tiếp theo)]\n'
             '1. Nguyên tắc lập và gửi hồ sơ:\n'
             '\n'
             'a) Hồ sơ phải được lập bằng tiếng Việt. Nếu giấy tờ trong hồ Sơ mở tài khoản thanh '
             'toán bằng tiếng nước ngoài thì phải được dịch sang tiếng Việt và được công chứng, '
             'chứng thực theo quy định của pháp luật; \n'
             ' l1\n'
             '\n'
             'b) Các bản sao giấy tờ, tài liệu trong hồ sơ mở, đóng tài khoản thanh toán phải là '
             'bán sao có chứng thực hoặc bản sao cấp từ số gốc hoặc bản sao kèm bản chính để đối '
             'chiếu theo quy định của pháp luật, trường hợp hồ sơ gửi trực tuyến thì thực hiện '
             'theo quy định về thủ tục hành chính trên môi trường điện tử; : \n'
             '©) Hồ sơ được gửi qua đường bưu điện (dịch vụ bưu chính) hoặc nộp trực tiếp tới Bộ '
             'phận Một cửa Ngân hàng Nhà nước nơi tổ chức đề nghị mở tài khoản thanh toán hoặc '
             'trực tuyến tại Cổng dịch vụ công Ngân hàng Nhà nước hoặc Cổng dịch vụ công quốc '
             'gia;\n'
             '\n'
             'd) Tổ chức đề nghị mở, đóng tài khoản thanh toán hoàn toàn chịu trách nhiệm trước '
             'pháp luật về tính chính xác, trung thực của các thông tin cung cấp.'},
 {'id': 23,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 14',
               'article_title': 'Hồ sơ và trình tự thủ tục mở, đóng tài khoản thanh toán tại Ngân '
                                'hàng Nhà nước của tô chức tín dụng, chỉ nhánh ngân hàng nước '
                                'ngoài, Kho bạc Nhà nước',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 14. Hồ sơ và trình tự thủ tục '
                                  'mở, đóng tài khoản thanh toán tại Ngân hàng Nhà nước của tô '
                                  'chức tín dụng, chỉ nhánh ngân hàng nước ngoài, Kho bạc Nhà nước',
               'topics': 'eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 11,
               'chunk_part': '3/5'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 14. Hồ sơ và trình tự thủ tục mở, đóng tài khoản '
             'thanh toán tại Ngân hàng Nhà nước của tô chức tín dụng, chỉ nhánh ngân hàng nước '
             'ngoài, Kho bạc Nhà nước (tiếp theo)]\n'
             '2. Hồ sơ mở tài khoản thanh toán tại Ngân hàng Nhà nước, bao gồm:\n'
             '\n'
             'a) Đơn đề nghị mở tài khoản thanh toán kèm bản đăng ký mẫu dấu, mẫu chữ ký theo Mẫu '
             'số 01 ban bành kèm theo Nghị định này do người đại diện theo pháp luật hoặc người '
             'đại diện theo ủy quyên của tổ chức mở tài khoản thanh toán ký tên, đóng dấu;\n'
             '\n'
             'b) Các tài liệu chứng minh tổ chức mở tài khoản thanh toán được thành lập và hoạt '
             'động hợp pháp, gồm: quyết định thành lập, giấy phép hoạt động, giây chứng nhận đăng '
             'kỹ doanh nghiệp, giấy chứng nhận đăng ký hợp tác xã hoặc giây tờ có giá trị tương '
             'đương;\n'
             '\n'
             'c) Các tài liệu chứng minh tư cách đại diện của người đại diện theo pháp luật hoặc '
             'người đại diện theo ủy quyền của tổ chức mở tài khoản thanh toán và thể căn cước, '
             'thẻ căn cước công dân, căn cước điện tử hoặc chứng minh nhân dân hoặc hộ chiếu còn '
             'thời hạn của người đó; \n'
             'đ) Văn bản hoặc quyết định bỗ nhiệm và thẻ căn cước, thẻ căn cước công dân, căn cước '
             'điện tử hoặc chứng minh nhân dân hoặc hộ chiếu còn thời hạn \n'
             'của kế toán trưởng hoặc người phụ trách kế toán, người kiểm soát chứng từ giao dịch '
             'với Ngân hàng Nhà nước của tổ chức mở tải khoản thanh toán.'},
 {'id': 24,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 14',
               'article_title': 'Hồ sơ và trình tự thủ tục mở, đóng tài khoản thanh toán tại Ngân '
                                'hàng Nhà nước của tô chức tín dụng, chỉ nhánh ngân hàng nước '
                                'ngoài, Kho bạc Nhà nước',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 14. Hồ sơ và trình tự thủ tục '
                                  'mở, đóng tài khoản thanh toán tại Ngân hàng Nhà nước của tô '
                                  'chức tín dụng, chỉ nhánh ngân hàng nước ngoài, Kho bạc Nhà nước',
               'topics': 'eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 11,
               'chunk_part': '4/5'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 14. Hồ sơ và trình tự thủ tục mở, đóng tài khoản '
             'thanh toán tại Ngân hàng Nhà nước của tô chức tín dụng, chỉ nhánh ngân hàng nước '
             'ngoài, Kho bạc Nhà nước (tiếp theo)]\n'
             '3. Trình tự, thủ tục mở tài khoản thanh toán:\n'
             '\n'
             'a) Khi có nhu cầu mở tài khoản thanh toán tại Ngân hàng Nhà nước, tổ chức để nghị mở '
             'tài khoản thanh toán gửi 01 bộ hồ sơ mở tài khoản thanh toán theo quy định tại khoản '
             '2 Điều này đến Ngân hàng Nhà nước (Sở Giao dịch, Ngân hàng Nhà nước chỉ nhánh tỉnh, '
             'thành phố) nơi đề nghị mở tài khoản thanh toán;\n'
             '\n'
             'b) Khi nhận được hồ sơ mở tài khoản thanh toán, Ngân hàng Nhà nước tiến hành kiểm '
             'tra các thành phần hồ sơ và đối chiếu với các yếu tô đã kê khai tại đơn đề nghị mở '
             'tài khoản thanh toán, đảm bảo sự khớp đúng, chính xác. \n'
             ' Trường hợp hồ sơ mở tài khoản thanh toán chưa đầy đủ, chưa hợp lệ hoặc còn có sự '
             'sai lệch giữa các yếu tổ kê khai tại đơn để nghị mở tài khoản thanh toán với các tài '
             'liệu liên quan trong hồ sơ, trong thời hạn 01 ngày làm việc kế từ ngày nhận được hồ '
             'sơ mở tài khoản thanh toán, Ngân hàng Nhà nước thông báo cho tô chức đề nghị mở tải '
             'khoản biết để hoàn thiện hồ sơ. \n'
             'Trong thời hạn 05 ngày làm việc kể từ ngày nhận được văn bản đề nghị bố sung, hoàn '
             'thiện hồ sơ của Ngân hàng Nhà nước mà tổ chức đề nghị mở tài khoản không gửi bổ '
             'sung, hoàn thiện hồ sơ thì Ngân hàng Nhà nước sẽ có văn bản từ chối mở tài khoản '
             'thanh toán và trả lại hồ sơ cho tổ chức đề nghị mở tài khoản;\n'
             '\n'
             'c) Trong thời hạn 02 ngày làm việc kế từ khi nhận đầy đủ hồ sơ mở tài khoản thanh '
             'toán hợp lệ của tổ chức đề nghị mở tài khoản thanh toán, Ngân hàng Nhà nước phải xử '
             'lý việc mở tài khoản thanh toán cho tổ chức. Trường hợp Ngân hàng Nhà nước từ chối '
             'mở tài khoản thanh toán thì phải thông báo lý do cho tô chức biết bằng văn bản.'},
 {'id': 25,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 14',
               'article_title': 'Hồ sơ và trình tự thủ tục mở, đóng tài khoản thanh toán tại Ngân '
                                'hàng Nhà nước của tô chức tín dụng, chỉ nhánh ngân hàng nước '
                                'ngoài, Kho bạc Nhà nước',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 14. Hồ sơ và trình tự thủ tục '
                                  'mở, đóng tài khoản thanh toán tại Ngân hàng Nhà nước của tô '
                                  'chức tín dụng, chỉ nhánh ngân hàng nước ngoài, Kho bạc Nhà nước',
               'topics': 'eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 11,
               'chunk_part': '5/5'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 14. Hồ sơ và trình tự thủ tục mở, đóng tài khoản '
             'thanh toán tại Ngân hàng Nhà nước của tô chức tín dụng, chỉ nhánh ngân hàng nước '
             'ngoài, Kho bạc Nhà nước (tiếp theo)]\n'
             '4. Trình tự, thú tục đóng tài khoản thanh toán:\n'
             '\n'
             'a) Tổ chức tín dụng, chỉ nhánh ngân hàng nước ngoài, Kho bạc Nhà nước có nhu cầu '
             'đóng tài khoản thanh toán tại Ngân hàng Nhà nước phải lập đơn đề nghị đóng tài khoản '
             'thanh toán và yêu cầu xử lý số dư tài khoản thanh toán (nếu có) theo Mẫu số 02 ban '
             'hành kèm theo Nghị định này do người đại diện hợp pháp của tổ chức ký tên, đóng dấu '
             'và gửi Ngân hàng Nhà nước (nơi mở tài khoản thanh toán);\n'
             '\n'
             'b) Khi nhận được đơn đề nghị đóng tài khoản thanh toán, Ngân hàng Nhà nước tiến hành '
             'kiểm tra, đối chiếu thông tin trên đơn đề nghị với thông tin tài khoản và xử lý số '
             'dư trên tài khoản thanh toán theo yêu cầu của chủ tài khoản (nếu có). Sau khi hoàn '
             'thành việc xử lý số dư còn lại trên tài khoản thanh toán, Ngân hàng Nhà nước thực '
             'hiện đóng tài khoản thanh toán; \n'
             '©) Trong thời hạn 02 ngày làm việc kể từ khi nhận được đơn đề nghị đóng tài khoản '
             'thanh toán của tô chức mở tài khoản thanh toán, Ngân hàng Nhà nước phải xử lý việc '
             'đóng tài khoản thanh toán; \n'
             'đ) Trường hợp đóng tài khoản thanh toán theo quy định tại điểm c, đ, đ khoản 1 Điều '
             '12 Nghị định này, việc xử lý số dư còn lại trên tài khoản thanh toán (nếu có) sau '
             'khi đã hoàn tất các nghĩa vụ với các bên liên quan thì được thực hiện theo yêu cầu '
             'bằng văn bản của chủ tài khoản trước khi có quyết định thu hồi Giấy phép thành lập '
             'và hoạt động, theo quyết định của cơ quan nhà nước có thâm quyền hoặc theo quy định '
             'pháp luật có liên quan. Sau khi đóng tài khoản thanh toán, trong thời hạn 05 ngày '
             'làm việc, Ngân hàng Nhà nước thông báo cho tổ chức mở tài khoản biết bằng văn bản. \n'
             '\n'
             'Mục 3 MỞ VÀ SỬ DỤNG TÀI KHOẢN THANH TOÁN TẠI CÁC TỎ CHỨC TÍN DỤNG, CHI NHÁNH NGÂN '
             'HÀNG NƯỚC NGOÀI'},
 {'id': 26,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 15',
               'article_title': 'Mở và sử dụng tài khoản thanh toán giữa các tổ chức tín dụng, chỉ '
                                'nhánh ngân hàng nước ngoài',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 15. Mở và sử dụng tài khoản '
                                  'thanh toán giữa các tổ chức tín dụng, chỉ nhánh ngân hàng nước '
                                  'ngoài',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 11,
               'chunk_part': '1/1'},
  'content': 'Điều 15. Mở và sử dụng tài khoản thanh toán giữa các tổ chức tín dụng, chỉ nhánh '
             'ngân hàng nước ngoài\n'
             '\n'
             '1. Việc mở tài khoản thanh toán giữa các tổ chức tín dụng, chỉ nhánh ngân hàng nước '
             'ngoài phải thực hiện theo đúng quy định tại Luật Các tổ chức tín dụng. Tài khoản '
             'thanh toán mở giữa các tổ chức tín dụng, chỉ nhánh ngân hàng nước ngoài chỉ phục vụ '
             'cho mục đích thanh toán, không được cho vay, thấu chỉ hoặc sử dụng cho các mục đích '
             'khác.\n'
             '\n'
             '2. Tổ chức tín dụng, chỉ nhánh ngân hàng nước ngoài được phép hoạt động ngoại hối '
             'được mở tài khoản thanh toán bằng ngoại tệ tại tô chức tín dụng được phép khác. Việc '
             'mở, sử dụng tài khoản thanh toán bằng ngoại tệ thực hiện theo quy định của pháp luật '
             'về ngoại hối.'},
 {'id': 27,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 16',
               'article_title': 'Mỡ tài khoản thanh toán cho khách hàng không phải là tổ chức tín '
                                'dụng',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 16. Mỡ tài khoản thanh toán cho '
                                  'khách hàng không phải là tổ chức tín dụng',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 11,
               'chunk_part': '1/1'},
  'content': 'Điều 16. Mỡ tài khoản thanh toán cho khách hàng không phải là tổ chức tín dụng\n'
             '\n'
             '1. Ngân hàng, chi nhánh ngân hàng nước ngoài hướng dẫn việc mở tài khoản thanh toán '
             'cho khách hàng phù hợp với quy định của Ngân hàng Nhà nước và các quy định pháp luật '
             'khác có liên quan.\n'
             '\n'
             '2. Chủ tài khoản thanh toán chung là tổ chức, cá nhân. Mục đích sử dụng tài khoản '
             'thanh toán chung, quyên và nghĩa vụ của các chủ tài khoản thanh toán chung và các '
             'quy định liên quan đến việc sử dụng tài khoản thanh toán chung phải được xác định rõ '
             'bằng văn bản và phù hợp với quy định pháp luật về mở, sử dụng tài khoản thanh toán.\n'
             '\n'
             'Chương HI DỊCH VỤ THANH TOÁN KHÔNG DÙNG TIỀN MẶT\n'
             '\n'
             'Mục 1 DỊCH VỤ THANH TOÁN QUA TÀI KHOẢN THANH TOÁN CỦA KHÁCH HÀNG'},
 {'id': 28,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 17',
               'article_title': 'Địch vụ thanh toán qua tài khoản thanh toán của khách hàng',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 17. Địch vụ thanh toán qua tài '
                                  'khoản thanh toán của khách hàng',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 14,
               'chunk_part': '1/1'},
  'content': 'Điều 17. Địch vụ thanh toán qua tài khoản thanh toán của khách hàng\n'
             '1. Dịch vụ thanh toán qua tài khoản thanh toán của khách hàng, bao gồm:\n'
             'a) Cung ứng phương tiện thanh toán;\n'
             '\n'
             'b) Thực hiện dịch vụ thanh toán: séc, lệnh chi, ủy nhiệm chi, nhờ thu, ủy nhiệm thu, '
             'thẻ ngân hàng, chuyền tiền, thu hộ, chỉ hộ; \n'
             '\n'
             'c) Các dịch vụ thanh toán khác thực biện theo quy định của Ngân hàng Nhà nước.\n'
             '\n'
             '2. Các tổ chức cung ứng dịch vụ thanh toán qua tài khoản thanh toán của khách hàng:\n'
             '\n'
             'a) Ngân hàng Nhà nước cung ứng các dịch vụ thanh toán cho các khách hàng mở tài '
             'khoản thanh toán tại Ngân hàng Nhà nước;\n'
             '\n'
             'b) Ngân hàng thương mại, chỉ nhánh ngân hàng nước ngoài, ngân hàng chính sách cung '
             'ứng tất cả các dịch vụ thanh toán quy định tại khoản 1 Điều này;\n'
             '\n'
             'c) Ngân hàng hợp tác xã được cung ứng một số dịch vụ thanh toán quy định tại khoản 1 '
             'Điều này sau khi được ghi trong Giấy phép thành lập và hoạt động do Ngân hàng Nhà '
             'nước cấp.\n'
             '\n'
             '3. Việc cung ứng dịch vụ thanh toán qua tài khoản thanh toán của khách hàng thực '
             'hiện theo quy định của Ngân hàng Nhà nước.\n'
             '\n'
             'Mục 2 ` DỊCH VỤ THANH TOÁN KHÔNG QUA TÀI KHOẢN THANH TOÁN CỦA KHÁCH HÀNG'},
 {'id': 29,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 18',
               'article_title': 'Dịch vụ thanh toán không qua tài khoản thanh toán của khách hàng',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 18. Dịch vụ thanh toán không qua '
                                  'tài khoản thanh toán của khách hàng',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 15,
               'chunk_part': '1/1'},
  'content': 'Điều 18. Dịch vụ thanh toán không qua tài khoản thanh toán của khách hàng\n'
             '\n'
             '1. Dịch vụ thanh toán không qua tài khoản thanh toán của khách hàng, bao gỗm:\n'
             '\n'
             'a) Thực hiện địch vụ thanh toán: chuyến tiền, thu hộ, chi hộ;\n'
             '\n'
             'b) Các dịch vụ thanh toán khác không qua tài khoản thực hiện theo quy định của Ngân '
             'hàng Nhà nước.\n'
             '\n'
             '2. Các tổ chức được cung ứng dịch vụ thanh toán không qua tài khoản thanh toán của '
             'khách hàng:\n'
             '\n'
             'a) Ngân hàng thương mại, chỉ nhánh ngân hàng nước ngoài, ngân hàng chính sách;\n'
             '\n'
             'b) Ngân hàng hợp tác xã được cung ứng một số dịch vụ thanh toán không qua tài khoản '
             'thanh toán của khách hàng sau khi được ghi trong Giấy phép thành lập và hoạt động do '
             'Ngân hàng Nhà nước cấp;\n'
             '\n'
             'c) Quỹ tín dụng nhân dân được cung ứng dịch vụ chuyên tiền, thực hiện nghiệp vụ thu '
             'hộ, chi hộ cho thành viên, khách hàng của quỹ tín dụng nhân dân đó sau khi được ghi '
             'trong Giấy phép thành lập và hoạt động do Ngân hàng Nhà nước cấp; \n'
             ' đ) Tổ chức tài chính vi mô được cung ứng dịch vụ chuyển tiền, thu hộ, chi hộ cho '
             'khách hàng của tổ chức tài chính vi mô sau khi được ghi trong Giầy phép thành lập và '
             'hoạt động do Ngân hàng Nhà nước câp; \n'
             'ổđ) Doanh nghiệp cung ứng dịch vụ bưu chính công ích được cung ứng địch vụ chuyên '
             'tiên, thu hộ, chi hộ sau khi đáp ứng điều kiện quy định tại\n'
             'Điều 19 Nghị định này và được Ngân hàng Nhà nước chấp thuận bằng văn bản.\n'
             '\n'
             '3. Việc cung ứng dịch vụ thanh toán không qua tài khoản thanh toán của khách hàng '
             'thực hiện theo quy định của Ngân hàng Nhà nước.'},
 {'id': 30,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 19',
               'article_title': 'Điều kiện cung ứng dịch vụ thanh toán không qua tài khoản thanh '
                                'toán của khách hàng của doanh nghiệp cung ứng dịch vụ bưu chính '
                                'công ích',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 19. Điều kiện cung ứng dịch vụ '
                                  'thanh toán không qua tài khoản thanh toán của khách hàng của '
                                  'doanh nghiệp cung ứng dịch vụ bưu chính công ích',
               'topics': 'Hạn mức & Giao dịch, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 16,
               'chunk_part': '1/1'},
  'content': 'Điều 19. Điều kiện cung ứng dịch vụ thanh toán không qua tài khoản thanh toán của '
             'khách hàng của doanh nghiệp cung ứng dịch vụ bưu chính công ích \n'
             'Doanh nghiệp cung ứng dịch vụ bưu chính công ích được cung ứng dịch vụ thanh toán '
             'không qua tài khoản thanh toán của khách hàng khi đáp ứng đầy đủ và phải đảm bảo duy '
             'trì đủ các điều kiện sau đây trong quá trình cung ứng dịch vụ thanh toán không qua '
             'tài khoản thanh toán của khách hàng:\n'
             '\n'
             '1. Có hệ thống thông tin phục vụ cho hoạt động cung ứng địch vụ thanh toán không qua '
             'tài khoản thanh toán của khách hàng đáp ứng yêu câu đảm bảo an toàn hệ thống thông '
             'tin cấp độ 3 theo quy định của pháp luật.\n'
             '\n'
             '2. Điều kiện về nhân sự: Người đại diện theo pháp luật, Tổng Giám đốc (Giám đốc), '
             'Người phụ trách cung ứng dịch vụ thanh toán không qua tài khoản thanh toán của khách '
             'hàng của doanh nghiệp cung ứng dịch vụ bưu chính công ích phải có bằng đại học trở '
             'lên về một trong các ngành kinh tế, quản trị kinh doanh, luật, công nghệ thông '
             'tin. \n'
             'Các cán bộ chủ chốt thực hiện cung ứng dịch vụ thanh toán không qua tài khoản thanh '
             'toán của khách hàng, (gồm Trưởng phòng (ban) hoặc tương đương và các cán bộ kỹ '
             'thuật) có bằng cao đẳng trở lên về một trong các ngành kinh tế, quản trị kinh doanh, '
             'luật, công nghệ thông tin hoặc lĩnh vực chuyên môn đảm nhiệm.\n'
             '\n'
             '3. Có quy trình nghiệp vụ kỹ thuật đối với từng loại dịch vụ; có biện pháp đảm bảo '
             'khả năng thanh toán, duy trì số dư tài khoản thanh toán của đơn vị mình tại ngân '
             'hàng và tiền mặt lớn hơn số tiền phải trả cho khách hàng tại thời điểm chỉ trả; quy '
             'trình kiểm tra, kiểm soát nội bộ; cơ chế quản lý rủi ro; các nguyên tắc chung và quy '
             'định nội bộ về phòng, chống rửa tiền, tài trợ khủng bố và tài trợ phổ biến vũ khí '
             'hủy diệt hàng loạt; quy trình và thủ tục giải quyết yêu cầu tra soát, khiếu nại, '
             'tranh chấp; quy định quyền và trách nhiệm của các bên có liên quan. \n'
             '\n'
             '4. Có phương án thu gom, vận chuyển tiền mặt đảm bảo cuối ngày nộp vào tài khoản '
             'thanh toán mở tại ngân hàng, đảm bảo an ninh, an toàn đôi với việc luận chuyển tiền '
             'mặt; trang bị các thiết bị đảm bảo việc giao nhận, bảo quản tiền mặt, quy định hạn '
             'mức chuyển tiền, nhận tiền, mức tồn quỹ tại các điểm cung cấp dịch vụ; đảm bảo công '
             'tác phòng cháy, chữa cháy theo quy định của pháp luật.'},
 {'id': 31,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 20',
               'article_title': 'Hồ sơ, quy trình, thủ tục chấp thuận bằng văn bắn, thu hồi văn '
                                'bản hoạt động cung ứng dịch vụ thanh toán không qua tài khoản '
                                'thanh toán của khách hàng của đoanh nghiệp cung ứng dịch vụ bưu '
                                'chính công ích',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 20. Hồ sơ, quy trình, thủ tục '
                                  'chấp thuận bằng văn bắn, thu hồi văn bản hoạt động cung ứng '
                                  'dịch vụ thanh toán không qua tài khoản thanh toán của khách '
                                  'hàng của đoanh nghiệp cung ứng dịch vụ bưu chính công ích',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 17,
               'chunk_part': '1/6'},
  'content': 'Điều 20. Hồ sơ, quy trình, thủ tục chấp thuận bằng văn bắn, thu hồi văn bản hoạt '
             'động cung ứng dịch vụ thanh toán không qua tài khoản thanh toán của khách hàng của '
             'đoanh nghiệp cung ứng dịch vụ bưu chính công ích'},
 {'id': 32,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 20',
               'article_title': 'Hồ sơ, quy trình, thủ tục chấp thuận bằng văn bắn, thu hồi văn '
                                'bản hoạt động cung ứng dịch vụ thanh toán không qua tài khoản '
                                'thanh toán của khách hàng của đoanh nghiệp cung ứng dịch vụ bưu '
                                'chính công ích',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 20. Hồ sơ, quy trình, thủ tục '
                                  'chấp thuận bằng văn bắn, thu hồi văn bản hoạt động cung ứng '
                                  'dịch vụ thanh toán không qua tài khoản thanh toán của khách '
                                  'hàng của đoanh nghiệp cung ứng dịch vụ bưu chính công ích',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 17,
               'chunk_part': '2/6'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 20. Hồ sơ, quy trình, thủ tục chấp thuận bằng văn '
             'bắn, thu hồi văn bản hoạt động cung ứng dịch vụ thanh toán không qua tài khoản thanh '
             'toán của khách hàng của đoanh nghiệp cung ứng dịch vụ bưu chính công ích (tiếp '
             'theo)]\n'
             '1. Hồ sơ đề nghị chấp thuận cung ứng dịch vụ thanh toán không qua tài khoản thanh '
             'toán của khách hàng gỗm:\n'
             '\n'
             'a) Đơn đề nghị cung ứng dịch vụ thanh toán không qua tải khoản thanh toán của khách '
             'hàng theo Mẫu số 03 ban hành kèm theo Nghị định này;\n'
             '\n'
             'b) Nghị quyết của Hội đồng thành viên, văn bản của người đại diện có thẩm quyên của '
             'chủ sở hữu phù hợp với thâm quyền quy định tại Điều lệ công ty về việc thông qua Bản '
             'thuyết minh điều kiện cung ứng dịch vụ thanh toán không qua tài khoản thanh toán của '
             'khách hàng;\n'
             '\n'
             'c) Bản thuyết minh các điều kiện cung ứng dịch vụ thanh toán không qua tài khoản '
             'thanh toán của khách hàng theo quy định tại Điều 19 Nghị định này; \n'
             'Q): Hồ sơ về nhân sự: Sơ yếu lý lịch, bản sao có chứng thực hoặc bản sao được cấp từ '
             'số gốc hoặc bản sao kèm xuất trình bản chính để đối chiếu các văn bằng chứng minh '
             'năng lực, trình độ chuyên môn nghiệp vụ của những người đại diện theo pháp luật, '
             'Tổng Giám đốc (Giám đốc), Người phụ trách và các cán bộ chủ chốt thực hiện cung ứng '
             'dịch vụ này; \n'
             'đ) Giấy phép thành lập hoặc giấy chứng nhận đăng ký doanh nghiệp hoặc giây tờ có giá '
             'trị tương đương do cơ quan nhà nước có thâm quyên cập,\n'
             'Điều lệ công ty (bản sao có chứng thực hoặc bản sao được cấp từ số gốc hoặc bản sao '
             'kèm xuất trình bản chính đề đối chiếu).'},
 {'id': 33,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 20',
               'article_title': 'Hồ sơ, quy trình, thủ tục chấp thuận bằng văn bắn, thu hồi văn '
                                'bản hoạt động cung ứng dịch vụ thanh toán không qua tài khoản '
                                'thanh toán của khách hàng của đoanh nghiệp cung ứng dịch vụ bưu '
                                'chính công ích',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 20. Hồ sơ, quy trình, thủ tục '
                                  'chấp thuận bằng văn bắn, thu hồi văn bản hoạt động cung ứng '
                                  'dịch vụ thanh toán không qua tài khoản thanh toán của khách '
                                  'hàng của đoanh nghiệp cung ứng dịch vụ bưu chính công ích',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 17,
               'chunk_part': '3/6'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 20. Hồ sơ, quy trình, thủ tục chấp thuận bằng văn '
             'bắn, thu hồi văn bản hoạt động cung ứng dịch vụ thanh toán không qua tài khoản thanh '
             'toán của khách hàng của đoanh nghiệp cung ứng dịch vụ bưu chính công ích (tiếp '
             'theo)]\n'
             '2. Quy trình, thủ tục chấp thuận:\n'
             '\n'
             'a) Doanh nghiệp cung ứng dịch vụ bưu chính công ích gửi 03 bộ hồ sơ để nghị cung ứng '
             'dịch vụ thanh toán không qua tài khoản thanh toán của khách hàng theo quy định tại '
             'khoản 1 Điều này qua đường bưu điện (dịch vụ bưu chính) hoặc nộp trực tiếp tới Bộ '
             'phận Một cửa Ngân hàng Nhà nước hoặc trực tuyến tại Cổng dịch vụ công Ngân hàng Nhà '
             'nước hoặc Công dịch vụ công quốc gia (hồ sơ gửi trực tuyến thì thực hiện theo quy '
             'định về thủ tục hành chính trên môi trường điện tử). Doanh nghiệp cung ứng dịch vụ '
             'bưu chính công ích đề nghị phải chịu hoàn toàn trách nhiệm trước pháp luật về tính '
             'chính xác, trung thực của các thông tin cung cấp. \n'
             ' __ Căn cứ vào hồ sơ đề nghị, Ngân hàng Nhà nước tiến hành thâm định hồ sơ trên cơ '
             'sở các điêu kiện quy định tại Điêu 19 Nghị định này;\n'
             '\n'
             'b) Trong thời hạn 05 ngày làm việc kể từ ngày nhận được hồ sơ đề nghị, Ngân hàng Nhà '
             'nước có văn bản gửi doanh nghiệp cung ứng dịch vụ bưu chính công ích đề nghị xác '
             'nhận đã nhận đầy đủ thành phần hỗ sơ hợp lệ hoặc không đầy đủ, hợp lệ theo quy '
             'định. \n'
             'Trường hợp hồ sơ đề nghị không đầy đủ, hợp lệ theo quy định, Ngân hàng Nhà nước có '
             'văn bản gửi doanh nghiệp này yêu. cầu bổ sung, hoàn thiện hồ sơ. Thời gian doanh '
             'nghiệp bổ sung, hoàn thiện hồ sơ không tính vào thời gian thâm định.\n'
             '\n'
             'c) Trong thời hạn 60 ngày làm việc kể từ ngày nhận đầy đủ thành phần hồ sơ hợp lệ, '
             'Ngân hàng Nhà nước tiến hành thẩm định hồ sơ. Nếu quá 60 ngày kể từ ngày Ngân hàng '
             'Nhà nước có văn bản yêu cầu bổ sung, hoàn thiện hồ sơ nhưng doanh nghiệp đề nghị '
             'không gửi bổ sung hồ sơ hoặc sau 02 lần gửi mà hồ sơ vẫn không đáp ứng điều kiện thì '
             'Ngân hàng Nhà nước có văn bản từ chối chấp thuận và trả lại bồ sơ cho doanh nghiệp '
             'cung ứng dịch vụ bưu chính công ích. \n'
             'Trong thời hạn 60 ngày làm việc kế từ ngày nhận được hồ sơ bỗ sung, hoàn thiện của '
             'doanh nghiệp cung ứng dịch vụ bưu chính công ích, Ngân hàng Nhà nước tiến hành thẩm '
             'định, chấp thuận bằng văn bản theo quy định. Trường hợp không chấp thuận, Ngân hàng '
             'Nhà nước có văn bản trả lời doanh nghiệp, trong đó nêu rõ lý do.'},
 {'id': 34,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 20',
               'article_title': 'Hồ sơ, quy trình, thủ tục chấp thuận bằng văn bắn, thu hồi văn '
                                'bản hoạt động cung ứng dịch vụ thanh toán không qua tài khoản '
                                'thanh toán của khách hàng của đoanh nghiệp cung ứng dịch vụ bưu '
                                'chính công ích',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 20. Hồ sơ, quy trình, thủ tục '
                                  'chấp thuận bằng văn bắn, thu hồi văn bản hoạt động cung ứng '
                                  'dịch vụ thanh toán không qua tài khoản thanh toán của khách '
                                  'hàng của đoanh nghiệp cung ứng dịch vụ bưu chính công ích',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 17,
               'chunk_part': '4/6'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 20. Hồ sơ, quy trình, thủ tục chấp thuận bằng văn '
             'bắn, thu hồi văn bản hoạt động cung ứng dịch vụ thanh toán không qua tài khoản thanh '
             'toán của khách hàng của đoanh nghiệp cung ứng dịch vụ bưu chính công ích (tiếp '
             'theo)]\n'
             '3. Thời hạn hiệu lực của văn bản chấp thuận là 10 năm tính từ ngày ký văn bản chấp '
             'thuận của Ngân hàng Nhà nước. Trường hợp gia hạn hiệu lực của văn bản chấp thuận '
             'hoạt động cung ứng dịch vụ thanh toán không qua tài khoản thanh toán của khách hàng, '
             'trong thời hạn tối thiểu 60 ngày trước khi văn bản chấp thuận hết thời hạn, doanh '
             'nghiệp đề nghị gửi 03 bộ hồ sơ đề nghị chấp thuận gia hạn văn bản gồm: đơn đề nghị '
             'cung ứng dịch vụ thanh toán không qua tài khoản thanh toán của khách hàng theo Mẫu '
             'số 03 ban hành kèm theo Nghị định này; báo cáo tình hình thực hiện hoạt động theo '
             'văn bản chấp thuận kế từ ngày được chấp thuận đến ngày nộp đơn đề nghị và bản sao '
             'văn bản chấp thuận đang có hiệu lực tới Ngân hàng Nhà nước. \n'
             'Trong thời hạn 30 ngày làm việc kế từ ngày nhận được hỗ sơ đề nghị gia hạn văn bản '
             'chấp thuận của doanh nghiệp, Ngân hàng Nhà nước sẽ xem xét và gia hạn văn bản hoặc '
             'có văn bản thông báo từ chối trong đó nêu rõ lý do. \n'
             'Thời hạn gia hạn văn bản chấp thuận là 10 năm tính từ ngày doanh nghiệp được Ngân '
             'hàng Nhà nước gia hạn văn bản. \n'
             ' 18 ị'},
 {'id': 35,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 20',
               'article_title': 'Hồ sơ, quy trình, thủ tục chấp thuận bằng văn bắn, thu hồi văn '
                                'bản hoạt động cung ứng dịch vụ thanh toán không qua tài khoản '
                                'thanh toán của khách hàng của đoanh nghiệp cung ứng dịch vụ bưu '
                                'chính công ích',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 20. Hồ sơ, quy trình, thủ tục '
                                  'chấp thuận bằng văn bắn, thu hồi văn bản hoạt động cung ứng '
                                  'dịch vụ thanh toán không qua tài khoản thanh toán của khách '
                                  'hàng của đoanh nghiệp cung ứng dịch vụ bưu chính công ích',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 17,
               'chunk_part': '5/6'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 20. Hồ sơ, quy trình, thủ tục chấp thuận bằng văn '
             'bắn, thu hồi văn bản hoạt động cung ứng dịch vụ thanh toán không qua tài khoản thanh '
             'toán của khách hàng của đoanh nghiệp cung ứng dịch vụ bưu chính công ích (tiếp '
             'theo)]\n'
             '4. Thu hồi văn bản chấp thuận hoạt động cung ứng dịch vụ thanh toán không qua tài '
             'khoản thanh toán của khách hàng trong các trường hợp sau:\n'
             '\n'
             'a) Doanh nghiệp được Ngân hàng Nhà nước chấp thuận bằng văn bản hoạt động cung ứng '
             'dịch vụ thanh toán không qua tài khoản thanh toán của khách hàng bị giải thê hoặc '
             'phá sản theo quy định của pháp luật;\n'
             '\n'
             'b) Doanh nghiệp được Ngân hàng Nhà nước chấp thuận bằng văn bản hoạt động cung ứng '
             'dịch vụ thanh toán không qua tài khoản thanh toán của khách hàng có đơn đề nghị thu '
             'hồi văn bản chấp thuận do chấm dứt hoạt động cung ứng dịch vụ thanh toán không qua '
             'tài khoản thanh toán của khách hàng theo Mẫu số 05 ban hành kèm theo Nghị định nảy;\n'
             '\n'
             'c) Khi có bản án, quyết định thi hành án, quyết định xử phạt vi phạm hành chính của '
             'cơ quan nhà nước có thấm quyền, cơ quan thi hành án hình sự có nội dung yêu câu thu '
             'hồi văn bản chấp thuận của doanh nghiệp cung ứng dịch vụ bưu chính công ích hoặc có '
             'văn bản yêu cầu của cơ quan nhà nước có thâm quyền, cơ quan thi hành án hình sự đề '
             'nghị thu hồi văn bản chấp thuận của doanh nghiệp cung ứng dịch vụ bưu chính công '
             'ích; \n'
             'đ) Doanh nghiệp được Ngân hàng Nhà nước chấp thuận bằng văn bản hoạt động cung ứng '
             'dịch vụ thanh toán không qua tài khoản thanh toán của khách hàng vi phạm hành vi bị '
             'cắm tại khoản 8 Điều 8 Nghị định này; \n'
             'đ) Sau thời hạn 03 tháng kể từ ngày Ngân chàng Nhà nước có văn bản thông báo cho tổ '
             'chức vi phạm một trong các điều kiện trong quá trình cung ứng dịch vụ quy định tại '
             'Điều 19 và phải thực hiện các biện pháp khắc phục nhưng doanh nghiệp không khắc phục '
             'được; \n'
             '-_ ©) Hoạt động không đúng nội dung chấp thuận của Ngân hàng Nhà nước về cung ứng '
             'dịch vụ thanh toán không qua tài khoản thanh toán cửa khách hàng; \n'
             'ø) Trong quá trình thanh tra, kiểm tra, giám sát hoạt động cung ứng dịch vụ thanh '
             'toán không qua tài khoản thanh toán của khách hàng phát hiện trong thời hạn 06 tháng '
             'liên tục, doanh nghiệp không thực hiện hoạt động cung ứng địch vụ thanh toán không '
             'qua tài khoản thanh toán của khách hàng.'},
 {'id': 36,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Phần mở đầu / Căn cứ pháp lý',
               'article': 'Điều 20',
               'article_title': 'Hồ sơ, quy trình, thủ tục chấp thuận bằng văn bắn, thu hồi văn '
                                'bản hoạt động cung ứng dịch vụ thanh toán không qua tài khoản '
                                'thanh toán của khách hàng của đoanh nghiệp cung ứng dịch vụ bưu '
                                'chính công ích',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 20. Hồ sơ, quy trình, thủ tục '
                                  'chấp thuận bằng văn bắn, thu hồi văn bản hoạt động cung ứng '
                                  'dịch vụ thanh toán không qua tài khoản thanh toán của khách '
                                  'hàng của đoanh nghiệp cung ứng dịch vụ bưu chính công ích',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 17,
               'chunk_part': '6/6'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 20. Hồ sơ, quy trình, thủ tục chấp thuận bằng văn '
             'bắn, thu hồi văn bản hoạt động cung ứng dịch vụ thanh toán không qua tài khoản thanh '
             'toán của khách hàng của đoanh nghiệp cung ứng dịch vụ bưu chính công ích (tiếp '
             'theo)]\n'
             '5. Quy trình, thủ tục xem xét thu hồi hoặc thu hồi văn bản chấp thuận hoạt động cung '
             'ứng dịch vụ thanh toán không qua tài khoản thanh toán của khách hàng của doanh '
             'nghiệp cung ứng dịch vụ bưu chính công ích:\n'
             '\n'
             'a) Trường hợp doanh nghiệp bị giải thể hoặc phá sản theo quy định tại điểm a khoản 4 '
             'Điều này, doanh nghiệp có văn bản thông báo với Ngân hàng Nhà nước trong thời hạn 07 '
             'ngày làm việc kể từ ngày thông, qua Quyết định giải thê doanh nghiệp theo quy định '
             'tại Luật Doanh nghiệp hoặc ngày nhận được Quyết định tuyên bố phá sản của Tòa án '
             'nhân dân theo quy định của \n'
             ' I9 \n'
             'pháp luật về phá sản. Sau 10 ngày làm việc kể từ ngày nhận được văn bản thông báo '
             'của doanh nghiệp, Ngân hàng Nhà nước ra quyết định thu hồi văn bản chấp thuận. \n'
             'Sau 10 ngày làm việc kế từ ngày Ngân hàng Nhà nước nhận được đơn đề nghị thu hồi văn '
             'bản chấp thuận hoạt động cung ứng dịch vụ thanh toán không qua tài khoản thanh toán '
             'của khách hàng quy định tại điểm b khoản 4\n'
             'Điều này, Ngân hàng Nhà nước ra quyết định thu hồi văn bản chấp thuận. \n'
             'Sau 10 ngày làm việc kế từ ngày phát sinh một trong các trường hợp quy định tại điểm '
             'c, điểm d, điểm đ khoản 4 Điều này, Ngân hàng Nhà nước ra quyết định thu hồi văn bản '
             'chấp thuận;\n'
             '\n'
             'b) Khi doanh nghiệp có dấu hiệu vi phạm một trong các trường hợp nêu tại điểm e, '
             'điểm ø khoản 4 Điều này, Ngân hàng Nhà nước ra thông báo và đề nghị doanh nghiệp '
             'giải trình. \n'
             'Sau I5 ngày làm việc kể từ ngày Ngân hàng Nhà nước ra thông báo nhưng doanh nghiệp '
             'được chấp thuận không có văn bản giải trình hoặc nội dung giải trình không xác đáng, '
             'Ngân hàng Nhà nước xem xét ra quyết định thu hồi văn bản chấp thuận.\n'
             '\n'
             '6. Ngay khi nhận được Quyết định của Ngân hàng Nhà nước về việc thu hồi văn bản chấp '
             'thuận, doanh nghiệp bị thu hồi văn bản chấp thuận phải lập tức ngừng cung ứng dịch '
             'vụ thanh toán không qua tài khoản thanh toán của khách hàng. \n'
             'Trong thời hạn 30 ngày kể từ ngày có quyết định của Ngân hàng Nhà nước về việc thu '
             'hồi văn bản chấp thuận, doanh nghiệp phải gửi thông báo bằng văn bản tới các tổ chức '
             'và cá nhân liên quan để thanh lý hợp đồng và hoàn tất các nghĩa vụ, trách nhiệm giữa '
             'các bên theo quy định pháp luật. \n'
             'Khi doanh nghiệp đã hoàn tất các nghĩa vụ, trách nhiệm giữa các bên theo quy định '
             'của pháp luật, sau thời hạn 03 năm kể từ ngày doanh nghiệp bị thu hồi văn bản chấp '
             'thuận trong trường hợp quy định tại khoản 4 (trừ điểm a khoản 4) Điều này, doanh '
             'nghiệp được đề nghị cung ứng dịch vụ thanh toán không qua tài khoản thanh toán của '
             'khách hàng theo quy định tại Điều 19 Nghị định này. Quy trình, thủ tục, hồ sơ thực '
             'hiện theo quy định tại khoản 1, khoản 2 Điều này.\n'
             '\n'
             'Mục 3 THAM GIA HỆ THÓNG THANH TOÁN QUỐC TÉ'},
 {'id': 37,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương IV DỊCH VỤ TRƯNG GIAN THANH TOÁN',
               'article': 'Điều 21',
               'article_title': 'Điều kiện tham gia hệ thống thanh toán quốc tế của ngân hàng '
                                'thương mại, chỉ nhánh ngân hàng nước ngoài',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 21. Điều kiện tham gia hệ thống '
                                  'thanh toán quốc tế của ngân hàng thương mại, chỉ nhánh ngân '
                                  'hàng nước ngoài',
               'topics': 'Hạn mức & Giao dịch, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 20,
               'chunk_part': '1/1'},
  'content': 'Điều 21. Điều kiện tham gia hệ thống thanh toán quốc tế của ngân hàng thương mại, '
             'chỉ nhánh ngân hàng nước ngoài\n'
             '\n'
             '1. Đã được phép thực hiện hoạt động ngoại hối cơ bản trên thị trường trong nước và '
             'quôc tê. \n'
             '\n'
             '2. Có chính sách và quy trình quản lý rủi ro về rửa tiền, tải trợ khủng bó, tài trợ '
             'phô biến vũ khí hủy diệt hàng loạt đáp ứng yêu cầu của pháp luật về phòng, chống rửa '
             'tiền, tài trợ khủng bố, tài trợ phổ biến vũ khí hủy diệt hàng loạt khi tham gia hệ '
             'thống thanh toán quốc tế.\n'
             '\n'
             '3. Có hệ thống thông tin đáp ứng yêu cầu về quản trị điều hành, an toàn, bảo mật '
             'theo quy định pháp luật của Việt Nam; có quy định nội bộ về tiêu chuẩn lựa chọn kết '
             'nối các hệ thống thanh toán quốc tế.\n'
             '\n'
             '4. Tổ chức vận hành hệ thống thanh toán quốc tế được thành lập và hoạt động hợp pháp '
             'ở nước ngoài.\n'
             '\n'
             'Chương IV DỊCH VỤ TRƯNG GIAN THANH TOÁN'},
 {'id': 38,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương IV DỊCH VỤ TRƯNG GIAN THANH TOÁN',
               'article': 'Điều 22',
               'article_title': 'Các địch vụ trung gian thanh toán và điều kiện cung ứng dịch vụ',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 22. Các địch vụ trung gian thanh '
                                  'toán và điều kiện cung ứng dịch vụ',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 21,
               'chunk_part': '1/6'},
  'content': 'Điều 22. Các địch vụ trung gian thanh toán và điều kiện cung ứng dịch vụ'},
 {'id': 39,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương IV DỊCH VỤ TRƯNG GIAN THANH TOÁN',
               'article': 'Điều 22',
               'article_title': 'Các địch vụ trung gian thanh toán và điều kiện cung ứng dịch vụ',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 22. Các địch vụ trung gian thanh '
                                  'toán và điều kiện cung ứng dịch vụ',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 21,
               'chunk_part': '2/6'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 22. Các địch vụ trung gian thanh toán và điều kiện '
             'cung ứng dịch vụ (tiếp theo)]\n'
             '1. Dịch vụ trung gian thanh toán bao gồm: dịch vụ chuyển mạch tài chính, dịch vụ '
             'chuyển mạch tài chính quốc tế, dịch vụ bù trừ điện tử, dịch vụ ví điện tử, dịch vụ '
             'hỗ trợ thu hộ, chi hộ và dịch vụ cổng thanh toán điện tử. Hoạt động cung ứng dịch vụ '
             'trung gian thanh toán thực hiện theo quy định của Ngân hàng Nhà nước.'},
 {'id': 40,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương IV DỊCH VỤ TRƯNG GIAN THANH TOÁN',
               'article': 'Điều 22',
               'article_title': 'Các địch vụ trung gian thanh toán và điều kiện cung ứng dịch vụ',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 22. Các địch vụ trung gian thanh '
                                  'toán và điều kiện cung ứng dịch vụ',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 21,
               'chunk_part': '3/6'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 22. Các địch vụ trung gian thanh toán và điều kiện '
             'cung ứng dịch vụ (tiếp theo)]\n'
             '2. Điều kiện cung ứng dịch vụ trung gian thanh toán: \n'
             'Tổ chức không phải là ngân hàng, chỉ nhánh ngân hàng nước ngoài được Ngân hàng Nhà '
             'nước cấp Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán khi đáp ứng đầy '
             'đủ và phải đảm bảo duy trì đủ các điều kiện sau đây trong quá trình cung ứng dịch vụ '
             'trung gian thanh toán, cụ thể như sau:\n'
             '\n'
             'a) Có giấy phép thành lập hoặc giây chứng nhận đăng ký doanh nghiệp đo cơ quan nhà '
             'nước có thâm quyền. cập và không đang trong quá trình chia, tách, hợp nhất, sáp '
             'nhập, chuyên đôi, giải thể, phá sản theo quyết định đã được ban hành trong quá trình '
             'đề nghị cấp Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán; trường hợp '
             'cung ứng dịch vụ chuyển mạch tài chính, dịch vụ bù trừ điện tử, tổ chức phải đâm bảo '
             'không kinh doanh ngành nghề khác ngoài hoạt động cung ứng địch vụ trung gian thanh '
             'toán;\n'
             '\n'
             'b) Có vốn điều lệ thực góp hoặc được cấp tối thiểu: 50 tỷ đồng đối với dịch vụ ví '
             'điện tử, dịch vụ hỗ trợ thu hộ, chi hộ và dịch vụ công thanh toán điện tử; 300 tỷ '
             'đồng đối với dịch vụ chuyển mạch tài chính, địch vụ chuyển mạch tài chính quôc tế, '
             'dịch vụ bù trừ điện tử; chịu hoàn toàn trách nhiệm về tính hợp pháp của nguồn vốn đã '
             'góp hoặc vốn được cấp; \n'
             " KÔ) Có Đề án cung ứng. dịch vụ trung gian thanh toán được _CẤP ' có thâm quyền theo "
             'quy định tại Điều lệ của tổ chức phê duyệt theo Mẫu số 08 ban hành kèm theo Nghị '
             'định này;\n'
             '\n'
             'd) Điều kiện. về nhân sự: Người đại diện (heo pháp luật, Tổng Giám đốc (Giám: đốc) '
             'của tổ chức phải có bằng đại học trở lên về một trong các ngành kinh tế, quản trị '
             'kinh doanh, luật, công nghệ thông tin và có ít nhất 05 năm kinh nghiệm là người quản '
             'lý, người điều hành của tổ chức trong lĩnh vực tài chính, ngân hàng và không thuộc '
             'những đối tượng bị cấm theo quy định của pháp luật; phải bảo đảm luôn có Ít nhất một '
             'người đại điện theo pháp luật cư trú tại Việt Nam (Khi chỉ còn lại một người đại '
             'diện theo pháp luật cư trú tại Việt Nam thì người này khi xuất cảnh khói Việt Nam '
             'phải ủ ủy quyền bằng văn bản cho cá nhân khác cư trú tại Việt Nam thực hiện quyên và '
             'nghĩa vụ của người đại diện theo pháp luật. Trường hợp này, người đại diện theo pháp '
             'luật vẫn phải chịu trách nhiệm về việc thực hiện quyên và nghĩa vụ đã ủy quyền). \n'
             'Phó Tổng Giám đốc (Phó Giám đốc) và các cán bộ chủ chốt thực hiện Đề án cung ứng '
             'địch vụ trung gian thanh toán (gồm Trưởng phòng (ban) hoặc tương đương và các cán bộ '
             'kỹ thuật) có bằng cao đẳng trở lên về một trong các ngành kinh tế, quản trị kinh '
             'doanh, luật, công nghệ thông tin hoặc lĩnh vực chuyên môn đám nhiệm;'},
 {'id': 41,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương IV DỊCH VỤ TRƯNG GIAN THANH TOÁN',
               'article': 'Điều 22',
               'article_title': 'Các địch vụ trung gian thanh toán và điều kiện cung ứng dịch vụ',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 22. Các địch vụ trung gian thanh '
                                  'toán và điều kiện cung ứng dịch vụ',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 21,
               'chunk_part': '4/6'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 22. Các địch vụ trung gian thanh toán và điều kiện '
             'cung ứng dịch vụ (tiếp theo)]\n'
             'Phó Tổng Giám đốc (Phó Giám đốc) và các cán bộ chủ chốt thực hiện Đề án cung ứng '
             'địch vụ trung gian thanh toán (gồm Trưởng phòng (ban) hoặc tương đương và các cán bộ '
             'kỹ thuật) có bằng cao đẳng trở lên về một trong các ngành kinh tế, quản trị kinh '
             'doanh, luật, công nghệ thông tin hoặc lĩnh vực chuyên môn đám nhiệm; \n'
             'đ) Có Bản thuyết minh giải pháp kỹ thuật phục vụ cho hoạt động cung ứng dịch vụ '
             'trung gian thanh toán đề nghị cấp Giấy phép được cấp có thâm quyền theo quy định tại '
             'Điều lệ của tổ chức phê duyệt đáp ứng yêu cầu đảm bảo an toàn hệ thống thông tin cấp '
             'độ 3 theo quy định của pháp luật;\n'
             '\n'
             'e) Đối với dịch vụ ví điện tử và dịch vụ hỗ trợ thu hộ, chi hộ cho các khách hàng có '
             'tài khoản tại nhiều ngân hàng, chỉ nhánh ngân hàng nước ngoài, tổ chức cung ứng dịch '
             'vụ phải được một tổ chức cũng ứng dịch vụ chuyên mạch tài chính và dịch vụ bù trừ '
             'điện tử được Ngân hàng Nhà nước cấp phép để thực hiện chuyển mạch giao dịch tài '
             'chính và xử lý bù trừ các nghĩa vụ phát sinh trong quá trình cung ứng dịch vụ trung '
             'gian thanh toán của tô chức; \n'
             '8) Đối với dịch vụ chuyên mạch tải chính, địch vụ bù trừ điện tử, ngoài các điều '
             'kiện quy định tại điểm a, điểm b, điểm c, điểm d và điểm đ khoản 2\n'
             'Điều này, tổ chức cung ứng dịch vụ phải: được một tổ chức thực hiện quyết toán kết '
             'quả bù trừ giữa các bên liên quan; có thỏa thuận kết nối với ít nhật 50 ngân hàng, '
             'chi nhánh ngân hàng nước ngoài có tổng vốn điều lệ trong năm liền kề trước năm nộp '
             'hồ sơ đề nghị cấp Giấy phép chiếm trên 65% tổng vốn điều lệ của các ngân hàng, chỉ '
             'nhánh ngân hàng nước ngoài trong hệ thống các tô chức tín dụng và ít nhất 20 tổ chức '
             '. cũng ứng dịch vụ trung gian thanh toán; có cơ sở hạ tầng thông tin đáp ứng tối '
             'thiểu theo yêu cầu về bảo đảm an toàn hệ thống thông tin cấp độ 4 theo quy định của '
             'pháp luật, đám bảo khả \n'
             ' năng tích hợp, kết nối được với hệ thống kỹ thuật của tổ chức tham gia có thỏa '
             'thuận kết nội; có hệ thông máy chủ thực hiện theo quy định pháp luật và đáp ứng năng '
             'lực xử lý tôi thiêu 10 triệu giao dịch thanh toán/ngày; \n'
             'Tổ chức tham gia không được kết nối quá 02 tổ chức cung ứng dịch vụ chuyên mạch tài '
             'chính, dịch vụ bù trừ điện tử;'},
 {'id': 42,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương IV DỊCH VỤ TRƯNG GIAN THANH TOÁN',
               'article': 'Điều 22',
               'article_title': 'Các địch vụ trung gian thanh toán và điều kiện cung ứng dịch vụ',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 22. Các địch vụ trung gian thanh '
                                  'toán và điều kiện cung ứng dịch vụ',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 21,
               'chunk_part': '5/6'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 22. Các địch vụ trung gian thanh toán và điều kiện '
             'cung ứng dịch vụ (tiếp theo)]\n'
             'năng tích hợp, kết nối được với hệ thống kỹ thuật của tổ chức tham gia có thỏa thuận '
             'kết nội; có hệ thông máy chủ thực hiện theo quy định pháp luật và đáp ứng năng lực '
             'xử lý tôi thiêu 10 triệu giao dịch thanh toán/ngày; \n'
             'Tổ chức tham gia không được kết nối quá 02 tổ chức cung ứng dịch vụ chuyên mạch tài '
             'chính, dịch vụ bù trừ điện tử;\n'
             '\n'
             'h) Đối với dịch vụ chuyển mạch tài chính quốc tế, tổ chức cung ứng dịch vụ phải có '
             'Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán chuyển mạch tài chính còn '
             'hiệu lực; được một tổ chức thực hiện quyết toán kết quả bù trừ giữa các bên liên '
             'quan; có quy định nội bộ về tiêu chuẩn lựa chọn kết nối các hệ thống thanh toán quốc '
             'tế để thực hiện chuyền mạch tài chính các giao dịch thanh toán quốc tế; có quy định '
             'nội bộ vê quy trình nghiệp vụ kỹ thuật đối với dịch vụ chuyển mạch tài chính quốc í '
             'tế đề nghị cấp phép và tổ chức vận hành hệ thống thanh toán. quốc tế kết nỗi với tổ '
             'chức cung ứng dịch vụ chuyển mạch tài chính quốc tế phải được thành lập và hoạt động '
             'hợp pháp ở nước ngoài.'},
 {'id': 43,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương IV DỊCH VỤ TRƯNG GIAN THANH TOÁN',
               'article': 'Điều 22',
               'article_title': 'Các địch vụ trung gian thanh toán và điều kiện cung ứng dịch vụ',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 22. Các địch vụ trung gian thanh '
                                  'toán và điều kiện cung ứng dịch vụ',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 21,
               'chunk_part': '6/6'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 22. Các địch vụ trung gian thanh toán và điều kiện '
             'cung ứng dịch vụ (tiếp theo)]\n'
             '3. Trong thời hạn tối đa 06 tháng kể từ ngày được Ngân hàng Nhà nước cấp Giấy phép '
             'hoạt động cung ứng dịch vụ trung gian thanh toán, tổ chức được cấp phép phải cung '
             'ứng dịch vụ trung gian thanh toán ra thị trường và chỉ được phép cung ứng dịch vụ '
             'trung gian thanh toán ra thị trường sau khi triển khai hệ thống kỹ thuật đáp ứng quy '
             'định tại điểm đ khoản 2 Điều này, đáp ứng quy định tại điểm g, điểm h khoản 2 Điều '
             'này đối với dịch vụ chuyển mạch tài chính, dịch vụ bù trừ điện tử, dịch vụ chuyển '
             'mạch tài chính quốc tế.'},
 {'id': 44,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương IV DỊCH VỤ TRƯNG GIAN THANH TOÁN',
               'article': 'Điều 23',
               'article_title': 'Nguyên tắc lập và gửi hồ sơ đề nghị cấp, cấp hại, sửa đổi, bỗ '
                                'sung, thu hồi Giấy phép hoạt động cung ứng dịch vụ trung gian '
                                'thanh toán',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 23. Nguyên tắc lập và gửi hồ sơ '
                                  'đề nghị cấp, cấp hại, sửa đổi, bỗ sung, thu hồi Giấy phép hoạt '
                                  'động cung ứng dịch vụ trung gian thanh toán',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 23,
               'chunk_part': '1/1'},
  'content': 'Điều 23. Nguyên tắc lập và gửi hồ sơ đề nghị cấp, cấp hại, sửa đổi, bỗ sung, thu hồi '
             'Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán\n'
             '\n'
             '1. Hồ sơ phải được lập bằng tiếng Việt. Trường hợp văn bản do cơ quan, tổ chức có '
             'thẩm quyền của nước ngoài cấp, công chứng hoặc chứng nhận phải được hợp pháp hóa '
             'lãnh sự theo quy định của pháp luật Việt Nam (trừ trường hợp được miễn hợp pháp hóa '
             'lãnh sự theo quy định pháp luật vê hợp pháp hóa lãnh sự) và dịch ra tiếng Việt.\n'
             '\n'
             '2. Các bản sao hồ sơ, tài liệu phải là bản sao có chứng thực hoặc bản sao cấp từ số '
             'gốc hoặc bản sao kèm bản chính để đối chiếu theo quy định của pháp luật, trường hợp '
             'hỗ sơ gửi trực tuyến thì thực hiện theo quy định về thủ tục hành chính trên môi '
             'trường điện tử.\n'
             '\n'
             '3. Sơ yếu lý lịch cá nhân tự lập được chứng thực chữ ký theo quy định của pháp '
             'luật.\n'
             '\n'
             '4. Hồ sơ được gửi qua đường bưu điện (dịch vụ bưu chính) hoặc nộp trục tiếp tới Bộ '
             'phận Một cửa Ngân hàng Nhà nước hoặc trực tuyển tại Công địch vụ công Ngân hàng Nhà '
             'nước hoặc Cổng dịch vụ công quốc gia. \n'
             '\n'
             '5. Tổ chức để nghị cấp, cấp lại, sửa đổi, bổ sung, thu hồi Giấy phép phải chịu hoàn '
             'toàn trách nhiệm trước pháp luật về tính chính xác, trung thực của các thông tin '
             'cung cấp.'},
 {'id': 45,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương IV DỊCH VỤ TRƯNG GIAN THANH TOÁN',
               'article': 'Điều 24',
               'article_title': 'Cấp Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 24. Cấp Giấy phép hoạt động cung '
                                  'ứng dịch vụ trung gian thanh toán',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 24,
               'chunk_part': '1/6'},
  'content': 'Điều 24. Cấp Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán'},
 {'id': 46,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương IV DỊCH VỤ TRƯNG GIAN THANH TOÁN',
               'article': 'Điều 24',
               'article_title': 'Cấp Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 24. Cấp Giấy phép hoạt động cung '
                                  'ứng dịch vụ trung gian thanh toán',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 24,
               'chunk_part': '2/6'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 24. Cấp Giấy phép hoạt động cung ứng dịch vụ trung '
             'gian thanh toán (tiếp theo)]\n'
             '1. Ngân hàng Nhà nước cấp Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán '
             'quy định tại khoản 1 Điều 22 Nghị định này cho tổ chức để nghị cấp Giấy phép hoạt '
             'động cung ứng dịch vụ trung gian thanh toán.'},
 {'id': 47,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương IV DỊCH VỤ TRƯNG GIAN THANH TOÁN',
               'article': 'Điều 24',
               'article_title': 'Cấp Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 24. Cấp Giấy phép hoạt động cung '
                                  'ứng dịch vụ trung gian thanh toán',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 24,
               'chunk_part': '3/6'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 24. Cấp Giấy phép hoạt động cung ứng dịch vụ trung '
             'gian thanh toán (tiếp theo)]\n'
             '2. Hồ sơ đề nghị cấp Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán:\n'
             '\n'
             'a) Đơn đề nghị cấp Giấy phép theo Mẫu số 07 ban hành kèm theo Nghị định này;\n'
             '\n'
             'b) Nghị quyết của Hội đồng thành viên, Hội đồng quản trị, Đại hội đồng cô đông, văn '
             'bản của người đại diện có thâm quyền của chủ sở hữu phù hợp với thâm quyền quy định '
             'tại Điều lệ công ty về việc thông qua Đề án cung ứng dịch vụ trung gian thanh toán '
             'và Bản thuyết minh giải pháp kỹ thuật;\n'
             '\n'
             'c) Đề án cung ứng dịch vụ trung gian thanh toán theo Mẫu số 08 ban hành kèm theo '
             'Nghị định này;\n'
             '\n'
             'd) Bán thuyết minh giải pháp kỹ thuật; \n'
             'đ) Hồ sơ về nhân sự: sơ yếu lý lịch theo Mẫu số 09 ban hành kèm theo Nghị định này, '
             'bản sao các văn bằng chứng minh năng lực, trình độ chuyên môn nghiệp vụ của người '
             'đại diện theo pháp luật, Tổng Giám đốc (Giám đốc), Phó Tổng Giám đốc (Phó Giám đốc) '
             'và các cán bộ chủ chốt thực hiện Đề án cung ứng dịch vụ trung gian thanh toán; phiếu '
             'lý lịch tư pháp hoặc văn bản có giá trị tương đương của người đại diện theo pháp '
             'luật, Tổng Giám đốc (Giám đốc) theo quy định của pháp luật (trước thời điểm nộp hồ '
             'sơ đề nghị cập Giấy phép không quá 06 tháng); văn bản của người đại diện có thâm '
             'quyền của đơn vị nơi người đại diện theo pháp luật, Tổng Giám đốc (Giám đốc) đã hoặc '
             'đang làm việc xác nhận chức vụ và thời gian đảm nhận chức vụ hoặc bản sao văn bản '
             'chứng minh chức vụ và thời gian đảm nhiệm chức vụ tại đơn vị của người đại diện theo '
             'pháp luật, Tổng Giám đốc (Giám đốc);\n'
             '\n'
             'e) Bản sao các tài liệu chứng minh tổ chức đề nghị cấp Giấy phép được thành lập và '
             'hoạt động hợp pháp, gồm: giấy phép thành lập hoặc giây chứng nhận đăng ký doanh '
             'nghiệp hoặc giầy tờ có giá trị tương đương; Điều lệ tô chức và hoạt động của tổ '
             'chức; giấy chứng nhận đầu tư của nhà đầu tư nước ngoài (nếu có); \n'
             ' ø) Văn bản cam kết và tài liệu chứng mỉnh của chủ sở hữu, thành viên sáng lập, cổ '
             'đông sáng lập của tổ chức về việc đảm bảo duy trì giá trị thực có của vốn điều lệ;\n'
             '\n'
             'h) Đối với dịch vụ chuyển mạch tài chính, dịch vụ bù trừ điện tử: phương án được một '
             'tổ chức thực hiện quyết toán kết quả bù trừ giữa các bên liên quan quy định tại Mẫu '
             'số 08 ban hành kèm theo Nghị định này; văn bản \n'
             'thỏa thuận kết nối với các tổ chức tham gia, có nội dung cam kết không được kết nối '
             'quá 02 tổ chức cung ứng dịch vụ. chuyển mạch tài chính, dịch vụ bù trừ điện tử; tài '
             'liệu chứng minh cơ sở hạ tầng thông tin, hệ thống máy chủ đáp ứng quy định tại điểm '
             'g khoản 2 Điều 22 Nghị định này;'},
 {'id': 48,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương IV DỊCH VỤ TRƯNG GIAN THANH TOÁN',
               'article': 'Điều 24',
               'article_title': 'Cấp Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 24. Cấp Giấy phép hoạt động cung '
                                  'ứng dịch vụ trung gian thanh toán',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 24,
               'chunk_part': '4/6'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 24. Cấp Giấy phép hoạt động cung ứng dịch vụ trung '
             'gian thanh toán (tiếp theo)]\n'
             'thỏa thuận kết nối với các tổ chức tham gia, có nội dung cam kết không được kết nối '
             'quá 02 tổ chức cung ứng dịch vụ. chuyển mạch tài chính, dịch vụ bù trừ điện tử; tài '
             'liệu chứng minh cơ sở hạ tầng thông tin, hệ thống máy chủ đáp ứng quy định tại điểm '
             'g khoản 2 Điều 22 Nghị định này; \n'
             '¡) Đối với dịch vụ chuyển mạch tài chính quốc tế: Quy định nội bộ về tiêu chuẩn lựa '
             'chọn hệ thống thanh toán quốc tế để kết nỗi thực hiện chuyên mạch tài chính các giao '
             'dịch thanh toán quốc tế; quy định nội bộ về quy trình nghiệp vụ với các biện pháp '
             'quản lý rủi ro đối với dịch vụ chuyển mạch tài chính quốc tế đề nghị cập phép; bản '
             'sao tài liệu chứng minh tổ chức vận hành hệ thông thanh toán quôc tế được thành lập '
             'và hoạt động hợp pháp ở nước ngoài do cơ quan có thâm quyền của quốc gia, vùng lãnh '
             'thổ nơi tổ chức thành lập hoặc đặt trụ sở chính câp; phương án được một tổ chức thực '
             'hiện quyết toán kết quả bù trừ giữa các bên liên quan quy định tại Mẫu số 08 ban '
             'hành kèm theo Nghị định này.'},
 {'id': 49,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương IV DỊCH VỤ TRƯNG GIAN THANH TOÁN',
               'article': 'Điều 24',
               'article_title': 'Cấp Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 24. Cấp Giấy phép hoạt động cung '
                                  'ứng dịch vụ trung gian thanh toán',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 24,
               'chunk_part': '5/6'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 24. Cấp Giấy phép hoạt động cung ứng dịch vụ trung '
             'gian thanh toán (tiếp theo)]\n'
             '3. Quy trình, thủ tục cấp Giấy phép \n'
             'Trường hợp nộp hồ sơ được gửi qua đường bưu điện (dịch vụ bưu chính) hoặc nộp trực '
             'tiếp tới Bộ phận Một cửa Ngân hàng Nhà nước, tổ chức đề nghị cấp Giấy phép gửi 02 bộ '
             'hồ sơ và 06 đĩa CD (hoặc 06 USB) lưu trữ bản quét Bộ hồ sơ đầy. đủ đề nghị cấp Giấy '
             'phép theo quy định tại khoản 2 Điều này. Căn cứ vào hồ sơ đề nghị cập Giấy phép, '
             'Ngân hàng Nhà nước phối hợp với các bộ, cơ quan liên quan tiến hành thẩm định hồ sơ '
             'trên cơ sở các điều kiện quy định tại khoản 2 Điều 22 Nghị định nảy.\n'
             '\n'
             'a) Trong thời hạn 05 ngày làm việc kể từ ngày nhận được hồ sơ đề nghị cấp Giấy phép, '
             'Ngân hàng Nhà nước có văn bản gửi tổ chức xác nhận đã nhận đầy đủ thành phần hồ sơ '
             'hợp lệ. Trường hợp thành phần hồ sơ không đầy đủ và hợp lệ, Ngân hàng Nhà nước có '
             'văn bản gửi tô chức yêu cầu bố sung, hoàn thiện thành phần hồ sơ. Thời gian bổ sung, '
             'hoàn thiện thành phần hồ sơ không tính vào thời gian thẩm định hồ sơ. \n'
             'Trong thời hạn 60 ngày kể từ ngày Ngân hàng Nhà nước có văn bản yêu cầu bổ sung '
             'thành phần hồ sơ nhưng tổ chức đề nghị cấp Giấy phép không gửi lại hồ sơ hoặc hồ sơ '
             'bỗổ sung của tổ chức không đáp ứng thành phần thì Ngân hàng Nhà nước có văn bản trả '
             'lại hỗ sơ cho tổ chức; \n'
             '\n'
             'b) Trong thời hạn 90 ngày làm việc kế từ ngày nhận đầy đủ thành phần hồ sơ hợp lệ, '
             'Ngân hàng Nhà nước tiến hành thầm định hồ SƠ. Trong thời hạn 60 ngày kể từ ngày Ngân '
             'hàng Nhà nước có văn bản yêu cầu giải trình, hoàn thiện hồ sơ mà tổ chức không gửi '
             'lại hồ sơ thì Ngân hàng Nhà nước có văn bản trả lại hồ sơ cho tổ chức. \n'
             'Trong thời hạn 90 ngày làm việc kế từ ngày nhận được hồ sơ bổ sung, hoàn thiện của '
             'tổ chức, Ngân hàng Nhà nước tiến hành thâm định, cấp Giấy phép theo quy định. Trường '
             'hợp không: cấp Giấy phép, Ngân hàng Nhà nước có văn bản trả lời tổ chức, trong đó '
             'nêu rõ lý do; \n'
             'lò) Kế từ ngày Ngân hàng Nhà nước nhận đầy đủ thành phần hồ sơ hợp lệ, tổ chức đề '
             'nghị câp Giấy phép được tự gửi bổ sung, hoàn thiện hồ sơ tối đa 02 lần; thời gian '
             'nộp hỗ sơ tự bổ sung, hoàn thiện của tổ chức tối đa không vượt quá 60 ngày kế từ '
             'ngày Ngân hàng Nhà nước có văn bản gửi tổ chức xác nhận đã nhận đầy đủ thành phần hồ '
             'sơ hợp lệ.'},
 {'id': 50,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương IV DỊCH VỤ TRƯNG GIAN THANH TOÁN',
               'article': 'Điều 24',
               'article_title': 'Cấp Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 24. Cấp Giấy phép hoạt động cung '
                                  'ứng dịch vụ trung gian thanh toán',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 24,
               'chunk_part': '6/6'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 24. Cấp Giấy phép hoạt động cung ứng dịch vụ trung '
             'gian thanh toán (tiếp theo)]\n'
             '4. Quy trình, thủ tục triển khai hoạt động sau khi được cấp Giấy phép \n'
             'Tối thiểu 30 ngày làm việc trước ngày dự kiến cung ứng dịch vụ trung gian thanh toán '
             'ra thị trường và không quá 06 tháng kê từ ngày được Ngân hàng Nhà nước cấp Giấy '
             'phép, tổ chức đã được Ngân hàng Nhà nước cập Giấy phép cung ứng dịch vụ trung gian '
             'thanh toán phải thông báo và cung câp tài liệu chứng minh cho Ngân hàng Nhà nước về '
             'việc: Hệ thống kỹ thuật đáp ứng điều kiện quy định tại điểm đ khoản 2 Điều 22 Nghị '
             'định này; bản sao Biên bản nghiệm thu kỹ thuật các dịch vụ trung gian thanh toán đã '
             'được cấp phép với một ngân hàng hợp tác đối với dịch vụ ví điện tử, dịch vụ hỗ trợ '
             'thu hộ, chi hộ và dịch vụ công thanh toán điện tử theo Mẫu số 10 ban hành kèm theo '
             'Nghị định này; tài liệu chứng minh được một tổ chức thực hiện quyết toán kết quả bù '
             'trừ giữa các bên liên quan đối với dịch vụ chuyển. mạch tài chính, dịch vụ bù trừ '
             'điện tử, dịch vụ chuyển mạch tài chính quốc tế đáp ứng điều kiện quy định tại điểm '
             'g, điểm h khoản 2 Điều 22 Nghị định này. \n'
             'Trong thời hạn 15 ngày làm việc kể từ ngày nhận được đầy đủ hồ sơ, tài liệu, Ngân '
             'hàng Nhà nước thực hiện kiểm tra thực tế tại tổ chức cung ứng dịch vụ trung gian '
             'thanh toán và có văn bản thông báo về việc đáp ứng hoặc không đáp ứng quy định tại '
             'điểm đ, điểm g, điểm h khoản 2 Điều 22 Nghị định này. Trường hợp không đáp ứng, Ngân '
             'hàng Nhà nước xem xét thực hiện thu hồi Giấy phép hoạt động cung ứng dịch vụ trung '
             'gian thanh toán theo quy định tại điểm e khoản 1 Điều 27 Nghị định này.\n'
             '\n'
             '5. Thời hạn Giấy phép Thời hạn hoạt động ghi trên Giấy phép là 10 năm tính từ ngày '
             'tổ chức được Ngân hàng Nhà nước cấp Giấy phép. Đối với dịch vụ chuyển mạch tài \n'
             'chính quốc. tế, thời hạn hoạt động không được vượt quá thời hạn hoạt động ghi trên '
             'Giấy phép cung ứng dịch vụ chuyển mạch tài chính.'},
 {'id': 51,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương IV DỊCH VỤ TRƯNG GIAN THANH TOÁN',
               'article': 'Điều 25',
               'article_title': 'Cấp lại Giấy phép hoạt động cung ứng dịch vụ trung gian thanh '
                                'toán',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 25. Cấp lại Giấy phép hoạt động '
                                  'cung ứng dịch vụ trung gian thanh toán',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 24,
               'chunk_part': '1/1'},
  'content': 'Điều 25. Cấp lại Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán \n'
             'Ngân hàng Nhà nước cấp lại Giấy phép hoạt động cung ứng dịch vụ trung gian thanh '
             'toán trong các trường hợp sau:\n'
             '\n'
             '1. Hết thời hạn Giấy phép \n'
             'Tối thiểu 60 ngày trước ngày hết thời hạn hoạt động ghi trên Giấy phép, tổ chức cung '
             'ứng dịch vụ trung gian thanh toán phải gửi Hồ sơ đề nghị cấp lại Giấy phép cho Ngân '
             'hàng Nhà nước. Trường hợp hồ sơ được gửi qua đường bưu điện (dịch vụ bưu chính) hoặc '
             'nộp trực tiếp tới Bộ phận Một cửa Ngân hàng Nhà nước, tổ chức cung ứng dịch vụ trung '
             'gian thanh toán gửi 03 bộ hồ sơ đề nghị cấp lại Giấy phép gồm: đơn đề nghị cấp lại '
             'Giấy phép theo Mẫu số 11 ban hành kèm theo Nghị định này, báo cáo tình hình thực '
             'hiện hoạt động theo Giấy phép kể từ ngày được cấp Giấy phép đến ngày nộp đơn đề nghị '
             'cấp lại Giấy phép và bản sao Giấy phép đang có hiệu lực tới Ngân hàng Nhà nước. \n'
             'Trong thời hạn 30 ngày làm việc kề từ ngày nhận được hồ sơ đề nghị cấp lại Giấy phép '
             'của tổ chức, Ngân hàng Nhà nước sẽ xem xét cấp lại Giấy phép hoặc có văn bản thông '
             'báo từ chối trong đó nêu rõ lý do; \n'
             'Thời hạn hoạt động phí trên Giấy phép là 10 năm tính từ ngày tổ chức được Ngân hàng '
             'Nhà nước cấp lại Giấy phép.\n'
             '\n'
             '2. Giấy phép bị mắt, bị rách, bị cháy hoặc bị hủy dưới hình thức khác \n'
             'Tổ chức cung ứng dịch vụ trung gian thanh toán gửi đơn đề nghị cấp lại Giấy phép '
             'theo Mẫu số I1 ban hành kèm theo Nghị định này trong đó nêu rỡ lý do. Trong thời hạn '
             '10 ngày làm việc kê từ ngày nhận được đơn đề nghị cấp lại Giấy phép của tổ chức, '
             'Ngân hàng Nhà nước xem xét cấp lại Giấy phép hoặc có văn bản thông báo từ chối trong '
             'đó nêu rõ lý do; \n'
             'Thời hạn hoạt động ghi trên Giấy phép được giữ nguyên như thời hạn hoạt động trên '
             'Giây phép đã bị mật, bị rách, bị cháy hoặc bị hủy dưới hình thức khác.'},
 {'id': 52,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương IV DỊCH VỤ TRƯNG GIAN THANH TOÁN',
               'article': 'Điều 26',
               'article_title': 'Sửa đỗi, bỗ sung Giấy phép hoạt động cung ứng dịch vụ trung gian '
                                'thanh toán',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 26. Sửa đỗi, bỗ sung Giấy phép '
                                  'hoạt động cung ứng dịch vụ trung gian thanh toán',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 27,
               'chunk_part': '1/1'},
  'content': 'Điều 26. Sửa đỗi, bỗ sung Giấy phép hoạt động cung ứng dịch vụ trung gian thanh '
             'toán\n'
             '\n'
             '1. Trường hợp thay đổi một trong các nội dung quy định trong Giấy phép hoạt động '
             'cung ứng dịch vụ trung gian thanh toán sau; tên tổ chức, địa điểm đặt trụ sở chính, '
             'ngừng cung, cấp một hoặc một số dịch vụ trung gian thanh toán đã được cấp phép, kết '
             'nối thêm hệ thống thanh toán quôc tế của tổ chức cung ứng dịch vụ chuyên mạch tài '
             'chính quốc tế: \n'
             '\n'
             'a) Tổ chức cung ứng dịch vụ trung gian thanh toán gửi 01 bộ hồ sơ đề nghị sửa đổi, '
             'bỗ sung Giấy phép gồm: đơn đề nghị sửa đổi, bổ sung Giấy phép hoạt động cung ứng '
             'dịch vụ trung gian thanh toán theo Mẫu số 12 ban hành kèm theo Nghị định này; bản '
             'sao Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán còn hiệu lực; bản sao '
             'giây chứng nhận đăng ký doanh nghiệp đã thay đổi tên doanh nghiệp, giấy chứng nhận '
             'đăng ký đầu tư đối với nhà đầu tư nước ngoài (nếu có); báo cáo tình hình thực hiện '
             'hoạt động cung ứng dịch vụ trung gian thanh toán kể từ ngày được cấp Giấy phép đến '
             'ngày nộp hồ sơ đề nghị sửa đổi, bố sung Giấy phép. \n'
             'Trường hợp đề nghị kết nối thêm hệ thống thanh toán quốc tế, tổ chức cung ứng địch '
             'vụ chuyển mạch tài chính quốc tế bổ sung thêm các tải liệu quy định tại điểm ¡ khoản '
             '2 Điều 24 Nghị định này;\n'
             '\n'
             'b) Ngân hàng Nhà nước tiếp nhận và xét tính hợp lệ của hồ sơ để nghị sửa đổi, bô '
             'sung Giấy phép trong thời hạn 05 ngày làm việc kể từ ngày nhận được hồ sơ và quyết '
             'định sửa đổi, bổ sung Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán '
             'trong thời hạn 30 ngày làm việc kể từ ngày nhận được đây đủ hồ sơ hợp lệ. Trường hợp '
             'từ chỗi sửa đổi, bỗổ sung Giấy phép, Ngân hàng Nhà nước có văn bản trả lời tổ chức '
             'trong đó nêu rõ lý do;\n'
             '\n'
             'c) Thời hạn hoạt động ghi trên Giấy phép sửa đổi, bổ sung không vượt quá thời hạn '
             'hoạt động ghi trên Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán;\n'
             '\n'
             'd) Đối với dịch vụ trung gian thanh toán đề nghị ngừng cung cấp, tổ chức cung ứng '
             'dịch vụ trung gian thanh toán phải ngừng cung câp dịch vụ sau khi Ngân hàng Nhà nước '
             'sửa đổi, bố sung Giấy phép và trong thời hạn 30 ngày kể từ ngày được Ngân hàng Nhà '
             'nước sửa đổi, bỗố sung Giấy phép, tổ chức cung ứng dịch vụ trung gian thanh toán '
             'phải gửi thông báo bằng văn bản tới các tổ chức và cá nhân liên quan để thanh lý hợp '
             'đồng và hoàn tất các nghĩa vụ, trách nhiệm giữa các bên liên quan theo quy định của '
             'pháp luật.\n'
             '\n'
             '2. Trường hợp thay đổi một trong các nội dung sau: người đại diện theo pháp luật, '
             'thay đôi vốn điều lệ, thay đổi tỷ lệ sở hữu vốn điều lệ của chủ sở hữu; thực hiện '
             'chia, tách, hợp nhất, sáp nhập, chuyển đổi doanh nghiệp, trong thời hạn hiệu lực của '
             'Giấy phép, tổ chức cung ứng dịch vụ trung gian thanh toán không phải làm thủ tục đề '
             'nghị sửa đối, bổ sung Giấy phép nhưng phải gửi thông báo cho Ngân hàng Nhà nước bằng '
             'văn bản và tài liệu chứng minh các thông tin liên quan (nếu có) trong thời hạn 30 '
             'ngày kể từ ngày có các thay đổi nêu trên.'},
 {'id': 53,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương IV DỊCH VỤ TRƯNG GIAN THANH TOÁN',
               'article': 'Điều 27',
               'article_title': 'Thu hồi Giấy phép hoạt động cung ứng dịch vụ trung gian thanh '
                                '(oán',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 27. Thu hồi Giấy phép hoạt động '
                                  'cung ứng dịch vụ trung gian thanh (oán',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 28,
               'chunk_part': '1/3'},
  'content': 'Điều 27. Thu hồi Giấy phép hoạt động cung ứng dịch vụ trung gian thanh (oán'},
 {'id': 54,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương IV DỊCH VỤ TRƯNG GIAN THANH TOÁN',
               'article': 'Điều 27',
               'article_title': 'Thu hồi Giấy phép hoạt động cung ứng dịch vụ trung gian thanh '
                                '(oán',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 27. Thu hồi Giấy phép hoạt động '
                                  'cung ứng dịch vụ trung gian thanh (oán',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 28,
               'chunk_part': '2/3'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 27. Thu hồi Giấy phép hoạt động cung ứng dịch vụ '
             'trung gian thanh (oán (tiếp theo)]\n'
             '1. Ngân hàng Nhà nước xem xét thực hiện thu hồi Giấy phép một trong các trường hợp '
             'sau đây: \n'
             '\n'
             'a) TỔ chức cung ứng dịch vụ trung gian thanh toán bị giải thể hoặc phá sản theo quy '
             'định của pháp luật; \n'
             '- b} Tổ chức cung ứng dịch vụ trung gian thanh toán có đơn đề nghị thu hôi Giây phép '
             'do châm dứt hoạt động cung ứng dịch vụ trung gian thanh toán được cập phép theo Mẫu '
             'sô 15 ban hành kèm theo Nghị định này;\n'
             '\n'
             'e) Khi có hiệu lực bản án, quyết định thi hành á án, quyết định xử phạt vỉ phạm hành '
             'chính của cơ quan nhà nước có thâm quyền, cơ quan thị hành án hình sự có nội dung '
             'yêu câu thu hồi Giấy phép của tổ chức cung ứng dịch vụ trung gian thanh toán hoặc có '
             'văn bản yêu cầu của cơ quan nhà nước có thâm quyên, cơ quan thi hành án hình sự đề '
             'nghị thu hồi Giấy phép của tổ chức cung ứng dịch vụ trung gian thanh toán; \n'
             'đ) Tổ chức cung ứng dịch vụ trung gian thanh toán sử dụng, lợi dụng tài khoản thanh '
             'toán, phương tiện thanh toán, dịch vụ thanh toán, dịch vụ trung gian thanh toán để '
             'đánh bạc, tổ chức đánh bạc, lừa đảo, kinh doanh trái pháp luật, rửa tiền, tài tợ '
             'khủng bố, tài trợ phổ biến vũ khí hủy diệt hàng loạt; \n'
             'đ) Sau thời hạn 03 tháng kế từ ngày Ngân chàng Nhà nước có văn bản thông báo cho tổ '
             'chức vi phạm một trong các điều kiện trong quá trình cung ứng dịch vụ quy định tại '
             'điểm a, điểm b, điểm c, điểm d, điểm e, điểm g, khoản 2 Điều 22, hệ thống kỹ thuật '
             'không đáp ứng quy định tại điểm đ khoản 2\n'
             'Điều 22 Nghị định này và phải thực hiện các biện pháp khắc phục nhưng tổ chức không '
             'khắc phục được;\n'
             '\n'
             'e) Tổ chức cung ứng dịch vụ trung gian thanh toán không đáp ứng quy định tại khoản 3 '
             'Điều 22 Nghị định này, không cung cấp được tài liệu chứng minh hoặc tài liệu chứng '
             'minh không đáp ứng quy định tại khoản 4 Điều 24 Nghị định này và sau thời hạn 03 '
             'tháng kể từ ngày Ngân hàng Nhà nước có văn bản thông báo yêu cầu tô chức thực hiện '
             'các biện pháp khắc phục nhưng tổ chức không khắc phục được; \n'
             'ø) Trong quá trình thanh tra, kiểm tra, giám sát hoạt động cung ứng dịch vụ trung '
             'gian thanh toán phát hiện trong thời hạn 06 tháng liên tục, tổ chức không thực hiện '
             'hoạt động cung ứng địch vụ trung gian thanh toán được cấp phép cho khách hàng;\n'
             '\n'
             'h) Tổ chức cung ứng dịch vụ trung gian thanh toán tái phạm việc báo cáo không trung '
             'thực số dư, sô lượng ví điện tử theo quy định.'},
 {'id': 55,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương IV DỊCH VỤ TRƯNG GIAN THANH TOÁN',
               'article': 'Điều 27',
               'article_title': 'Thu hồi Giấy phép hoạt động cung ứng dịch vụ trung gian thanh '
                                '(oán',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 27. Thu hồi Giấy phép hoạt động '
                                  'cung ứng dịch vụ trung gian thanh (oán',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 28,
               'chunk_part': '3/3'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 27. Thu hồi Giấy phép hoạt động cung ứng dịch vụ '
             'trung gian thanh (oán (tiếp theo)]\n'
             '2. Quy trình, thủ tục thu hồi Giấy phép\n'
             '\n'
             'a) Trường hợp tô chức bị giải thể hoặc phá, sản theo quy định tại điểm a khoản 1 '
             'Điều này, tổ chức có đơn đề nghị thu hồi Giấy phép theo Mẫu số 15 ban hành kèm theo '
             'Nghị định này gửi Ngân hàng Nhà nước trong thời hạn 07 ngày làm việc kể từ ngày '
             'thông qua Quyết định giải thể doanh nghiệp theo \n'
             ' quy định tại Luật Doanh nghiệp hoặc ngày nhận được Quyết định tuyên bố phá sản của '
             'Tòa án nhân dân theo quy định của pháp luật về phá sản. Sau 10 \n'
             'ngày làm việc kể từ ngày nhận được văn bản thông báo của tổ chức, Ngân hàng Nhà nước '
             'ra Quyết định thu hồi Giấy phép. \n'
             'Sau 10 ngày làm việc kế từ ngày Ngân hàng Nhà nước nhận được đơn đề nghị thu hôi '
             'Giấy phép do chấm dứt hoạt động cung ứng dịch vụ trung gian thanh toán của tô chức '
             'cung ứng dịch vụ trung gian thanh toán quy định tại điểm b khoản 1 Điều này, Ngân '
             'hàng Nhà nước ra quyết định thu hồi Giấy phép. \n'
             'Sau 20 ngày làm việc kế từ ngày phát sinh một trong các trường hợp quy định tại điểm '
             'c, điểm d, điểm đ, điểm e khoản 1 Điều này, Ngân hàng Nhà nước ra quyết định thu hồi '
             'Giấy phép.\n'
             '\n'
             'b) Khi tổ chức cung ứng dịch vụ trung gian thanh toán có dấu biệu vi phạm trường hợp '
             'nêu tại điểm g, điểm h khoản 1 Điều này, Ngân hàng Nhà nước xem xét thu hồi Giấy '
             'phép và ra thông báo đề nghị tổ chức giải trình. \n'
             'Sau 20 ngày làm việc kể từ ngày Ngân hàng Nhà nước ra thông báo nhưng tổ chức được '
             'cấp Giấy phép không có văn bản giải trình hoặc nội dung giải trình không xác đáng, '
             'Ngân hàng Nhà nước ra quyết định thu hồi Giấy phép.\n'
             '\n'
             '3. Ngay khi nhận được Quyết định của Ngân hàng Nhà nước về việc thu hồi Giấy phép, '
             'tổ chức bị thu hồi Giấy phép phải lập tức ngừng cung ứng dịch vụ trung gian thanh '
             'toán. \n'
             'Trong thời hạn 30 ngảy kế từ ngày có Quyết định của Ngân hàng Nhà nước về việc thu '
             'hồi Giấy phép, tổ chức bị thu hồi Giấy phép phải gửi thông báo bằng ` văn bản tới '
             'các tổ chức và cá nhân liên quan để thanh lý hợp đồng và hoàn tất các nghĩa vụ, '
             'trách nhiệm giữa các bên theo quy định pháp luật. \n'
             'Khi tổ chức bị thu hồi Giấy phép đã hoàn tất các nghĩa vụ, trách nhiệm giữa các bên '
             'theo quy định của pháp luật, sau thời hạn 03 nắm kể từ ngày tô chức bị thu hồi Giấy '
             'phép trong trường hợp quy định tại khoản 1 Điều này, tổ chức được đề nghị cấp Giấy '
             'phép hoạt động cung ứng dịch vụ trung gian thanh toán theo quy định tại Điều 22 Nghị '
             'định này. Quy trình, thủ tục, hỗ sơ đề nghị cấp Giấy phép thực hiện theo quy định '
             'tại Điều 23 và Điều 24 Nghị định này. \n'
             '_'},
 {'id': 56,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương V TRÁCH NHIỆM CỦA CÁC BÊN LIÊN QUAN',
               'article': 'Điều 28',
               'article_title': 'Trách nhiệm phối hợp cấp, cấp lại, sửa đỗi, bỗ sung và thu hồi '
                                'Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 28. Trách nhiệm phối hợp cấp, '
                                  'cấp lại, sửa đỗi, bỗ sung và thu hồi Giấy phép hoạt động cung '
                                  'ứng dịch vụ trung gian thanh toán',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 30,
               'chunk_part': '1/1'},
  'content': 'Điều 28. Trách nhiệm phối hợp cấp, cấp lại, sửa đỗi, bỗ sung và thu hồi Giấy phép '
             'hoạt động cung ứng dịch vụ trung gian thanh toán\n'
             '\n'
             '1. Trách nhiệm phối hợp trong quá trình thâm định\n'
             '\n'
             'a) Ngân hàng Nhà nước có văn bản gửi lẫy ý kiến của Bộ Công an Và các cơ quan liên '
             'quan về việc tuân thủ quy định pháp luật liên quan của tổ chức và người đại diện '
             'theo pháp luật của tổ chức trong thời gian hoạt động trước \n'
             ' khi được xem xét cấp, cấp lại, sửa đổi, bỗ sung và thu hồi Giấy phép theo quy định '
             'tại Điều 24, khoản 1 Điều 25, khoản 1 Điều 26, điểm đ, điểm g khoản 1\n'
             'Điều 27 Nghị định này;\n'
             '\n'
             'b) Trong thời hạn 15 ngày làm việc kể từ ngày nhận được văn bản của Ngân hàng Nhà '
             'nước, các bộ, cơ quan liên quan tại điểm a khoản này có ý kiến bằng văn bản gửi Ngân '
             'hàng Nhà nước; \n'
             'c©) Ngân hàng Nhà nước xem xét cấp, cấp lại, sửa đổi, bổ sung và thu hồi Giấy phép '
             'trên cơ sở hồ sơ và ý kiến của các bộ, cơ quan liên quan.\n'
             '\n'
             '2. Ngân hàng Nhà nước công bố công khai về việc cấp, cấp lại, sửa đổi, bổ sung và '
             'thu hôi Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán của tổ chức cung '
             'ứng dịch vụ trung gian thanh toán trên Công thông tin điện tử của Ngân hàng Nhà '
             'nước. Tổ chức cung ứng dịch vụ trung gian thanh toán phải công bế công khai Giấy '
             'phép, Quyết định sửa đổi, bỗổ sung, Quyết định thu hồi Giấy phép hoạt động trên '
             'trang thông tin điện tử chính thức của tổ chức đó để khách hàng biết.\n'
             '\n'
             'Chương V TRÁCH NHIỆM CỦA CÁC BÊN LIÊN QUAN'},
 {'id': 57,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương V TRÁCH NHIỆM CỦA CÁC BÊN LIÊN QUAN',
               'article': 'Điều 29',
               'article_title': 'Đảm bảo an toàn, bảo mật trong thanh toán',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 29. Đảm bảo an toàn, bảo mật '
                                  'trong thanh toán',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 31,
               'chunk_part': '1/1'},
  'content': 'Điều 29. Đảm bảo an toàn, bảo mật trong thanh toán\n'
             '\n'
             '1. Tổ chức cung ứng dịch vụ thanh toán, tổ chức cung ứng dịch vụ trung gian thanh '
             'toán phải áp dụng các biện pháp đảm bảo an toàn, bảo mật trong thanh toán theo quy '
             'định của pháp luật và được quy định các biện pháp đảm bảo an toàn trong hoạt động '
             'thanh toán phù hợp với các quy định pháp luật liên quan. Khách hàng sử dụng dịch vụ '
             'có nghĩa vụ tuân thủ các biện pháp đảm bảo an toàn, bảo mật trong thanh toán theo '
             'quy định và hướng dẫn của các tô chức cung ứng dịch vụ.\n'
             '\n'
             '2. Tổ chức cung ứng dịch vụ thanh toán, tổ chức cung ứng dịch vụ trung gian thanh '
             'toán phải có biện pháp kiểm tra, giám sát các đơn vị chấp nhận thanh toán nhằm phát '
             'hiện và giải quyết hoặc đề nghị cơ quan nhà nước có thâm quyền xử lý các hành vi vi '
             'phạm về hoạt động thanh toán theo quy định của pháp luật.\n'
             '\n'
             '3. Tổ chức cung ứng dịch vụ thanh toán, tổ chức cung ứng dịch vụ trung gian thanh '
             'toán phải có biện pháp quản trị rủi ro, phòng, chống việc sử dụng, lợi dụng việc '
             'cung ứng phương tiện thanh toán, dịch vụ thanh toán, dịch vụ trung gian thanh toán '
             'để đánh bạc, tổ chức đánh bạc, thực hiện giao dịch thanh toán khống (giao dịch thanh '
             'toán hàng hóa, dịch-vụ mà không phát sinh việc mua bán hàng hóa hoặc cung ứng dịch '
             'vụ) hoặc các hành vi vị phạm pháp luật khác. \n'
             '\n'
             '4. Tổ chức cung ứng dịch vụ trung gian thanh toán và các tổ chức, cá nhân liên quan '
             'có nghĩa vụ cung cấp đầy đủ thông tin, giấy tờ và chứng từ theo quy định của tổ chức '
             'cung ứng dịch vụ thanh toán khi thực hiện thanh toán ngoại tệ và thanh toán quốc tế '
             'quy định tại khoản 3, khoản 5 Điều 5 Nghị định này và chịu trách nhiệm trước pháp '
             'luật về tính xác thực của các loại giấy tờ, chứng từ đã xuất trình. Tổ chức cung ứng '
             'dịch vụ thanh toán phải áp dụng các biện pháp xem Xét, kiểm tra, lưu giữ các giấy tờ '
             'và chứng từ liên quan đến các giao dịch thực tế để đâm bảo việc thanh toán bằng '
             'ngoại tệ và thanh toán quốc tế thực hiện đúng mục đích và phù hợp với quy định của '
             'pháp luật.'},
 {'id': 58,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương V TRÁCH NHIỆM CỦA CÁC BÊN LIÊN QUAN',
               'article': 'Điều 30',
               'article_title': 'Quyền về thông tin, báo cáo',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 30. Quyền về thông tin, báo cáo',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 31,
               'chunk_part': '1/1'},
  'content': 'Điều 30. Quyền về thông tin, báo cáo\n'
             '\n'
             '1. Ngân hàng Nhà nước có quyền yêu cầu các tô chức cung ứng dịch vụ thanh toán và '
             'các tổ chức cung ứng dịch vụ trung gian thanh toán cung cấp thông tin có liên quan '
             'tới việc cung ứng dịch vụ theo định kỳ và đột xuất; việc cung cấp thông tin này thực '
             'hiện theo quy định của Ngân hàng Nhà nước.\n'
             '\n'
             '2. Tổ chức cung ứng dịch vụ thanh toán, tổ chức cung ứng dịch vụ trung gian thanh '
             'toán có quyên yêu cầu khách hàng sử dụng dịch vụ cung cấp thông tin có liên quan khi '
             'sử dụng dịch vụ của mình.'},
 {'id': 59,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương V TRÁCH NHIỆM CỦA CÁC BÊN LIÊN QUAN',
               'article': 'Điều 31',
               'article_title': 'Nghĩa vụ về thông fin, báo cáo',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 31. Nghĩa vụ về thông fin, báo '
                                  'cáo',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 31,
               'chunk_part': '1/1'},
  'content': 'Điều 31. Nghĩa vụ về thông fin, báo cáo\n'
             '\n'
             '1. Tổ chức cung ứng dịch vụ thanh toán, tổ chức cung ứng địch vụ trung gian thanh '
             'toán có nghĩa vụ báo cáo và cung cấp thông tin cho các cơ quan nhà nước có thâm '
             'quyền theo quy định của pháp luật.\n'
             '\n'
             '2. Tổ chức cung ứng dịch vụ thanh toán, tô chức cung ứng dịch vụ trung gian thanh '
             'toán có nghĩa vụ cung cấp thông tin về giao dịch và số dư trên tài khoản của chủ tài '
             'khoản thanh toán, giao địch và số dư trên ví điện tử của chủ ví điện tử theo thỏa '
             'thuận với chủ tài khoản thanh toán, chủ ví điện tử.'},
 {'id': 60,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 32',
               'article_title': 'Bảo mật thông tin',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 32. Bảo mật thông tin',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 32,
               'chunk_part': '1/1'},
  'content': 'Điều 32. Bảo mật thông tin\n'
             '1. Quyền từ chối cung cấp thông tin \n'
             'Tổ chức cung ứng dịch vụ thanh toán, tổ chức cung ứng dịch vụ trung gian thanh toán '
             'có quyên từ chối yêu cầu của tổ chức, cá nhân khác vê việc cung cấp thông tin liên '
             'quan đến tài khoản, giao dịch và số dư trên tài khoản, ví điện tử, giao dịch và sô '
             'dư trên ví điện tử trừ trường hợp theo yêu cầu của cơ quan nhà nước có thâm quyền '
             'theo quy định của pháp luật hoặc được sự chấp thuận của khách hàng.\n'
             '\n'
             '2. Nghĩa vụ bảo mật thông tin \n'
             'Tổ chức cung ứng dịch vụ thanh toán, tổ chức cung ứng dịch vụ trung gian thanh toán '
             'có trách nhiệm giữ bí mật các thông tin liên quan đến tài khoản, giao dịch và số dư '
             'trên tài khoản, ví điện tử, giao dịch và sô dư trên ví điện tử, trừ trường hợp được '
             'sự chấp thuận của khách hàng hoặc pháp luật có quy định khác. \n'
             '\n'
             'Chương VỊ HOẠT ĐỘNG GIÁM SÁT'},
 {'id': 61,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 33',
               'article_title': 'Giám sát các hệ thống thanh toán quan trọng trong nền kinh tế',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 33. Giám sát các hệ thống thanh '
                                  'toán quan trọng trong nền kinh tế',
               'topics': 'Hạn mức & Giao dịch, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 32,
               'chunk_part': '1/1'},
  'content': 'Điều 33. Giám sát các hệ thống thanh toán quan trọng trong nền kinh tế\n'
             '\n'
             '1. Ngân hàng Nhà nước ban hành các quy định về giám sát và thực hiện giám sát các hệ '
             'thống thanh toán quan trọng trong nền kinh tế để góp phần tăng cường đảm bảo sự hoạt '
             'động ôn định, an toàn của các hệ thống thanh toán quan trọng trong nên kinh tế.\n'
             '\n'
             '2. Ngân hàng Nhà nước xác định các hệ thống thanh toán quan trọng chịu sự giám sát '
             'của Ngân hàng Nhà nước. Ngân hàng Nhà nước giám sát các hệ thống thanh toán quan '
             'trọng bằng các biện pháp giám sát từ xa, kiểm tra tại chỗ và các biện pháp khác khi '
             'cần thiết theo nguyên tắc không làm cản trở hoạt động bình thường của hệ thống và tô '
             'chức vận hành hệ thống thanh toán quan trọng.\n'
             '\n'
             '3. Ngân hàng Nhà nước có quyền yêu cầu tổ chức vận hành, ngân hàng quyết toán, các '
             'thành viên tham gia hệ thống thanh toán quan trọng cung cấp thông tin liên quan đến '
             'hệ thống.\n'
             '\n'
             '4. Tổ chức vận hành, ngân hàng quyết toán, thành viên tham gia hệ thống thanh toán '
             'quan trọng có trách nhiệm cung cấp thông tin liên quan đến hệ thống theo yêu cầu của '
             'Ngân hàng Nhà nước.\n'
             '\n'
             '5. Các tổ chức vận hành, ngân hàng quyết toán hệ thống thanh toán quan trọng có '
             'trách nhiệm chấp hành các quy định và thực hiện các yêu cầu giám sát của Ngân hàng '
             'Nhà nước; ban hành các quy định nội bộ về kiểm soát rủi ro và đâm bảo sự hoạt động '
             'liên tục của hệ thông.'},
 {'id': 62,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 34',
               'article_title': 'Giám sát hoạt động cung ứng dịch vụ thanh toán',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 34. Giám sát hoạt động cung ứng '
                                  'dịch vụ thanh toán',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 33,
               'chunk_part': '1/1'},
  'content': 'Điều 34. Giám sát hoạt động cung ứng dịch vụ thanh toán\n'
             '\n'
             '1. Ngân hàng Nhà nước xác định phạm vi, ban hành quy định về giám sát và thực hiện '
             'giám sát hoạt động cung ứng dịch vụ thanh toán.\n'
             '\n'
             '2. Ngân hàng Nhà nước giám sát hoạt động cung ứng dịch vụ thanh toán bằng các biện '
             'pháp giám sát từ xa, kiểm tra tại chỗ và các biện pháp khác khi cần thiết theo '
             'nguyên tắc không làm cản trở hoạt động bình thường của các tổ chức cung ứng dịch vụ '
             'thanh toán.\n'
             '\n'
             '3. Ngân hàng Nhà nước có quyền yêu cầu tổ chức cung ứng địch vụ thanh toán cung cập '
             'thông tin liên quan đên hoạt động cung ứng dịch vụ thanh toán.\n'
             '\n'
             '4. Tổ chức cung ứng dịch vụ thanh toán có trách nhiệm cung cấp. thông tin liên quan '
             'đến hoạt động cung ứng dịch vụ thanh toán theo yêu câu của Ngân hàng Nhà nước và có '
             'trách nhiệm chấp hành các quy định, thực hiện các yêu cầu giám sát của Ngân hàng Nhà '
             'nước.'},
 {'id': 63,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 35',
               'article_title': 'Giám sát hoạt động cung ứng dịch vụ trung gian thanh toán',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 35. Giám sát hoạt động cung ứng '
                                  'dịch vụ trung gian thanh toán',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 33,
               'chunk_part': '1/1'},
  'content': 'Điều 35. Giám sát hoạt động cung ứng dịch vụ trung gian thanh toán\n'
             '\n'
             '1. Ngân hàng Nhà nước ban hành quy định về giám sát và thực hiện giám sát hoạt động '
             'cung ứng dịch vụ trung gian thanh toán của tổ chức được Ngân hàng Nhà nước cập Giấy '
             'phép.\n'
             '\n'
             '2. Ngân hàng Nhà nước giám sát hoạt động cung ứng dịch vụ trung gian thanh toán '
             'bằng. các biện pháp giám sát từ xa, kiểm tra tại chỗ và các biện pháp khác khi cần '
             'thiết theo nguyên tắc không làm cản trở hoạt động bình thường của các tổ chức cung '
             'ứng dịch vụ trung gian thanh toán.\n'
             '\n'
             '3. Ngân hàng Nhà nước có quyền yêu cầu tổ chức cung ứng dịch vụ trung gian thanh '
             'toán cung cấp thông tin liên quan đến hoạt động cung ứng địch vụ trung gian thanh '
             'toán.\n'
             '\n'
             '4. Tổ chức cung ứng dịch vụ trung gian thanh toán có trách nhiệm cung cấp thông tin '
             'liên quan đến hoạt động cung ứng dịch vụ trung gian thanh toán theo yêu cầu của Ngân '
             'hàng Nhà nước và có trách nhiệm chấp hành các quy định, thực hiện các yêu câu giám '
             'sát của Ngân hàng Nhà nước.\n'
             '\n'
             '5. Tổ chức cung ứng dịch vụ trung gian thanh toán phải xây dựng và cung cấp công cụ '
             'giám sát trực tuyến đề Ngân hàng Nhà nước giám sát hoạt động cung ứng dịch vụ trung '
             'gian thanh toán trước khi chính thức cung ứng dịch vụ ra thị trường cho khách hàng.\n'
             '\n'
             '6. Ngân hàng hợp tác với tô chức cung ứng dịch vụ trung gian thanh toán có trách '
             'nhiệm báo cáo, cung cấp thông tin liên quan đến hoạt động hợp tác cung ứng dịch vụ '
             'trung gian thanh toán; thông tin dữ liệu liên quan đến việc mở và quản lý tài khoản '
             "đảm báo thanh toán cho tổ chức cung ứng dịch vụ trung gian ' thanh toán đối với dịch "
             'vụ ví điện tử, dịch vụ hỗ trợ thu hộ, chỉ hộ theo yêu cầu của Ngân hàng Nhà nước. \n'
             '. ChươngVH ĐIEU KHOẢN THỊ HÀNH'},
 {'id': 64,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 36',
               'article_title': 'Quy định chuyển tiếp',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 36. Quy định chuyển tiếp',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 34,
               'chunk_part': '1/1'},
  'content': 'Điều 36. Quy định chuyển tiếp\n'
             '\n'
             '1. Ngân hàng thương mại, chỉ nhánh ngân hàng nước ngoài đã tham gia các hệ thống '
             'thanh toán quốc tế trước ngày Nghị định này có hiệu lực thi hành được tiếp tục tham '
             'gia các hệ thống thanh toán quốc tế đó. Trong thời hạn 24 tháng kê từ ngày Nghị định '
             'này có hiệu lực thi hành, ngân hàng thương mại, chỉ nhánh ngân hàng nước ngoài có '
             'trách nhiệm đáp ứng điều kiện kinh doanh quy định tại Điều 21 Nghị định này và hoàn '
             'thành thủ tục đề nghị chấp thuận tham gia hệ thống thanh toán quốc tế theo quy định '
             'của Thống đốc Ngân hàng Nhà nước; quá thời hạn nêu trên, ngân hàng thương mại, chỉ '
             'nhánh ngân hàng nước ngoài phải chấm dứt tham gia các hệ thống thanh toán quốc tế '
             'không ghi trong Giấy phép. \n'
             '\n'
             '2. Tổ chức cung ứng dịch vụ trung gian thanh toán được cấp phép dịch vụ hỗ trợ '
             'chuyên tiên điện tử (theo quy định tại khoản 5 Điều 3 Thông tư số 39/2014/T1-NHNN '
             'ngày 11 tháng 12 năm 2014 của Ngân hàng Nhà nước hướng dẫn về dịch vụ trung gian '
             'thanh toán) ghi trong Giấy phép hoạt động cung ứng địch vụ trung gian thanh toán '
             'trước ngày Nghị định này có hiệu lực thi hành thì dịch vụ này được tiếp tục thực '
             'hiện theo thỏa thuận giữa các bên. \n'
             '..3. Tổ chức cung ứng dịch vụ chuyển mạch tài chính được cấp phép trước ngày Nghị '
             'định này có hiệu lực thi hành đã kết nối với các hệ thống thanh toán quốc tế thì '
             'được tiếp tục kết nối với các hệ thống thanh toán quốc tế đó. Trong thời hạn 24 '
             'tháng kế từ ngày Nghị định này có hiệu lực thị bành, tô chức cung ứng dịch vụ chuyển '
             'mạch tài chính được cấp phép có trách nhiệm đáp ứng điều kiện kinh doanh và hoàn '
             'thành thủ tục đề nghị cấp Giấy phép hoạt động cung ứng dịch vụ chuyển mạch tài chính '
             'quốc tế theo quy định tại Nghị định này; quá thời hạn nêu trên, tổ chức phải chấm '
             'đứt kết nỗi với các hệ thống thanh toán quốc tế không ghi trong Giấy phép hoạt động '
             'cung ứng dịch vụ chuyển mạch tài chính quốc tế.\n'
             '\n'
             '4. Doanh nghiệp cung ứng dịch vụ bưu chính công ích được tiếp tục thực hiện cung ứng '
             'địch vụ thanh toán không qua tài khoản thanh toán theo quy định của pháp luật. Trong '
             'thời hạn 24 tháng kể từ ngày Nghị định này có hiệu lực thi hành, doanh nghiệp cung '
             'ứng dịch vụ bưu chính công ích có trách nhiệm đáp ứng điều kiện kinh doanh và hoàn '
             'thành thủ tục đề nghị chấp thuận hoạt động cung ứng dịch vụ thanh toán không qua tài '
             'khoản thanh toán của khách hàng theo quy định tại Nghị định này; quá thời hạn nêu '
             'trên, doanh nghiệp phải chấm dứt hoạt động cung ứng dịch vụ thanh toán.\n'
             '\n'
             '5. Tổ chức cung ứng dịch vụ trung gian thanh toán được cấp phép cung ứng dịch vụ '
             'trung gian thanh toán trước ngày Nghị định này có hiệu lực thì hành được tiếp tục '
             'thực hiện theo Giấy phép đã được Ngân hàng Nhà nước cấp cho đến khi Giấy phép hết '
             'thời hạn, trừ trường hợp tổ chức đề nghị cấp Giấy phép theo quy định tại Nghị định '
             'này.'},
 {'id': 65,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 37',
               'article_title': 'Hiệu lực thi hành',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 37. Hiệu lực thi hành',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 35,
               'chunk_part': '1/1'},
  'content': 'Điều 37. Hiệu lực thi hành\n'
             '1. Nghị định này có hiệu lực thi hành từ ngày 01 tháng 7 năm 2024.\n'
             '\n'
             '2. Nghị định này thay thế cho Nghị định số 101/2012/NĐ-CP ngày 22 tháng 11 năm 2012 '
             'của Chính phủ vê thanh toán không dùng tiền mặt; Nghị định số 80/2016/NĐ-CP ngày 01 '
             'tháng 7 năm 2016 của Chính phủ sửa đôi, bổ sung một số điều của Nghị định số '
             '101/2012/NĐ-CP ngày 22 tháng 11 năm 2012 của Chính phủ về thanh toán không dùng tiền '
             'mặt và bãi bỏ Điều 3 của Nghị định số 16/2019/NĐ-CP ngày 01 tháng 02 năm 2019 của '
             'Chính phủ sửa đổi, bổ sung một số điều của các Nghị định quy định về điều kiện kinh '
             'doanh thuộc phạm vi quản lý nhà nước của Ngân hàng Nhà nước.'},
 {'id': 66,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 38',
               'article_title': 'Trách nhiệm thi hành',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 38. Trách nhiệm thi hành',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 36,
               'chunk_part': '1/7'},
  'content': 'Điều 38. Trách nhiệm thi hành \n'
             'Các Bộ trưởng, Thủ trưởng cơ quan ngang bộ, Thủ trưởng cơ quan thuộc Chính phủ; Chủ '
             'tịch Ủy ban nhân dân tỉnh, thành phố trực thuộc trung ương; Chủ tịch Hội đồng quản '
             'trị, Chủ tịch Hội đồng thành viên và Tổng Giám đốc (Giám đốc) tổ chức tín dụng, chỉ '
             'nhánh ngân hàng nước ngoài; Chủ tịch Hội đồng quản trị, Chủ tịch Hội đồng thành '
             'viên, Chủ tịch công ty và Tổng Giám đốc (Giám đốc) Doanh nghiệp cung ứng dịch vụ bưu '
             'chính công ích và tổ chức, cá nhân có liên quan chịu trách nhiệm thi hành Nghị định '
             'này. \n'
             'Nơi nhận: TM. CHÍNH PHỦ \n'
             '- Ban Bí thư Trung ương Đảng, š \n'
             '- Thủ tướng, các Phó Thủ tướng Chính phủ; \n'
             '- Các bộ, cơ quan ngang bộ, cơ quan thuộc Chính phủ; \n'
             '- HĐND, UBND các tỉnh, thành phố trực thuộc trung ương; Ệ \n'
             '- Văn phòng Trung ương và các Ban của Đảng; : \n'
             '- Văn phòng Tổng Bí thư; \n'
             '- Văn phòng Chủ tịch nước; \n'
             '- Hội đồng Dân tộc và các Ủy ban của Quốc hội; \n'
             '- Văn phòng Quốc hội; \n'
             '- Tòa án nhân dân tối cao; \n'
             '- Viện kiểm sát nhân dân tối cao; \n'
             '- Kiểm toán nhà nước; . \n'
             '- Ủy ban Giám sát tài chính quốc gia; \n'
             '- Ngân hàng Chính sách xã hội; \n'
             '- Ngân hàng Phát triển Việt Nam; \n'
             '- Ủy ban trung ương Mặt trận Tô quốc Việt Nam; \n'
             '- Cơ quan trung ương của các đoàn thể; \n'
             '- VPCP: BTCN, các PCN, Trợ lý TTg, TGĐÐ Cổng TTĐT, các Vụ, Cục, đơn vị trực thuộc, '
             'Công báo; \n'
             '- Lưu: VT, KTTH (2). 3ƒ0 \n'
             'Lễ Minh Khái \n'
             ' Mu số 01 \n'
             '¬- C MÁU VÁN BẢN HỖ SƠ, THỦ TỤC \n'
             '\\ Mác A\\ 077,2 Mẫu đơn đề nghị mở tài khoản thanh toán tại Ngân hàng Nhà nước \n'
             'Mẫu số 02 \n'
             'Mẫu đơn đề nghị đóng tài khoản thanh toán tại Ngân hàng Nhà nước \n'
             'Mẫu số 03 \n'
             'Mẫu đơn đề nghị cung ứng dịch vụ thanh toán không qua tài khoản \n'
             'thanh toán của khách hàng \n'
             'Mẫu số 04 \n'
             'Mẫu Quyết định về việc chấp thuận hoạt động cung ứng dịch vụ thanh toán không qua '
             'tài khoản thanh toán của khách hàng \n'
             'Mẫu số 05 \n'
             'Mẫu đơn đề nghị thu hồi văn bản chấp thuận hoạt động cung ứng dịch vụ thanh toán '
             'không qua tài khoản thanh toán của khách hàng \n'
             'Mẫu số 06 \n'
             'Mẫu Quyết định về việc thu hồi văn bản chấp thuận hoạt động cung \n'
             'ứng dịch vụ thanh toán không qua tài khoản thanh toán của khách hàng \n'
             'Mẫu số 07 \n'
             'Mẫu đơn đề nghị cấp Giấy phép hoạt động cung ứng dịch vụ trung \n'
             'Mẫu số 08 \n'
             'Mẫu số 09 \n'
             'Nội dung để án cung ứng địch vụ trung gian thanh toán \n'
             'gian thanh toán \n'
             'Mẫu sơ yếu lý lịch \n'
             'Mẫu số 10 \n'
             'Mẫu biên bản nghiệm thu kỹ thuật \n'
             'Mẫu số 1] \n'
             'Mẫu số 12 \n'
             'Mẫu đơn đề nghị cấp lại Giấy phép hoạt động cung ứng địch vụ trung gian thanh toán \n'
             'Mẫu đơn đề nghị sửa đổi, bỗ sung Giấy phép hoạt động cung ứng địch vụ trung gian '
             'thanh toán — \n'
             'Mẫu số 13 \n'
             '——] \n'
             'Mẫu Giấy phép cắp/cấp lại hoạt động cung ứng dịch vụ trung gian thanh toán \n'
             'Mẫu số 14 \n'
             '= \n'
             'Mẫu Quyết định về việc chấp thuận sửa đổi, bỗ sung nội dung Giấy | phép hoạt động '
             'cung ứng dịch vụ trung gian thanh toán \n'
             'Mẫu số 15'},
 {'id': 67,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 38',
               'article_title': 'Trách nhiệm thi hành',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 38. Trách nhiệm thi hành',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 36,
               'chunk_part': '2/7'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 38. Trách nhiệm thi hành (tiếp theo)]\n'
             'Mẫu số 1] \n'
             'Mẫu số 12 \n'
             'Mẫu đơn đề nghị cấp lại Giấy phép hoạt động cung ứng địch vụ trung gian thanh toán \n'
             'Mẫu đơn đề nghị sửa đổi, bỗ sung Giấy phép hoạt động cung ứng địch vụ trung gian '
             'thanh toán — \n'
             'Mẫu số 13 \n'
             '——] \n'
             'Mẫu Giấy phép cắp/cấp lại hoạt động cung ứng dịch vụ trung gian thanh toán \n'
             'Mẫu số 14 \n'
             '= \n'
             'Mẫu Quyết định về việc chấp thuận sửa đổi, bỗ sung nội dung Giấy | phép hoạt động '
             'cung ứng dịch vụ trung gian thanh toán \n'
             'Mẫu số 15 \n'
             'Mẫu đơn đề nghị thu hồi Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán \n'
             'Mẫu Quyết định về việc thu hồi Giấy phép hoạt động cung ứng địch | vụ trung gian '
             'thanh toán \n'
             '\n'
             ' Mẫu số 01 \n'
             'TÊN TÔ CHỨC MỞ TÀI KHOẢN CỘNG HOÀ XÃ HỘI CHỦ NGHĨA VIỆT NAM ————— Độc lập - Tự do - '
             'Hạnh phúc \n'
             'Số:....... ...., Hgày....thẳng..... HĂm..... \n'
             'ĐƠN ĐÈ NGHỊ MỞ TÀI KHOẢN THANH TOÁN TẠI NGÂN HÀNG NHÀ NƯỚC \n'
             'Kính gửi:.................... \n'
             '\n'
             ' Số định danh cá nhân (đối với thẻ căn cước, thể Căn cước công dân) hoặc số chứng '
             'minh nhân dân hoặc số hộ chiếu:.......... ; Ngày cấp:...... Có giá trị đến:.. \n'
             'Họ và tên Kế toán trưởng (hoặc người phụ trách kế toán hoặc người kiểm soát chứng từ '
             'giao dịch với Ngân hàng Nhà nước): ....................... co eeteeeeei \n'
             "Ngày, tháng, năm sinh'”:.........................- Giới tính "
             "(Nam/Nữ)':..................... \n"
             'Số định danh cá nhân (đối với thẻ căn cước, thẻ căn cước công dân) hoặc số chứng '
             'minh nhân dân hoặc sô hộ chiếu:......... ; Ngày cấp:....., Có giá trị đến:....; jNơi '
             'cấp:................. chinh \n'
             'Quyết định bỗ nhiệm số:........................ ngày........ tháng........... '
             'năm............ \n'
             'Đề nghị mở tài khoản thanh toán fại:.............................. Ăn re \n'
             'Loại tiền tệ: n VND USD r1 Loại khác........ \n'
             'Chúng tôi cam kết: \n'
             '- Những thông tin trên đây là đúng sự thật và hoàn toàn chịu trách nhiệm về tính '
             'chính xác, đúng sự thật của các giấy tờ trong hồ sơ mở tài khoản thanh toán đính '
             'kèm. \n'
             '- Chấp hành đúng, đầy đủ các quy định của pháp luật hiện hành về mở và sử dụng tài '
             'khoản thanh toán tại Ngân hàng Nhà nước và xin chịu trách nhiệm đối với mọi vấn đề '
             'phát sinh trong trường hợp chúng tôi không thực hiện đúng, đầy đủ quy định về mở và '
             'sử dụng tài khoản do Ngân hàng Nhà nước quy định. \n'
             '- Có văn bản (kèm các giấy tờ liên quan) gửi Ngân hàng Nhà nước khi có bắt kỳ sự '
             'thay đỗi nào về thông tin mở tài khoản thanh toán hoặc mẫu dấu, mẫu chữ ký đã đăng '
             'ký sử dụng với Ngân hàng Nhà nước. \n'
             '` NGƯỜI ĐẠI DIỆN HỢP PHÁP Hồ sơ đính kèm: (Kỹ, ghi rõ họ tên, đóng dấu) 1) Bản đăng '
             'ký mẫu dấu và chữ ký kèm văn bản ủy. quyền (nếu có); 2) Quyết định '
             'số............................. \n'
             ' Á \n'
             'PHÀN DÀNH CHO NGÂN HÀNG NHÀ NƯỚC \n'
             'Sau khi kiểm soát và xác định hỗ sơ mở tài khoản thanh toán của............. .. là '
             'đầy đủ và hợp lệ, Ngân hàng Nhà nước .....................................'},
 {'id': 68,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 38',
               'article_title': 'Trách nhiệm thi hành',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 38. Trách nhiệm thi hành',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 36,
               'chunk_part': '3/7'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 38. Trách nhiệm thi hành (tiếp theo)]\n'
             '` NGƯỜI ĐẠI DIỆN HỢP PHÁP Hồ sơ đính kèm: (Kỹ, ghi rõ họ tên, đóng dấu) 1) Bản đăng '
             'ký mẫu dấu và chữ ký kèm văn bản ủy. quyền (nếu có); 2) Quyết định '
             'số............................. \n'
             ' Á \n'
             'PHÀN DÀNH CHO NGÂN HÀNG NHÀ NƯỚC \n'
             'Sau khi kiểm soát và xác định hỗ sơ mở tài khoản thanh toán của............. .. là '
             'đầy đủ và hợp lệ, Ngân hàng Nhà nước ..................................... \n'
             'đồng ý ý mở ở tài khoản thanh toán số: ......................... 0.00... .... Ngày '
             'bắt đầu hoạt động:......................... ch HH ke TRƯỞNG PHÒNG KÉ TOÁN GIÁM ĐÓC '
             'SỞ GIAO DỊCH (Ký, ghi rõ họ tên) _ NGÂN HÀNG NHÀNƯỚC/ GIÁM ĐÓC NGÂN HÀNG NHÀ NƯỚC '
             'CHI NHÁNH............ \n'
             '(Ký, ghi rõ họ tên, đóng dấu) \n'
             'Ghi chú: Trường hợp Ngân hàng Nhà nước nơi tiếp nhận Đơn đề nghị mở tài khoản thanh '
             'toán đủ điều kiện truy cập, khai thác thông tin trên Cơ sở đữ liệu quốc gia về dân '
             'cư thì không bắt buộc tổ chức mở tài khoản thanh toán phải kê khai các thông tin '
             'đánh dấu (*). \n'
             ' : ___ BẢN ĐĂNG KÝ MẪU DÁU, CHỮ KÝ . SỬ DỤNG TÀI KHOÁN THANH TOÁN TẠI NGÂN HÀNG NHÀ '
             'NƯỚC \n'
             'z \n'
             '(Đính kèm Đơn đà nghị mở tài khoản số...... ngày... của......) \n'
             'Đăng ký mẫu chữ ký và mẫu dấu sẽ sử dụng trên các chứng từ giao dịch với Ngân hàng '
             'Nhà nước......................... tt nhe như sau:'},
 {'id': 69,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 38',
               'article_title': 'Trách nhiệm thi hành',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 38. Trách nhiệm thi hành',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 36,
               'chunk_part': '4/7'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 38. Trách nhiệm thi hành (tiếp theo)]\n'
             '1. Mẫu chữ ký \n'
             'Người đăng ký mẫu chữ ký —_—\n'
             '\n'
             '1. Người đại diện hợp pháp của chủ tài khoẩn:\n'
             '\n'
             '1.1. Người đại diện hợp pháp thứ nhất: \n'
             'Mẫu chữ ký thứ hai \n'
             'Số định danh cá nhân (đối với thẻ căn cước, thẻ căn cước công dân)/số chứng \n'
             'Số định danh cá nhân (đối với thẻ căn cước, thẻ căn cước công dân)/số chứng'},
 {'id': 70,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 38',
               'article_title': 'Trách nhiệm thi hành',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 38. Trách nhiệm thi hành',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 36,
               'chunk_part': '5/7'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 38. Trách nhiệm thi hành (tiếp theo)]\n'
             '1.3. Người đại diện hợp pháp thứ ba: \n'
             'Số định danh cá nhân (đối với thẻ căn cước, thẻ căn cước công dân)/số chứng minh '
             'nhân dân/số hộ chiếu:.................... Ngày 00000... .... Có giá trị đến: '
             '.........................cccvvccceccrcee\n'
             '\n'
             '2. Kế toán trưởng (hoặc người phụ trách kế toán, người kiếm soát chứng từ giao dịch '
             'với Ngân hàng Nhà nước) và người được ủy quyền\n'
             '\n'
             '2.1. Kế toán trưởng (hoặc người phụ trách kế toán, người kiểm soát chứng từ giao '
             'dịch với Ngân hàng Nhà nước): \n'
             'Số định danh cá nhân (đối với thẻ căn cước, thẻ căn cước công dân)/số chứng minh '
             'nhân dân/số hộ chiêu:...................- \n'
             'Số định danh cá nhân (đối với thẻ căn cước, thẻ căn cước công dân)/sô chứng minh '
             'nhân dân/số hộ chiếu:.................... \n'
             'Số định danh cá nhân (đối với thẻ căn cước, thể căn cước công dân)/sô chứng minh '
             'nhân dân/số hộ chiêu:................... \n'
             'Văn bản ủy quyền SỐ.. Thời hạn ủ ủy quyền: đe'},
 {'id': 71,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 38',
               'article_title': 'Trách nhiệm thi hành',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 38. Trách nhiệm thi hành',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 36,
               'chunk_part': '6/7'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 38. Trách nhiệm thi hành (tiếp theo)]\n'
             '2. Mẫu dấu \n'
             'Mẫu dấu thứ nhất Mẫu dẫu thứ hai \n'
             '"- ngày..... tháng..... nĂm..... NGƯỜI ĐẠI DIỆN HỢP PHÁP (Ký, ghi rõ họ tên, đóng '
             'đâu) \n'
             'PHÀN DÀNH CHO NGÂN HÀNG NHÀ NƯỚC \n'
             'Sau khi xác định mẫu chữ ký và mẫu dấu của............................. là hợp lệ, '
             'Ngân hàng Nhà nước...........................--- chấp thuận với Bản đăng ký mẫu dâu, '
             'chữ ký sử dụng tài khoản thanh toán tại Ngân hàng Nhà nước.................... '
             '---ccccesice« kèm theo tài khoản thanh toán SỐ: .......................-.Ă cà nhere '
             'CỦA........... is \n'
             'Thời điểm bất đầu có hiệu lực:..................... 22-ccctnhtrhnnhnHmeerree \n'
             'TRƯỞNG PHÒNG KẾ TOÁN GIÁM ĐÓC SỞ GIAO DỊCH (Ký, ghi rõ họ tên) _ NGÂN HÀNG NHÀ NƯỚC/ '
             'GIÁM ĐÓC NGÂN HÀNG NHÀ NƯỚC CHI NHÁNH..... \n'
             '(Ký, ghi rõ họ tên, đóng dấu) \n'
             ' Mẫu số 02 \n'
             'TÊN TỎ CHỨC ĐÓNG TÀI KHOẢN CỘNG HOẢ XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh '
             'phúc \n'
             'ĐƠN ĐÈ NGHỊ ĐÓNG TÀI KHOẢN THANH TOÁN TẠI NGÂN HÀNG NHÀ NƯỚC \n'
             'Kính gửi:....................--. \n'
             'Chúng tôi cam kết: 1 - Chấp hành đúng, đầy đủ các quy định của pháp luật hiện hành '
             'về đóng tài __ khoản thanh toán tại Ngân hàng Nhà nước; - Thực hiện đầy đủ các nghĩa '
             'vụ liên quan đến tài khoản thanh toán với Ngân hàng Nhà nước. \n'
             'NGƯỜI ĐẠI DIỆN HỢP PHÁP (Ký, ghi rõ họ tên, đóng dấu) \n'
             ' PHÀN DÀNH CHO NGÂN HÀNG NHÀ NƯỚC \n'
             'Sau khi kiểm tra, xử lý Đơn đề nghị đóng tài khoản thanh toán của............ ¬— và '
             'đảm bảo.. đã hoàn thành đầy đủ các nghĩa vụ liên quan đến tài khoản thanh toán. tại '
             'Ngân hàng Nhà nước, Ngân hăng À Nhà nước ..................................e. đồng ý '
             'đóng tài khoản thanh toán sô hiệu.. ¬—........s........... TỪ TƯẦY........ QC HH TH '
             'KH C01 TT 01116040115 C01 01k kh \n'
             'TRƯỞNG PHÒNG KÉ TOÁN GIÁM ĐỐC SỞ GIAO DỊCH (Ký, ghỉ rõ họ tên) .__ NGÂN HÀNG NHÀ '
             'NƯỚC/ GIÁM ĐÓC NGÂN HÀNG NHÀ NƯỚC \n'
             '(Kỹ, ghi rõ họ tên, đóng dấu) \n'
             ' Mẫu số 03 \n'
             'DOANH NGHIỆP CUNGỨNG CỘNG HOÀ XÃ HỘI CHỦ NGHĨA VIỆT NAM DỊCH VỤ BƯU CHÍNH CÔNG ÍCH '
             'Độc lập - Tự do - Hạnh phúc \n'
             '.„ Hgày....tháng..... năm..... \n'
             'ĐƠN ĐẺ NGHỊ CUNG ỨNG DỊCH VỤ THANH TOÁN KHÔNG QUA. TÀI KHOẢN THANH TOÁN CỦA KHÁCH '
             'HÀNG \n'
             'Kính gửi: Thống đốc Ngân hàng Nhà nước Việt Nam. \n'
             'Căn cứ Nghị định số .../.../NĐ- CP ngày.. . thẳng .. . của Chính phủ quy định về. '
             'thanh toán không dùng tiền mặt và các văn bảng sửa ứa đổi bổ sung (nếu có); \n'
             'Căn cứ Nghị quyết cuộc họp Hội đông thành viên/Hội đông quản trị/Đại hội đẳng cổ '
             'đông của Doanh nghiệp cung ứng dịch vụ bưu chính công ích ngày... tháng... năm.. . '
             'thống nhất về việc đề nghị được cung ứng dịch vụ thanh toán không qua tài khoản '
             'thanh toán của khách hàng; \n'
             '(Doanh nghiệp cung ứng dịch vụ bưu chính công ích) đè nghị Ngân hàng Nhà nước xem '
             'xét chấp thuận về việc cung ứng dịch vụ thanh toán không qua tài khoản thanh toán '
             'của khách hàng với nội dụng cụ thể như sau:'},
 {'id': 72,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 38',
               'article_title': 'Trách nhiệm thi hành',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 38. Trách nhiệm thi hành',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 36,
               'chunk_part': '7/7'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 38. Trách nhiệm thi hành (tiếp theo)]\n'
             '1. Tên Doanh nghiệp cung ứng dịch vụ bưu chính công ích: - Tên đầy đủ bằng tiếng '
             'Việt: \n'
             '- Tên viết tắt bằng tiếng Việt (nếu có): \n'
             '- Tên đầy đủ bằng tiếng Anh (nếu có): \n'
             '- Tên viết tắt bằng tiếng Anh (nếu có): \n'
             '- Tên dùng để giao dịch (nếu có): \n'
             ': 2. Giấy phép thành lập/Giấy chứng nhận đẳng ký doanh nghiệp số...... được cập '
             'bởi... ngày... tháng... năm...\n'
             '\n'
             '3. Mã số doanh nghiệp/Mã số thuế:\n'
             '4. Địa điểm đặt trụ sở chính, số điện thoại, số Fax, Email.\n'
             '\n'
             '5. Tên, nội dung (các) dịch vụ thanh toán không qua tài khoản thanh toán của khách '
             'hàng đề nghị cung ứng: \n'
             ' Chúng tôi cam. kết đáp ứng và chấp hành nghiêm chỉnh các điều kiện quy định tại '
             'Nghị định số .../ ./NÐ-CP ngày... tháng ... năm ... của Chính phủ quy định về thanh '
             'toán không dùng tiền mặt và các văn bản có liên quan. Nếu vi phạm chúng tôi xin hoàn '
             'toàn chịu trách nhiệm trước pháp luật. \n'
             '` NGƯỜI ĐẠI DIỆN HỢP PHÁP Hồ sơ đính kèm: (Ký, ghi rõ họ tên, đông dấu)\n'
             '1.\n'
             '2. \n'
             ' Mẫu số 04 \n'
             'NGÂN HÀNG NHÀ NƯỚC VIỆTNAM CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM —— Độc lập - Tự do - '
             'Hạnh phúc \n'
             'Số: .../QĐ-NHNN Hà Nội, ngày... tháng... nắm \n'
             'QUYÉT ĐỊNH Về việc chấp thuận hoạt động cung ứng dịch vụ thanh toán không qua tài '
             'khoản thanh toán của khách hàng \n'
             'THÓNG ĐÓC NGÂN HÀNG NHÀ NƯỚC VIỆT NAM Căn cứ Luật Ngân hàng Nhà nước Việt Nam ngày '
             '16 thẳng 6 năm 2010; \n'
             'Căn cứ Nghị định số .../ .NÐ-CP ngày... tháng... năm.. . của Chính phủ quy định về '
             'thanh toán không dùng tiền mặt và các văn bản sửa đổi, bỗ sung (nếu có); \n'
             'Xét Đơn đề nghị chấp thuận hoạt động cung ứng dịch vụ thanh toán không qua tài khoản '
             'thanh toán của khách hàng và Hồ sơ đính kèm của Doanh nghiệp cung ứng dịch vụ bưu '
             'chính công ích; \n'
             'Theo đề nghị của Vụ trưởng Vụ Thanh toán. QUYẾT ĐỊNH:'},
 {'id': 73,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 1',
               'article_title': 'Chấp thuận cho phép hoạt động cung ứng dịch vụ thanh toán không '
                                'qua tài khoản thanh toán của khách hàng:',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 1. Chấp thuận cho phép hoạt động '
                                  'cung ứng dịch vụ thanh toán không qua tài khoản thanh toán của '
                                  'khách hàng:',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 36,
               'chunk_part': '1/1'},
  'content': 'Điều 1. Chấp thuận cho phép hoạt động cung ứng dịch vụ thanh toán không qua tài '
             'khoản thanh toán của khách hàng:\n'
             '\n'
             '1. Tên tổ chức: \n'
             '- Tên đầy đủ bằng tiếng ViệUtiếng Anh (nếu có): \n'
             '- Tên viết tắt bằng tiếng Việt và tiếng Anh, tên giao dịch (nếu có):\n'
             '\n'
             '2. Mã số doanh nghiệp:\n'
             '\n'
             '3. Địa điểm đặt trụ sở chính:'},
 {'id': 74,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 2',
               'article_title': '(Các) dịch vụ thanh toán không qua tải khoản thanh toán của khách '
                                'hàng được chấp thuận:',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 2. (Các) dịch vụ thanh toán '
                                  'không qua tải khoản thanh toán của khách hàng được chấp thuận:',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 36,
               'chunk_part': '1/1'},
  'content': 'Điều 2. (Các) dịch vụ thanh toán không qua tải khoản thanh toán của khách hàng được '
             'chấp thuận:'},
 {'id': 75,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 3',
               'article_title': '(Các) điều khoản khác:',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 3. (Các) điều khoản khác:',
               'topics': 'Quy định chung',
               'is_core_fintech': False,
               'page': 36,
               'chunk_part': '1/1'},
  'content': 'Điều 3. (Các) điều khoản khác:'},
 {'id': 76,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 4',
               'article_title': 'Trong quá trình cung ứng các dịch vụ thanh toán không qua tài '
                                'khoản thanh toán của khách hàng, Doanh nghiệp cung ứng dịch vụ 2 '
                                'hưu chính công ích',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 4. Trong quá trình cung ứng các '
                                  'dịch vụ thanh toán không qua tài khoản thanh toán của khách '
                                  'hàng, Doanh nghiệp cung ứng dịch vụ 2 hưu chính công ích',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 48,
               'chunk_part': '1/1'},
  'content': 'Điều 4. Trong quá trình cung ứng các dịch vụ thanh toán không qua tài khoản thanh '
             'toán của khách hàng, Doanh nghiệp cung ứng dịch vụ 2 hưu chính công ích \n'
             'phải tuân thủ quy định tại Nghị định sô.. . ngày... tháng... , của Chính phủ quy '
             'định về thanh toán không dùng tiền mặt và các văn bản: sửa sửa đấi, bổ sung (nếu '
             'có), Thông tư số... ngày... tháng... . của Ngân hàng Nhà nước Việt Nam \n'
             'quy định về dịch vụ thanh toán không - đồng tiền mặt và các văn bản sửa đổi, bỗ sung '
             '(nếu có); các quy định của pháp luật có liên quan.'},
 {'id': 77,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 5',
               'article_title': 'Hiệu lực thi hành ˆ Quyết định này có thời hạn hoạt động 10 năm '
                                'kế từ ngày ký.',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 5. Hiệu lực thi hành ˆ Quyết '
                                  'định này có thời hạn hoạt động 10 năm kế từ ngày ký.',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 49,
               'chunk_part': '1/2'},
  'content': 'Điều 5. Hiệu lực thi hành ˆ Quyết định này có thời hạn hoạt động 10 năm kế từ ngày '
             'ký. \n'
             'Nơi nhận: THÓNG ĐÓC - Như Điều 1; \n'
             '~ Ban Lãnh đạo NHNN; \n'
             '- Cục CNTT, Vụ PC \n'
             '- CQTTGSNH (để phối hợp); \n'
             '- NHẬNN chỉ nhánh ... \n'
             ' Mẫu số 05 DOANH NGHIỆP CUNG ỨNG CỘNG HOÀ XÃ HỘI CHỦ NGHĨA VIỆT NAM DỊCH VỤ BƯU '
             'CHÍNH CÔNG ÍCH Độc lập - Tự do - Hạnh phúc \n'
             'ĐƠN ĐÈ NGHỊ THU HÒI VĂN BẢN CHẤP THUẬN . HOẠT ĐỘNG CUNG ỨNG DỊCH VỤ THANH TOÁN KHÔNG '
             'QUA TÀI KHOẢN THANH TOÁN CỦA KHÁCH HÀNG \n'
             'Kính gửi: Thống đốc Ngân hàng Nhà nước Việt Nam. . Căn cứ Nghị định số -‹../NĐ-CP '
             'ngày... tháng... năm.. . của Chính phủ quy \n'
             'định về thanh toán không dùng tiên mặt và các văn bản sửa đổi, bỗ sung (nếu có); \n'
             'Căn cứ Nghị quyết cuộc họp Hội đông thành viên/Hội đông quản trị/Đại hội đồng cô '
             'đông của Doanh nghiệp cung ứng dịch vụ bưu chính công ích ngày... tháng... năm... '
             'thống nhất về việc đề nghị thu hồi văn bản chấp thuận hoạt động Cung ứng dịch vụ '
             'thanh toán không qua tài khoản thanh toán của khách hàng tại Quyết định số... '
             'ngày... tháng... năm... do Ngân hàng Nhà nước cấp; \n'
             '(Doanh nghiệp cung ứng dịch vụ bưu chính công ích) đề nghị thu hồi văn bản chấp '
             'thuận hoạt động cung ứng dịch vụ thanh toán không qua tài khoản thanh toán của khách '
             'hàng kế từ nẦY... tháng... năm.. „ do đó chúng tôi đề nghị Ngân hàng Nhà nước xem '
             'xét thu hồi Quyết định số... ngày... tháng... năm... về việc chấp thuận hoạt động '
             'cung ứng dịch vụ thanh toán không qua tài khoản thanh toán của khách hàng với nội '
             'dung cụ thể như sau:\n'
             '\n'
             '1. Tên Doanh nghiệp cung ứng địch vụ bưu chính công ích: - Tên đầy đủ bằng tiếng '
             'Việt: \n'
             '- Tên viết tắt bằng tiếng Việt (nếu có): \n'
             '- Tên đầy đủ bằng tiếng Anh (nếu có): \n'
             '- Tên viết tắt bằng tiếng Anh (nếu có): \n'
             '~ Tên dùng để giao dịch (nếu có): \n'
             '. 2. Giấy phép thành lập Giây chứng nhận đăng ký doanh nghiệp số... được cấp bởi... '
             'ngày... tháng... năm..\n'
             '\n'
             '3. Mã số doanh nghiệp/Mã số thuế:\n'
             '\n'
             '4. Địa điểm đặt trụ sở chính, số điện thoại, số Fax, Email: \n'
             '\n'
             '5. Văn bản chấp thuận hoạt động cung ứng . địch vụ thanh toán không qua tài khoản '
             'thanh toán của khách nàng Quyết định số... do Ngân hàng Nhà nước Việt Nam cấp '
             'ngày... tháng... năm..\n'
             '\n'
             '6. Lý do đề nghị thu hồi: (chủ cụ hệ neo đuy định tại khoản 4 Điều 20 Nghị : định số '
             '.../...NĐ-CP ngày... tháng ... ... của Chính phủ quy định về thanh toắn không dùng '
             'tiên mặt).\n'
             '\n'
             '7. Thời gian chấm dứt cung ứng dịch vụ thanh toán không qua tài khoản thanh toán của '
             'khách hàng: \n'
             'Chúng tôi cam kết đã hoàn thành việc xử lý nghĩa vụ thuế, các khoản nỢ, tài ¿ sản, '
             'quyên lợi khách hàng và chịu trách nhiệm hoàn toàn trước pháp luật về tính hợp pháp, '
             'chính xác, trung thực của nội dung Đơn này. Nếu vi phạm, chúng tôi xin chịu hoàn '
             'toàn trách nhiệm trước pháp luật. \n'
             ': NGƯỜI ĐẠI DIỆN THEO PHÁP LUẬT | Hồ sơ đính kèm: (Ký, ghi rõ họ tên, đóng dâu)'},
 {'id': 78,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 5',
               'article_title': 'Hiệu lực thi hành ˆ Quyết định này có thời hạn hoạt động 10 năm '
                                'kế từ ngày ký.',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 5. Hiệu lực thi hành ˆ Quyết '
                                  'định này có thời hạn hoạt động 10 năm kế từ ngày ký.',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 49,
               'chunk_part': '2/2'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 5. Hiệu lực thi hành ˆ Quyết định này có thời hạn '
             'hoạt động 10 năm kế từ ngày ký. (tiếp theo)]\n'
             '1.\n'
             '\n'
             '2. | \n'
             '\n'
             ' l6 Mẫu số 06 \n'
             'NGÂN HÀNG NHÀ NƯỚC VIỆT NAM CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM —— Độc lập - Tự do - '
             'Hạnh phúc \n'
             'Số: ....QĐ-NHNN ... ngày... tháng... năm \n'
             'QUYÉT ĐỊNH Về việc thu hồi Văn bản chấp thuận hoạt động cung ứng dịch vụ thanh toán '
             'không qua tài khoản thanh toán của khách hàng \n'
             'THÓNG ĐÓC NGÂN HÀNG NHÀ NƯỚC VIỆT NAM \n'
             'Căn cứ Luật Ngân hàng Nhà nước Việt Nam ngày l6 tháng 6 năm 2010; \n'
             'Căn cứ Nghị định số «ò4NĐ-CP ngày... tháng... năm... của Chính phủ quy định về thanh '
             'toán Không dùng tiền mặt và các văn bản sửa đổi, bổ sung (nếu có); \n'
             'Căn cứ Thông tư số .../...TT-NHNN ngày... tháng... năm... của Thống đốc Ngân hàng '
             'Nhà nước Việt Nam quy, định về cung ứng địch vụ thanh toán không dùng tiền mặt và '
             'các văn bản sửa đổi, bỗ sung (nếu có); \n'
             'e8. 7 PP. . . ẻ Ề ..... ˆ. ẽe nắn \n'
             'Theo đề nghị của Vụ trưởng Vụ Thanh toán. \n'
             'QUYÉT ĐỊNH:'},
 {'id': 79,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 1',
               'article_title': 'Thu hồi Văn bản chấp thuận hoạt động cung ứng dịch vụ thanh toán '
                                'không qua tài khoản thanh toán của khách hàng số ...QÐ-NHNN '
                                'ngày... tháng... năm... của Ngân hàng Nhà nước Việt Nam cập cho '
                                'Doanh nghiệp cung ứng dịch vụ bưu chính công ích.',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 1. Thu hồi Văn bản chấp thuận '
                                  'hoạt động cung ứng dịch vụ thanh toán không qua tài khoản thanh '
                                  'toán của khách hàng số ...QÐ-NHNN ngày... tháng... năm... của '
                                  'Ngân hàng Nhà nước Việt Nam cập cho Doanh nghiệp cung ứng dịch '
                                  'vụ bưu chính công ích.',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 49,
               'chunk_part': '1/1'},
  'content': 'Điều 1. Thu hồi Văn bản chấp thuận hoạt động cung ứng dịch vụ thanh toán không qua '
             'tài khoản thanh toán của khách hàng số ...QÐ-NHNN ngày... tháng... năm... của Ngân '
             'hàng Nhà nước Việt Nam cập cho Doanh nghiệp cung ứng dịch vụ bưu chính công ích.'},
 {'id': 80,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 2',
               'article_title': '(Các) điều khoản khác',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 2. (Các) điều khoản khác',
               'topics': 'Quy định chung',
               'is_core_fintech': False,
               'page': 49,
               'chunk_part': '1/1'},
  'content': 'Điều 2. (Các) điều khoản khác'},
 {'id': 81,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 3',
               'article_title': 'Quyết định này có hiệu lực Kể từ ngày ký.',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 3. Quyết định này có hiệu lực Kể '
                                  'từ ngày ký.',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản, '
                         'Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 52,
               'chunk_part': '1/10'},
  'content': 'Điều 3. Quyết định này có hiệu lực Kể từ ngày ký. \n'
             'Nơi nhận: THÓNG ĐÓC - Như Điều 1; \n'
             '- Ban Lãnh đạo NHNN; \n'
             '- Cục CNTT, Vụ PC \n'
             '- CQTTGSNH (để phối hợp), \n'
             '- NHNN chỉ nhánh.. \n'
             '- Lưu:. \n'
             ' Mẫu số 07 TÊN TỎCHỨC CỘNG HOÀ XÃ HỘI CHỦ NGHĨA VIỆT NAM ĐỀ NGHỊ CÁP GIÁY PHÉP Độc '
             'lập - Tự do - Hạnh phúc \n'
             '„ Ngày..... tháng..... năm..... \n'
             'ĐƠN ĐÈ NGHỊ CÁP GIÁY PHÉP HOẠT ĐỘNG CƯNG ỨNG DỊCH VỤ TRUNG GIAN THANH TOÁN \n'
             'Kính gửi: Thống đốc Ngân hàng Nhà nước Việt Nam. \n'
             'Căn cứ Nghị định số .../ .../NĐ-CP ngày... tháng... năm.. . của Chính phủ quy định '
             'về thanh toán không dùng. tiền mặt và các văn bản sửa đổi, bổ sung (nếu có); \n'
             'Căn cứ Thông tư số .../ .T1-NHNN ngày... tháng... năm... của Ngân hàng Nhà nước Việt '
             'Nam 4⁄7 định về hoạt động cung ứng dịch vụ trung gian thanh toán và các văn bản sửa '
             'đổi, bỗ sung (nếu có); \n'
             'Căn cứ Nghị quyết cuộc họp Hội đồng thành viên/Đại hội đông cổ ) đông/Hội đồng quản '
             'tr/Wăn bản của người đại điện có thẩm quản của chủ sở hữu của...... '
             '(])......SỐ....... ngày... tháng... năm.. . thống nhất về việc đề nghị cắp Giấy pháp '
             'hoạt động cung ứng dịch vụ trung gian thanh toán, \n'
             '...(1)... đề nghị Ngân hàng Nhà nước Việt Nam xem xét cấp Giấy phép hoạt động cung '
             'ứng dịch vụ trung gian thanh toán với nội dung cụ thể như sau:\n'
             '\n'
             '1. Tên Tổ chức để nghị cấp Giấy phép hoạt động cung ứng dịch vụ trung gian thanh '
             'toán: \n'
             '- Tên đầy đủ bằng tiếng Việt: \n'
             '- Tên viết tắt bằng tiếng Việt (nếu có): - Tên đầy đủ bằng tiếng Anh (nếu có): - Tên '
             'viết tắt bằng tiếng Anh (nếu có): - Tên dùng để giao dịch (nếu có):\n'
             '\n'
             '2. Giấy phép thành lập/Giấy chứng nhận đăng ký doanh nghiệp Giây chứng nhận đăng ký '
             'kinh doanh sô..... được cấp bởi...... ngày... tháng... năm..\n'
             '\n'
             '3. Mã số doanh nghiệp/Mã số thuế...................\n'
             '4. Địa điểm đặt trụ sở chính, số điện thoại, số Fax, Email.\n'
             '5. Tên (các) dịch vụ trung gian thanh toán đề nghị cấp Giấy phép: \n'
             ' , \n'
             'Chúng tôi cam kết đáp ứng và chấp hành nghiêm chỉnh các quy định của pháp luật, của '
             'Ngân hàng Nhà nước; chịu hoàn toàn trách nhiệm về tính chính xác, trung thực của nội '
             'dung Hồ sơ đề nghị cấp Giấy phép; không sao chép, giả mạo các giấy tờ chứng minh đủ '
             'điều kiện để được câp Giấy phép hoạt động cung, ứng dịch vụ trung gian thanh toán '
             'trong Hồ sơ đề nghị cấp Giấy phép. Nếu vi phạm, chúng tôi xin chịu hoàn toàn trách '
             'nhiệm trước pháp luật. \n'
             "NGƯỜI ĐẠI DIỆN THEO PHÁP LUẬT ' \n"
             'Hà sơ đính kèm: (Ký, ghi rõ họ tên, đóng dấu)'},
 {'id': 82,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 3',
               'article_title': 'Quyết định này có hiệu lực Kể từ ngày ký.',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 3. Quyết định này có hiệu lực Kể '
                                  'từ ngày ký.',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản, '
                         'Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 52,
               'chunk_part': '2/10'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 3. Quyết định này có hiệu lực Kể từ ngày ký. (tiếp '
             'theo)]\n'
             '1.'},
 {'id': 83,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 3',
               'article_title': 'Quyết định này có hiệu lực Kể từ ngày ký.',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 3. Quyết định này có hiệu lực Kể '
                                  'từ ngày ký.',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản, '
                         'Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 52,
               'chunk_part': '3/10'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 3. Quyết định này có hiệu lực Kể từ ngày ký. (tiếp '
             'theo)]\n'
             '2. \n'
             'Ghi chú: ...(1)... Tên tổ chức đề nghị cấp Giấy phép hoạt động cung ứng dịch vụ '
             'trung gian thanh toán. . \n'
             ' Mẫu số 08 \n'
             'NỘI DUNG ĐÈ ÁN CUNG ỨNG DỊCH VỤ TRUNG GIAN THANH TOÁN \n'
             'Trong Hồ sơ đề nghị cấp Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán, '
             'Đề á án cung ứng dịch vụ trung gian \n'
             'thanh toán tối thiểu phải có các nội dung sau: \n'
             'Nội dung \n'
             'Tiêu chí cụ thể \n'
             'Giới thiệu chung về tổ chức đề nghị câp Giây phép hoạt động cung ứng địch vụ trung '
             'gian thanh toán. . \n'
             'Quy trình nghiệp vụ kỹ thuật của dịch vụ đê nghị câp phép \n'
             'các bên liên quan đối với dịch vụ ví điện tử và dịch vụ hỗ trợ thu hộ, chi hộ. \n'
             'Cơ chế mở, duy trì số dư tài khoản đảm bảo thanh toán, mục đích sử dụng tài khoản '
             'đảm bảo thanh đối với dịch vụ ví điện tử \n'
             '- Tên tô chức. - Quá trình hình thành và phát triển. - Các ngành, nghề kinh doanh. - '
             'Định hướng cung ứng và triển khai cung ứng địch vụ trung gian thanh toán sau khi '
             'được cấp Giấy phép. - Các nội dung liên quan khác (nếu có). - Tên dịch vụ. - Phạm vi '
             'cung ứng. - Đối tượng khách hàng. - Điều kiện sử dụng. - Sơ đỗ và diễn giải các bước '
             'thực hiện nghiệp vụ. - Quy trình dòng tiền từ lúc khởi tạo giao dịch đến lúc quyết '
             'toán nghĩa vụ giữa \n'
             '- Tổ chức cung ứng dịch vụ ví điện tử phải mở tài khoản đảm bảo thanh toán cho việc '
             'cung ứng dịch vụ ví điện tử. - Việc duy trì tổng số dư trên tất cả các tài khoản đảm '
             'bảo thanh toán cho dịch vụ \n'
             'toán | ví điện tử. - Mục đích sử dụng tài khoản đảm bảo thanh toán cho dịch vụ ví '
             'điện tử theo quy \n'
             'định của pháp luật. \n'
             '\n'
             ' Cơ chế đảm bảo khả năng thanh toán \n'
             'đối với dịch vụ hỗ trợ thu hộ, chỉ hộ - Xây dựng cơ.chế đâm bảo khả năng thanh toán '
             'đối với dịch vụ hỗ trợ thu hộ, chỉ | _ hộ tuân thủ quy định pháp luật. \n'
             'Quy trình kiểm tra, kiểm soát nội bộ \n'
             '- Xây dựng quy định nội bộ về kiểm tra, kiểm soát nội bộ, trong đó có các nội dung '
             'bao gồm: Mục đích, yêu câu; trình tự thực hiện; trách nhiệm của bộ phận \n'
             'liên quan. — \n'
             'Quy trình quản lý rủi ro, đảm bảo an toàn, bảo mật \n'
             '- Quy định quản lý rủi ro, đảm bảo an toàn, bảo mật đảm bảo tuân thủ các quy định '
             'của NHNN trong hoạt động ngân hàng điện tử. \n'
             '- Quy định về việc lập, sử dụng, bảo quản, lưu trữ chứng từ điện tử theo quy định '
             'của pháp luật về giao dịch điện tử trong hoạt động ngân hàng. \n'
             '- Đánh giá các rủi ro có thể xảy ra và các biện pháp quản lý rủi ro tương ứng trong '
             'quá trình cung ứng dịch vụ trung gian thanh toán. \n'
             '- Xây dựng quy trình nộ! bộ về chính sách, biện pháp xử lý rủi ro cụ thê. \n'
             'Các nguyên tắc chung và quy định nội bộ vê phòng, chống rửa tiền, tài trợ khủng bố '
             'và tài trợ phổ biến vũ khí hủy diệt hàng loạt \n'
             '- Các nguyên tắc chung về phòng, chống rửa tiền, tài trợ khủng bố, tài trợ phổ biến '
             'vũ khí hủy diệt hàng loạt.'},
 {'id': 84,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 3',
               'article_title': 'Quyết định này có hiệu lực Kể từ ngày ký.',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 3. Quyết định này có hiệu lực Kể '
                                  'từ ngày ký.',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản, '
                         'Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 52,
               'chunk_part': '4/10'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 3. Quyết định này có hiệu lực Kể từ ngày ký. (tiếp '
             'theo)]\n'
             '- Xây dựng quy trình nộ! bộ về chính sách, biện pháp xử lý rủi ro cụ thê. \n'
             'Các nguyên tắc chung và quy định nội bộ vê phòng, chống rửa tiền, tài trợ khủng bố '
             'và tài trợ phổ biến vũ khí hủy diệt hàng loạt \n'
             '- Các nguyên tắc chung về phòng, chống rửa tiền, tài trợ khủng bố, tài trợ phổ biến '
             'vũ khí hủy diệt hàng loạt. \n'
             '- Quy định nội bộ về phòng, chống rửa tiền, tài trợ khủng bố, tài trợ phô biến vũ '
             'khí hủy diệt hàng loạt phù hợp với quy định của pháp luật về phòng, chống rửa tiền, '
             'tài trợ khủng bố, tài trợ phổ biến vũ khí hủy điệt hàng loạt. \n'
             'Quy trình và thủ tục giải quyết yêu cầu tra soát, khiếu nại, tranh chấp \n'
             '- Quy định về việc giải quyết hoặc trả lời các khiếu nại, tra soát của khách hàng, '
             'trong đó thể hiện rõ: Các kênh tiếp nhận, bộ phận tiếp nhận, các bước hướng dẫn '
             'khách hàng khiếu nại, khâu xử lý khiếu nại trong nội bộ tổ chức trung gian thanh '
             'toán và khi phải phối hợp với các đơn vị liên quan. \n'
             '- Quy trình, thủ tục, trách nhiệm cụ thể của từng bên liên quan, thời gian thực hiện '
             'từng bước và kết quả trả lại cho khách hàng khiếu nại. \n'
             '\n'
             ' Phương án được một tổ chức thực hiện quyết toán kết quả bù trừ giữa các bên liên '
             'quan đối với dịch vụ chuyên mạch tài chính, bù trừ điện tử, chuyển mạch tài chính '
             'quốc tế \n'
             '- Mối liên hệ với các hệ thống thanh toán khác (nếu có). - Tổ chức thực hiện quyết '
             'toán. - Phạm vi quyết toán. - Điều kiện thực hiện quyết toán. - Đồng tiền quyết '
             'toán; tỷ giá áp dụng (nếu có). - Tài khoản quyết toán. \n'
             '- Quy trình quyết toán: Thời gian quyết toán, sơ đồ và diễn giải các bước thực hiện '
             'quyết toán, giải quyết tranh chấp, phí, các biện pháp nhận diện, phòng ngừa, xử lý \n'
             'rủi ro trong quá trình quyết toán (bao gồm rủi ro quyết toán các nghĩa vụ tài chính '
             'khi các bên tham gia mất khả năng thanh toán). \n'
             '- Quyên và trách nhiệm các thành viên và các bên liên quan. \n'
             '\n'
             ' Mẫu số 09 \n'
             'CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM l Độc lập - Tự do - Hạnh phúc \n'
             'SƠ YẾU LÝ LỊCH ®'},
 {'id': 85,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 3',
               'article_title': 'Quyết định này có hiệu lực Kể từ ngày ký.',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 3. Quyết định này có hiệu lực Kể '
                                  'từ ngày ký.',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản, '
                         'Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 52,
               'chunk_part': '5/10'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 3. Quyết định này có hiệu lực Kể từ ngày ký. (tiếp '
             'theo)]\n'
             '1. Về bản thân - Họ và tên đầy đủ: Ảnh hộ chiếu (4x 6cm) \n'
             '- Ngày, tháng, năm sinh: - Quốc tịch/các quốc tịch (nếu có) `: - Nơi thường trú”: \n'
             '- Nơi ở hiện tại (trường hợp khác với nơi đăng ký thường trú): \n'
             '- Số định danh cá nhân (đối với thẻ căn cước, thẻ căn cước công dân) hoặc số chứng '
             'mỉnh nhân dân hoặc số hộ chiều hoặc sô giây tờ chứng minh cá nhân hợp pháp '
             'khác:...... "—— ; Ngày câấp:................ ; Có giá trị đên:...., Nơi\n'
             '\n'
             '2. Trình độ học vẫn \n'
             '- Giáo dục phổ thông: \n'
             '= Học hàm, học vị (nêu rõ tên, địa chỉ trường; chuyên ngành học; thời gian học; băng '
             'cập (liệt kê đây đủ các băng cấp):\n'
             '\n'
             '3. Quá trình công tác: \n'
             '- Nghề nghiệp, đơn vị, chức vụ công tác từ năm 18 tuổi đến nay 3; \n'
             'Thời gian . Lĩnh vực hoạt động (từ tháng/năm ˆ của doanh nghiệp và \n'
             'đến tháng/năm) ® nhiệm vụ được giao \n'
             '- Khen thưởng, kỷ luật:\n'
             '4. Năng lực hành vi dân sự 6) \n'
             '5, Thông tin về cấm đảm nhiệm chức vụ, thành lập, quản lý doanh nghiệp, hợp tác xã '
             '(đôi với trường hợp phiêu lý lịch tư pháp hoặc văn bản có giá trị tương đương do cơ '
             'quan có thâm quyên nước ngoài cấp không có thông tin này)\n'
             '\n'
             '6. Quan hệ nhân thân (cha, mẹ, vợ, chông, con và anh, chị, em ruột) \n'
             '\n'
             '7. Tôi (họ và tên của người kê khai sơ yếu lý lịch) xin cam kết: \n'
             '- Đáp ứng điều kiện để giữ chức danh... tại... (tên Tổ chức đề nghị cấp Giấy phép '
             'hoạt động cung ứng dịch vụ trung gian thanh toán). \n'
             '- Không vi phạm các quy định của pháp luật trong lĩnh vực tiền tệ và ngân hàng. \n'
             '- Các thông tin cá nhân Tôi cung cấp trên đây là đúng sự thật. Tôi xin chịu hoàn '
             'toàn trách nhiệm đối với bất kỳ thông tin nào không đúng với sự thật tại bản khai '
             'này. \n'
             '.„ ngày... tháng... năm.... \n'
             'XÁC NHẬN CỦA CÁP CÓ THẢM QUYỀN NGƯỜI KHAI CỦA ĐƠN VỊ CÔNG TÁC (Ký và ghi rõ họ và '
             'tên) Ghỉ chú:'},
 {'id': 86,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 3',
               'article_title': 'Quyết định này có hiệu lực Kể từ ngày ký.',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 3. Quyết định này có hiệu lực Kể '
                                  'từ ngày ký.',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản, '
                         'Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 52,
               'chunk_part': '6/10'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 3. Quyết định này có hiệu lực Kể từ ngày ký. (tiếp '
             'theo)]\n'
             '1. Người khai phải kê khai đầy đủ thông tin theo yêu câu và chịu trách nhiệm trước '
             'pháp luật-về tính đầy đủ, chính xác, trung thực của hề sơ, trường hợp không phát '
             'sinh thì ghi rõ không có.\n'
             '\n'
             '2. Người khai phải kê khai đầy đủ công việc, đơn vị công tác, các chức vụ đã và đang '
             'nắm giữ.\n'
             '\n'
             '3. Phải đảm bảo tính liên tục về mặt thời gian.\n'
             '\n'
             '4. Nêu TỐ nếu nhân sự là người quản lý, người điều hành theo quy định của pháp luật '
             'và\n'
             'Điều lệ của tổ chức đề nghị cấp Giấy phép hoạt động cung ứng địch vụ trung gian '
             'thanh toán.\n'
             '\n'
             '5. Ghi cụ thể: đầy đủ/hạn chế/mất năng lực hành vi dân sự. \n'
             'Trường hợp NHNN có thể truy cập, khai thác thông tin trên Cơ sở dữ liệu quốc gia về '
             'dân cư, người khai là công đân Việt Nam thì không bắt buộc phải kê khai các thông '
             'tin đánh dấu (*). \n'
             'R (Ngoài những nội dung cơ bản trên, người khai có thể bể sung các nội dung khác nếu '
             'thấy cần thiếp \n'
             ' Mẫu số 10 BIÊN BẢN NGHIỆM THU KỸ THUẬT \n'
             '(Biên bản nghiệm thu kỹ thuật các dịch vụ trung gian thanh toán đã được cắp Giấy '
             'phép với một ngân hàng hợp tác đối với dịch vụ ví điện tử, dịch vụ hỗ trợ thu hộ, '
             'chỉ hộ và dịch vụ công thanh toán điện tử tối thiểu phải có các nội dưng sau:)'},
 {'id': 87,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 3',
               'article_title': 'Quyết định này có hiệu lực Kể từ ngày ký.',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 3. Quyết định này có hiệu lực Kể '
                                  'từ ngày ký.',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản, '
                         'Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 52,
               'chunk_part': '7/10'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 3. Quyết định này có hiệu lực Kể từ ngày ký. (tiếp '
             'theo)]\n'
             '1. Thông tin Tổ chức cung ứng dịch vụ trung gian thanh toán: - Tên tổ chức. \n'
             '- Người đại diện hợp pháp của tổ chức; chức vụ. \n'
             '- Địa chỉ trụ sở chính. \n'
             '- Điện thoại liên lạc. \n'
             '~ Các thông tin liên quan khác (nếu có).\n'
             '\n'
             '2. Thông tin ngân hàng hợp tác: \n'
             '- Tên ngân hàng. \n'
             '- Người đại diện hợp pháp của ngân hàng; chức vụ. \n'
             '- Địa chỉ trụ sở chính. \n'
             '- Điện thoại liên lạc. \n'
             '- Các thông tin liên quan khác (nếu có).\n'
             '\n'
             '3. Các nội dung chính của Biên bản nghiệm thu kỹ thuật \n'
             'Căn cứ kết :. quả triển khai nghiệm thu kỹ thuật dịch vụ trung gian thanh toán \n'
             'đã được cấp Giấy phép, hai bên cùng ký kết Biên bản nghiệm thu kỹ thuật (các) dịch '
             'vụ trung gian thanh toán với các nội dung sau: \n'
             '- Tên (các) dịch vụ trung gian thanh toán thực biện nghiệm thu kỹ thuật. \n'
             '- Thời gian thực hiện (sau thời điểm cấp.Giấy phép cung ứng (các) dịch vụ trung gian '
             'thanh toán). \n'
             '- Môi trường thực hiện: \n'
             '- Các nghiệp vụ/chức năng thực hiện nghiệm thu kỹ thuật (nếu có). \n'
             '- Nội dung thử nghiệm \n'
             '- Kết quả nghiệm thu kỹ thuật (đối với từng dịch vụ trung gian thanh toán). \n'
             '- Đánh giá kết quả nghiệm thu kỹ thuật. \n'
             '- Kịch bản nghiệm thu kỹ thuật (nếu có). \n'
             '- Các nội dung liên quan khác (nếu có). \n'
             'NGƯỜI ĐẠI DIỆN THEO PHÁP LUẬT NGƯỜI ĐẠI DIỆN THEO PHÁP LUẬT CỦA NGÂN HÀNG HỢP TÁC '
             'CỦA TỎ CHỨC CUNG ỨNG DỊCH VỤ (Ký, ghi rõ họ tên, đóng dấu) TRUNG GIAN THANH TOÁN '
             '(Ký, ghi rõ họ tên, đóng dấu) \n'
             ' Mẫu số 11 \n'
             'TÊN TỎ CHỨC CUNG ỨNG DỊCH VỤ CỘNG HOÀ XÃ HỘI CHỦ NGHĨA VIỆT NAM TRUNG GIAN THANH '
             'TOÁN Độc lập - Tự do - Hạnh phúc \n'
             '. Hgày.... thẳng.... nắm \n'
             'ĐƠN ĐÈ NGHỊ CÁP LẠI GIÁY PHÉP HOẠT ĐỘNG CUNG ỨNG DỊCH VỤ TRUNG GIAN THANH TOÁN \n'
             'Kính gửi: Thống đốc Ngân hàng Nhà nước Việt Nam. \n'
             'Căn cứ Nghị định số ....WĐ-CP ngày... tháng... năm.. . của Chính phủ quy định về '
             'thanh toán không dùng tiền mặt và các văn bản sửa đổi, bỗ sung (nếu có); \n'
             'Căn cứ Thông tư số .../ „7T-NHNN ngày... tháng... năm.. . của Ngân hàng Nhà nước '
             'Việt Nam q⁄y định về hoạt động cung ứng dịch vụ trưng gian thanh toán và các văn bản '
             'sửa đổi, bỗ sung (nếu có); \n'
             '...(1)... đề nghị Ngân hàng Nhà nước Việt Nam cấp lại Giấy phép hoạt động cung ứng '
             'dịch vụ trung gian thanh toán với nội dung cụ thê như sau:'},
 {'id': 88,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 3',
               'article_title': 'Quyết định này có hiệu lực Kể từ ngày ký.',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 3. Quyết định này có hiệu lực Kể '
                                  'từ ngày ký.',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản, '
                         'Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 52,
               'chunk_part': '8/10'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 3. Quyết định này có hiệu lực Kể từ ngày ký. (tiếp '
             'theo)]\n'
             '1. Tên Tổ chức đề nghị cấp lại Giấy phép hoạt động cung ứng dịch vụ trung gian thanh '
             'toán: \n'
             '- Tên đầy đủ bằng tiếng Việt: \n'
             '- Tên viết tắt bằng tiếng Việt (nếu có): - Tên đầy đủ bằng tiếng Anh (nếu có): - Tên '
             'viết tắt bằng tiếng Anh (nếu có): - Tên dùng để giao dịch (nếu có):\n'
             '\n'
             '2. Giấy phép thành lập/Giấy chứng nhận đăng ký doanh nghiệp/Giấy chứng nhận đăng ký '
             'kinh doanh số... được cấp bởi... ngày... tháng... năm..\n'
             '\n'
             '3. Mã số doanh nghiệp/Mã số thuế:\n'
             '4. Địa điểm đặt trụ sở chính, số điện thoại, số Fax, Email:\n'
             '\n'
             '5. Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán đã được cấp số... '
             'ngày... tháng... năm\n'
             '\n'
             '6. Lý do đề nghị cấp lại Giấy phép (chọn ô tương ứng): Cl Hết thời hạn Giấy phép. D '
             'Giấy phép bị mắt, bị rách, bị cháy hoặc bị hủy dưới hình thức khác. \n'
             " Trường hợp tổ chức đề nghị cắp lại Giấy pháp vì hết thời hạn Giáy phép, Tổ ' chức "
             'cung cấp tài liệu chứng mình việc duy trì các điều kiện cung ứng dịch vụ trung gian '
             'thanh toán đã được cấp Giáy phép tại thời điểm gửi ẩơn đề nghị cấp. lại Giấy phép '
             'đến Ngân hàng Nhà nước. \n'
             '(Ngoài nhiững nội dưng cơ bản trên, Tổ chức cung ứng dịch vụ trung gian thanh toán '
             'có thể bồ sung các nội dung khác nếu thấy cần thiết) \n'
             'Chúng tôi cam kết đáp ứng và chấp hành nghiêm chỉnh các quy định của pháp luật, của '
             'Ngân hàng Nhà nước Việt Nam; chịu hoàn toàn trách nhiệm về tính | chính xác, trung '
             'thực của nội dung trong Đơn và Hồ sơ đính kèm (nếu có). Nếu vi phạm, chúng tôi xin '
             'chịu hoàn toàn trách nhiệm trước pháp luật. \n'
             '` NGƯỜI ĐẠI DIỆN THEO PHÁP LUẬT Hỗ sơ đính kèm: (Ký, ghi rõ họ tên, đóng dấu)'},
 {'id': 89,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 3',
               'article_title': 'Quyết định này có hiệu lực Kể từ ngày ký.',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 3. Quyết định này có hiệu lực Kể '
                                  'từ ngày ký.',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản, '
                         'Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 52,
               'chunk_part': '9/10'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 3. Quyết định này có hiệu lực Kể từ ngày ký. (tiếp '
             'theo)]\n'
             '1.\n'
             '2. \n'
             'Gi chú: ...(1)... Tên tễ chức đề nghị cấp lại Giấy phép hoạt động cung ứng dịch vụ '
             'trung gian thanh toán. : | \n'
             ' Mẫu số 12 TÊN TÔ CHỨC CUNG ỨNG DỊCH VỤ . CỘNG HOÀ XÃ HỘI CHỦ NGHĨA VIỆT NAM TRUNG '
             'GIAN THANH TOÁN Độc lập - Tự do - Hạnh phúc \n'
             '„ ngày....tháng.... năm \n'
             'ĐƠN ĐỀ NGHỊ SỬA ĐỎI, BỒ SUNG GIÁY PHÉP HOẠT ĐỘNG CUNG ỨNG DỊCH VỤ TRUNG GIAN THANH '
             'TOÁN \n'
             'Kính gửi: Thống đốc Ngân hàng Nhà nước Việt Nam. \n'
             'Căn cứ Nghị định số ò.0NĐ-CP ngày... tháng... năm... của Chính phủ quy định về thanh '
             'toán không dùng tiền mặt và các văn bản sửa đổi, bổ sung (nếu có); \n'
             'Căn cứ Thông tư số ‹../T1-NHNN ngày... tháng... năm... của Ngân hàng Nhà nước Việt '
             'Nam quy định về hoạt động cung ứng địch vụ trung gian thanh toán và các văn bản sửa '
             'đôi, bô sung (nêu có); \n'
             '.Căn cứ Nghị quyết cuộc họp Hội đồng thành viên/Hội đồng quản tr/Đại hội đồng cổ '
             'đông, văn bản của người đại điện có thâm quyền của chủ sở hữu của .-.(1)... thống '
             'nhất về việc thay đổi tên tổ chúức/địa điểm đặt trụ sở chínhngừng cung cấp một hoặc '
             'một sô dịch vụ trung gian thanh toán đã được cấp pháp/kêt nội thêm hệ thống thanh '
             'toán quốc tế của tổ chức cung ứng dịch vụ chuyên mạch tài chính quốc tế tại Giấy '
             'phép số... ngày... tháng... năm.. \n'
             'Căn cứ Giấy phép thành lập/Giấy chứng nhận đăng Đ doanh nghiệp/Giáy “ nhận đăng ký '
             'kinh doanh số... ngày... tháng... năm... được cấp lần thú... bởi... \n'
             'Căn cứ lý do khác (nếu có) ...(7)... đề nghị Ngân hàng Nhà nước Việt Nam sửa đổi, bổ '
             'sung Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán với nội dung cụ thê '
             'như sau:'},
 {'id': 90,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 3',
               'article_title': 'Quyết định này có hiệu lực Kể từ ngày ký.',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 3. Quyết định này có hiệu lực Kể '
                                  'từ ngày ký.',
               'topics': 'Ví điện tử, eKYC & Định danh, Hạn mức & Giao dịch, Mở & Đóng tài khoản, '
                         'Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 52,
               'chunk_part': '10/10'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 3. Quyết định này có hiệu lực Kể từ ngày ký. (tiếp '
             'theo)]\n'
             '1. Tên Tổ chức đề nghị sửa đổi, bổ sung Giấy phép hoạt động cung ứng dịch vụ trung '
             'gian thanh toán: \n'
             '- Tên đầy đủ bằng tiếng Việt: \n'
             '- Tên viết tắt bằng tiếng Việt (nếu có): \n'
             '- Tên đầy đủ bằng tiếng Anh (nếu có): \n'
             '- Tên viết tắt bằng tiếng Anh (nếu có): \n'
             '- Tên dùng để giao dịch (nếu có):\n'
             '\n'
             '2. Mã số doanh nghiệp/Mã số thuế:\n'
             '\n'
             '3. Địa điểm đặt trụ sở chính, số điện thoại, số Fax, Email: \n'
             '\n'
             '4. Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán SỐ... ngày... tháng... '
             'năm... của Thống đốc Ngân hàng Nhà nước Việt Nam.\n'
             '\n'
             '5. Lý do đề nghị sửa đổi, bổ sung Giấy phép (chọn ô tương ứng): \n'
             'Thay đổi tên tổ chức \n'
             'Tên Tế chức cung ứng dịch vụ trung gian thanh toán sau khi thay đổi: \n'
             '- Tên đầy đủ bằng tiếng Việt: \n'
             '- Tên viết tắt bằng tiếng Việt (nếu có): \n'
             '- Tên đầy đủ bằng tiếng Anh (nếu có): \n'
             '- Tên viết tắt bằng tiếng Anh (nếu có): \n'
             '- Tên dùng để giao dịch (nếu có): \n'
             'Thay đổi địa điểm đặt trụ sở chính \n'
             'Địa điểm đặt trụ sở chính thay đôi: \n'
             '„ 1 Ngừng cung cấp một hoặc một số dịch vụ trung gian thanh toán đã được \n'
             'cập phép \n'
             '- Tên (các) dịch vụ trung gian thanh toán ngừng cung ứng: \n'
             '- Thời gian dự kiến ngừng cung ứng dịch vụ trung gian thanh toán: \n'
             'L1 Kết nối thêm hệ thống thanh toán quốc tế của tổ chức cung ứng dịch vụ chuyển mạch '
             'tài chính quốc tê. \n'
             'Chúng tôi cam kết chịu trách nhiệm hoàn toàn trước pháp luật về tính hợp pháp, chính '
             'xác, trung thực của nội dung Đơn này. Đối với trường hợp ngừng cung cấp một hoặc một '
             'số dịch vụ trung gian thanh toán đã được câp phép, chúng tôi cam kết đã hoàn thành '
             'việc xử lý nghĩa vụ thuế, các khoản nợ, tài sản, quyền lợi khách hàng. Nếu vi phạm, '
             'chúng tôi xin chịu hoàn toàn trách nhiệm trước pháp \n'
             'luật. \n'
             '‹ NGƯỜI ĐẠI DIỆN THEO PHÁP LUẬT ˆ Hồ sơ đính kèm: (Kỷ, ghi rõ họ tên, đóng dấu)\n'
             '1.\n'
             '\n'
             '2. \n'
             'Ghi chú: ...(1)... Tên tổ chức đề nghị sửa đổi, bổ sung Giấy phép hoạt động cung ứng '
             'dịch vụ trung gian thanh toán. \n'
             ' Mẫu số 13 NGÂN HÀNG NHÀ NƯỚC VIỆTNAM CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM ———— Độc '
             'lập - Tự do - Hạnh phúc \n'
             'Số: .../GP-NHNN .., ngày... tháng... năm... \n'
             'GIẦY PHÉP Hoạt động cung ứng dịch vụ trung gian thanh toắn \n'
             'THÓNG ĐÓC NGÂN HÀNG NHÀ NƯỚC VIỆT NAM \n'
             'Căn cứ Luật Ngân hàng Nhà nước Việt Nam ngày 16 tháng 6 năm 2010; \n'
             'Căn cứ Nghị định số ‹NÐ-CP ngày... tháng... năm.. . của Chính phủ quy định về thanh '
             'toán không dùng tiền mặt và các văn bản sửa đổi, bồ sung (nếu có); \n'
             'Căn cứ Thông tư số .../ ./TT-NHNN ngày... tháng... năm... của Ngân hàng Nhà nước '
             'Việt Nam quy định về hoạt động cung ứng dịch vụ trung gian thanh toán và các văn bản '
             'sửa đổi, bổ sung (nếu có); \n'
             'Xét Đơn đề nghị cắp/cắp lại Giấy pháp hoạt động cung ứng dịch vụ trung gian thanh '
             'toán số... ngày... tháng... năm... và Hà sơ đính kèm của ... (1)..., \n'
             'Theo đề nghị của Vụ trưởng Vụ Thanh toán. QUYẾT ĐỊNH:'},
 {'id': 91,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 1',
               'article_title': 'Cấp phép hoạt động cung ứng dịch vụ trung gian thanh toán cho:',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 1. Cấp phép hoạt động cung ứng '
                                  'dịch vụ trung gian thanh toán cho:',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 52,
               'chunk_part': '1/1'},
  'content': 'Điều 1. Cấp phép hoạt động cung ứng dịch vụ trung gian thanh toán cho:\n'
             '\n'
             '1. Tên tổ chức: ...(1)... \n'
             '- Tên đầy đủ bằng tiếng Việt/tiếng Anh (nếu có): \n'
             '- Tên viết tắt bằng tiếng Việt và tiếng Anh, tên giao dịch (nếu có):\n'
             '\n'
             '2. Mã số doanh nghiệp:\n'
             '\n'
             '3. Địa điểm đặt trụ sở chính:'},
 {'id': 92,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 2',
               'article_title': '(Các) dịch vụ trung gian thanh toán được cắp/cấp lại Giấy phép:',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 2. (Các) dịch vụ trung gian '
                                  'thanh toán được cắp/cấp lại Giấy phép:',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 52,
               'chunk_part': '1/1'},
  'content': 'Điều 2. (Các) dịch vụ trung gian thanh toán được cắp/cấp lại Giấy phép:'},
 {'id': 93,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 3',
               'article_title': '(Các) điều khoản khác:',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 3. (Các) điều khoản khác:',
               'topics': 'Quy định chung',
               'is_core_fintech': False,
               'page': 52,
               'chunk_part': '1/1'},
  'content': 'Điều 3. (Các) điều khoản khác:'},
 {'id': 94,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 4',
               'article_title': 'Trong quá trình cung ứng các dịch vụ trung gian thanh toán, '
                                '...(1)... phải tuân thủ quy định tại Nghị định sô ...,..NĐÐ-CP '
                                'ngày... tháng... năm... của Chính phủ quy định về thanh toán '
                                'không dùng tiền mặt và các văn = sửa đổi, bỗ sung (nêu có); Thông '
                                'tư số .../.../TT-NHNN ngày... tháng... . Của Thống đốc Ngân hàng '
                                'Nhà nước Việt Nam quy định về hoạt động cùng ủi ứng dịch',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 4. Trong quá trình cung ứng các '
                                  'dịch vụ trung gian thanh toán, ...(1)... phải tuân thủ quy định '
                                  'tại Nghị định sô ...,..NĐÐ-CP ngày... tháng... năm... của Chính '
                                  'phủ quy định về thanh toán không dùng tiền mặt và các văn = sửa '
                                  'đổi, bỗ sung (nêu có); Thông tư số .../.../TT-NHNN ngày... '
                                  'tháng... . Của Thống đốc Ngân hàng Nhà nước Việt Nam quy định '
                                  'về hoạt động cùng ủi ứng dịch',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 65,
               'chunk_part': '1/1'},
  'content': 'Điều 4. Trong quá trình cung ứng các dịch vụ trung gian thanh toán, ...(1)... phải '
             'tuân thủ quy định tại Nghị định sô ...,..NĐÐ-CP ngày... tháng... năm... của Chính '
             'phủ quy định về thanh toán không dùng tiền mặt và các văn = sửa đổi, bỗ sung (nêu '
             'có); Thông tư số .../.../TT-NHNN ngày... tháng... . Của Thống đốc Ngân hàng Nhà nước '
             'Việt Nam quy định về hoạt động cùng ủi ứng dịch \n'
             'vụ trung gian thanh toán và các văn bản sửa đổi, bổ sung (nếu có); các quy định khác '
             'của pháp luật có liên quan.'},
 {'id': 95,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 5',
               'article_title': 'Hiệu lực thi hành Giấy phép này có thời hạn hoạt động L0 nắm kể '
                                'từ ngày ký (đối với trường hợp cấp Giấy phép quy định tại Điều 24 '
                                'Nghị định này).',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 5. Hiệu lực thi hành Giấy phép '
                                  'này có thời hạn hoạt động L0 nắm kể từ ngày ký (đối với trường '
                                  'hợp cấp Giấy phép quy định tại Điều 24 Nghị định này).',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 66,
               'chunk_part': '1/1'},
  'content': 'Điều 5. Hiệu lực thi hành Giấy phép này có thời hạn hoạt động L0 nắm kể từ ngày ký '
             '(đối với trường hợp cấp Giấy phép quy định tại Điều 24 Nghị định này). \n'
             'Giấy phép này có hiệu lực kê từ ngày ký và thay thế Giấy phép SỐ... . ngày.. '
             'tháng... . của Thống đốc Ngân hàng Nhà nước Việt Nam về việc cập Giấy phép hoạt động '
             'cung ứng dịch vụ trung gian thanh toán cho...(1)... (đổi với trường hợp cấp lại Giáy '
             'phép quy định tại khoản 1 Điễu 25 Nghị định này). \n'
             'Giấy phép nảy có hiệu lực kế từ ngày ký, có "ẻ hạn đến ngày... tháng... năm... và '
             'thay thế Giấy phép SỐ.. „ ngày... tháng... . của Thông đốc Ngân hàng Nhà nước Việt '
             'Nam về việc cấp Giây phép hoạt động cung ứng dịch vụ trung gian thanh toán '
             'cho...(1)... (đối với trường hợp quy định tại khoản 2 Điều 25 Nghị định này). \n'
             'Nơi nhận: THÓNG ĐÓC - Như Điều 1; \n'
             '- Ban Lãnh đạo NHNN; \n'
             '- Cục CNTT, Vụ PC \n'
             '- CQTTGSNH (đẻ phối hợp); \n'
             '- Sở Giao dịch \n'
             '~ NHNN chỉ nhánh ... \n'
             'Gñhỉ chú: (1) Tên tổ chức đề nghị cấp hoặc cấp lại Giấy phép hoạt động cung ứng dịch '
             'vụ trung gian thanh toán. \n'
             ' Mẫu số 14 \n'
             'NGÂN HÀNG NHÀ NƯỚC VIỆT NAM CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - '
             'Hạnh phúc \n'
             'Số../QĐNHNNN , ngày... thẳng... năm... \n'
             'QUYẾT ĐỊNH Về việc chấp thuận sửa đẫi, bố sung nội dung Giấy phép hoạt động cung ứng '
             'dịch vụ trung gian thanh toán của...(1)... \n'
             'THÓNG ĐÓC NGÂN HÀNG NHÀ NƯỚC VIỆT NAM \n'
             'Căn cứ Luật Ngân hàng Nhà nước Việt Nam ngày 16 tháng 6 năm 2010; \n'
             'Căn cứ Nghị ã¡h số -ẢNĐ- CP ngày... tháng... năm... của Chính phủ quy định về thanh '
             'toán Không dùng tiền mặt và các văn bản sửa đổi, bỗ sung (nếu có); \n'
             'Căn cứ Thông tư số .../.../TT-NHNN ngày... tháng... năm.. . của Thống đốc Ngân hàng '
             'Nhà nước Việt Nam quy định về hoạt động cung ứng dịch vụ trung gian thanh toán và '
             'các văn bản sửa đổi, bỗ sung (nếu có); \n'
             'Xét Đơn đề nghị sửa đổi, bồ sung Giấy phép hoạt động cung ứng dịch vụ trung gian '
             'thanh toán SỐ... ngày... tháng... năm... và Hồ sơ đính kèm (nếu có) của...(1)...; \n'
             'Theo đề nghị của Vụ trưởng Vụ Thanh toán. QUYÉT ĐỊNH:'},
 {'id': 96,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 1',
               'article_title': 'Sửa đổi, bổ sung Điều 1 và/hoặc Điều 2 Giấy phép hoạt động cung '
                                'ứng địch vụ trung gian thanh toán cho...(1)..., như sau:',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 1. Sửa đổi, bổ sung Điều 1 '
                                  'và/hoặc Điều 2 Giấy phép hoạt động cung ứng địch vụ trung gian '
                                  'thanh toán cho...(1)..., như sau:',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 66,
               'chunk_part': '1/1'},
  'content': 'Điều 1. Sửa đổi, bổ sung Điều 1 và/hoặc Điều 2 Giấy phép hoạt động cung ứng địch vụ '
             'trung gian thanh toán cho...(1)..., như sau:'},
 {'id': 97,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 2',
               'article_title': 'Quyết định này có hiệu lực kế từ ngày ký và là một bộ phận không '
                                'thể tách rời của Giấy r phép hoạt động cung ứng dịch vụ trung '
                                'gian thanh toán số. . ngày... tháng... . của Thống đốc Ngân hàng '
                                'Nhà nước Việt Nam cấp cho...(1)...',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 2. Quyết định này có hiệu lực kế '
                                  'từ ngày ký và là một bộ phận không thể tách rời của Giấy r phép '
                                  'hoạt động cung ứng dịch vụ trung gian thanh toán số. . ngày... '
                                  'tháng... . của Thống đốc Ngân hàng Nhà nước Việt Nam cấp '
                                  'cho...(1)...',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 67,
               'chunk_part': '1/1'},
  'content': 'Điều 2. Quyết định này có hiệu lực kế từ ngày ký và là một bộ phận không thể tách '
             'rời của Giấy r phép hoạt động cung ứng dịch vụ trung gian thanh toán số. . ngày... '
             'tháng... . của Thống đốc Ngân hàng Nhà nước Việt Nam cấp cho...(1)...\n'
             '\n'
             'Điều 3....(1)... phải:\n'
             '\n'
             '1. Sửa đổi, bổ sung Điều lệ phù hợp với nội dung thay đổi tại Điều 1 Quyết định '
             'này.\n'
             '\n'
             '2. Gửi thông báo bằng văn bản tới các tổ chức và cá nhân liên quan để thanh lý hợp '
             'đồng và hoàn tất các nghĩa vụ, trách nhiệm giữa các bên liên quan theo quy \n'
             'định của pháp luật trong thời hạn 30 ngày kê từ ngày Quyết định này có hiệu lực '
             '(trường hợp tổ chức đề nghị ngừng cung cấp dịch vụ trung gian thanh toán)..'},
 {'id': 98,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 4',
               'article_title': '(Nội dung...nêu tại) Quyết định số.. . ngày... tháng... năm.. của '
                                'Thống đốc Ngân hàng Nhà nước Việt Nam về việc sửa đôi, bố sung '
                                'nội dung Giấy phép hoạt động cung ứng dịch vụ trung gian thanh '
                                'toán của.. .(1)... hết hiệu lực kể từ ngày Quyết định này có hiệu '
                                'lực (nếu có).',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 4. (Nội dung...nêu tại) Quyết '
                                  'định số.. . ngày... tháng... năm.. của Thống đốc Ngân hàng Nhà '
                                  'nước Việt Nam về việc sửa đôi, bố sung nội dung Giấy phép hoạt '
                                  'động cung ứng dịch vụ trung gian thanh toán của.. .(1)... hết '
                                  'hiệu lực kể từ ngày Quyết định này có hiệu lực (nếu có).',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 67,
               'chunk_part': '1/1'},
  'content': 'Điều 4. (Nội dung...nêu tại) Quyết định số.. . ngày... tháng... năm.. của Thống đốc '
             'Ngân hàng Nhà nước Việt Nam về việc sửa đôi, bố sung nội dung Giấy phép hoạt động '
             'cung ứng dịch vụ trung gian thanh toán của.. .(1)... hết hiệu lực kể từ ngày Quyết '
             'định này có hiệu lực (nếu có).\n'
             '\n'
             'Điều 5..'},
 {'id': 99,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 6',
               'article_title': 'Chánh Văn phòng, Vụ trưởng Vụ Thanh toán, Người đại diện theo, '
                                'pháp luật của. ..(1)}... chịu trách nhiệm thi hành Quyết định '
                                'này.',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 6. Chánh Văn phòng, Vụ trưởng Vụ '
                                  'Thanh toán, Người đại diện theo, pháp luật của. ..(1)}... chịu '
                                  'trách nhiệm thi hành Quyết định này.',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 68,
               'chunk_part': '1/2'},
  'content': 'Điều 6. Chánh Văn phòng, Vụ trưởng Vụ Thanh toán, Người đại diện theo, pháp luật '
             'của. ..(1)}... chịu trách nhiệm thi hành Quyết định này. \n'
             'Nơi nhận: THÓNG ĐÓC - Như Điều 5; \n'
             '- Ban Lãnh đạo NHNN; \n'
             '~ Cục CNTT, Vụ PC \n'
             '- CQTTGSNH (để phối hợp); \n'
             '- Sở Giao dịch \n'
             '- NHNN chỉ nhánh... \n'
             '- Vụ Truyền thông (để công bổ); \n'
             '- Lưu:...... \n'
             'Ghi chú: ...(1)... Tên tổ chức đề nghị sửa đổi, bỗ sung Giấy phép hoạt động cung ứng '
             'địch vụ trung gian thanh toán. \n'
             ' Mẫu số 15 \n'
             'TÊN TÔ CHỨC CUNG ỨNG DỊCH VỤ _ CỘNG HOÀ XÃ HỘI CHỦ NGHĨA VIỆT NAM TRUNG GIAN THANH '
             'TOÁN Độc lập - Tự do - Hạnh phúc \n'
             '..› Ngày...... thẳng...... nĂm..... \n'
             'ĐƠN ĐÈ NGHỊ THU HÔI GIÁY PHÉP HOẠT ĐỘNG CUNG ỨNG DỊCH VỤ TRUNG GIAN THANH TOÁN \n'
             'Kính gửi: Thống đốc Ngân hàng Nhà nước Việt Nam. \n'
             'Căn cứ Nghị định số .../ -.NĐ-CP ngày... tháng... nĂm.. . của Chính phủ quy định về '
             'thanh toán không đùng tiễn mặt và các văn bản sửa đối, bố sung (nếu có); \n'
             'Căn cứ Thông tư số .,../TT-NHNN ngày... tháng... năm.. . của Ngân hàng Nhà nước Việt '
             'Nam 4⁄y đinh về hoạt động cung ứng dịch vụ trung gian thanh toán và các văn bản sửa '
             'đối, bồ sung (nếu có); \n'
             'Căn cứ Nghị quyết cuộc họp Hội đồng thành viên/Hội đồng quản tr/Đại hội \n'
             'đẳng cổ đông, văn bản của người đại điện có thẩm quyên của chủ sở hữu của \n'
             '(1)... thống nhất về việc ngừng cung ứng dịch vụ trung gian Tang toán Siêu được Ngân '
             'hàng Nhà nước cấp phép tại Giấy phép số... ngày... tháng... " \n'
             '...(1)... đề nghị Ngân hàng Nhà nước thu hồi Giấy phép hoạt động cung ứng dịch vụ '
             'trung gian thanh toán số .... ngày... tháng... năm ... với nội dung cụ thể như sau:\n'
             '\n'
             '1. Tên Tổ chức cung ứng dịch vụ trung gian thanh toán: - Tên đẩy đủ bằng tiếng '
             'Việt: \n'
             '- Tên viết tắt bằng tiếng Việt (nếu có): \n'
             '- Tên đầy đủ bằng tiếng Anh (nếu có): \n'
             '- Tên viết tắt bằng tiếng Anh (nếu có): \n'
             '- Tên dùng để giao dịch (nếu có):\n'
             '\n'
             '2. Giấy phép thành lập/Giấy chứng nhận đăng ký doanh nghiệp Giấy chứng nhận đăng ký '
             'kinh doanh số... được cấp bởi... ngày... tháng... năm..\n'
             '\n'
             '3. Mã số doanh nghiệp/Mã số thuế:\n'
             '4. Địa điểm đặt trụ sở chính, số điện thoại, số Fax, Email:\n'
             '\n'
             '5. Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán sỐ... ngày... tháng... '
             'năm... do Ngân hàng Nhà nước Việt Nam cập.\n'
             '\n'
             '6. Lý do đề nghị thu hồi (chọn ô tương ứng): Œ Tổ chức bị giải thể hoặc phá sản. \n'
             ' - Tài liệu đính kèm: \n'
             '1 Chấm dứt hoạt động cung ứng dịch vụ trung gian thanh toán \n'
             '- Tài liệu đính kèm: \n'
             '- Thời gian dự kiến chấm đứt cung ứng dịch vụ trung gian thanh toán: \n'
             'Chúng tôi cam kết đã hoàn thành việc xử lý nghĩa vụ thuế, các khoản nợ, tài sản, '
             'quyền lợi khách hàng và chịu trách nhiệm hoàn toàn trước pháp luật về tính \n'
             'hợp pháp, chính xác, trung thực của nội dung Đơn này. Ñếu vi phạm, chúng tôi xin '
             'chịu hoàn toàn trách nhiệm trước pháp luật. \n'
             '` NGƯỜI ĐẠI DIỆN THEO PHÁP LUẬT Hồ sơ đính kèm: (Ký, ghỉ rõ họ tên, đóng dấu)'},
 {'id': 100,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 6',
               'article_title': 'Chánh Văn phòng, Vụ trưởng Vụ Thanh toán, Người đại diện theo, '
                                'pháp luật của. ..(1)}... chịu trách nhiệm thi hành Quyết định '
                                'này.',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 6. Chánh Văn phòng, Vụ trưởng Vụ '
                                  'Thanh toán, Người đại diện theo, pháp luật của. ..(1)}... chịu '
                                  'trách nhiệm thi hành Quyết định này.',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 68,
               'chunk_part': '2/2'},
  'content': '[Nghị định 52/2024/NĐ-CP - Điều 6. Chánh Văn phòng, Vụ trưởng Vụ Thanh toán, Người '
             'đại diện theo, pháp luật của. ..(1)}... chịu trách nhiệm thi hành Quyết định này. '
             '(tiếp theo)]\n'
             '1.\n'
             '2. \n'
             'Ghi chú: ...(1)... Tên tổ chức cung ứng dịch vụ trung gian thanh toán. \n'
             ' Mẫu số 16 \n'
             'NGÂN HÀNG NHÀ NƯỚC VIỆT NAM CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - '
             'Hạnh phúc \n'
             'Số: ....QĐ-NHNN .., ngày... tháng... năm \n'
             'QUYÉT ĐỊNH Về việc thu hồi Giấy phép hoạt động cung ứng dịch vụ trung gian thanh '
             'toán của...(1)... \n'
             'THÓNG ĐÓC NGÂN HÀNG NHÀ NƯỚC VIỆT NAM \n'
             'Căn cứ Luật Ngân hàng Nhà nước Việt Nam ngày 16 tháng 6 năm 2010; Căn cứ Nghị đjnh '
             'số ‹./NÐ-CP ngày... tháng... năm... của Chính phú quy định về thanh toán không dùng '
             'tiền mặt và các văn bản sửa đổi, bổ sung (nếu có); \n'
             'Căn cứ Thông tư số .../...TT-NHNN ngày... tháng... năm... của Thống đốc Ngân hàng '
             'Nhà nước Việt Nam quy định về hoạt động cung ứng dịch vụ trung gian thanh toán và '
             'các văn bản sửa đổi, bồ sung (nếu có); \n'
             'Xét đề ñghị của ...(])... (vường hợp tổ chức có Đơn đề nghị thu hôi Giấy phép đo bị '
             'giải thể hoặc phá sản hoặc ngừng cưng ứng hoại động cung ứng dịch Vụ rUNE gian thanh '
             'toán)/ đê nghị của cơ quan nhà nước có thẩm quyên (rường hợp có bản án, quyết định '
             'thì hành án, quyết định xử phạt vi phạm hành chính có nội dung yêu cấu thu hồi Giấy '
             'phép); \n'
             'Theo đề nghị của Vụ trưởng Vụ Thanh toán. QUYÉT ĐỊNH:'},
 {'id': 101,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 1',
               'article_title': 'Thu hồi Giấy phép hoạt động cung ứng dịch vụ trung gian thanh '
                                'toán số... ngày... tháng... năm... của Thống đốc Ngân hàng Nhà '
                                'nước Việt Nam cấp cho:',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 1. Thu hồi Giấy phép hoạt động '
                                  'cung ứng dịch vụ trung gian thanh toán số... ngày... tháng... '
                                  'năm... của Thống đốc Ngân hàng Nhà nước Việt Nam cấp cho:',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 71,
               'chunk_part': '1/1'},
  'content': 'Điều 1. Thu hồi Giấy phép hoạt động cung ứng dịch vụ trung gian thanh toán số... '
             'ngày... tháng... năm... của Thống đốc Ngân hàng Nhà nước Việt Nam cấp cho:\n'
             '\n'
             '1. Tên tổ chức: \n'
             '- Tên đầy đủ bằng tiếng Việt: - Tên đầy đủ bằng tiếng Anh: - Tên viết tắt: \n'
             '~- Tên giao dịch:\n'
             '\n'
             '2. Mã số doanh nghiệp:\n'
             '\n'
             '3. Địa điểm đặt trụ sở chính:\n'
             '4. Lý do thu hồi:'},
 {'id': 102,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 2',
               'article_title': 'Ngay khi nhận được Quyết định này, ...(1)... phải lập tức ngừng '
                                'cung ứng các dịch vụ trưng gian thanh toán. Trong thời hạn 30 '
                                'ngày kê từ ngày Quyết định này có hiệu lực, ...(1)... phải gửi '
                                'thông báo bằng văn bản tới các tổ | chức và cá nhân liên quan để '
                                'thanh lý hợp đồng và hoàn tất các nghĩa vụ, trách, nhiệm giữa các '
                                'bên.',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 2. Ngay khi nhận được Quyết định '
                                  'này, ...(1)... phải lập tức ngừng cung ứng các dịch vụ trưng '
                                  'gian thanh toán. Trong thời hạn 30 ngày kê từ ngày Quyết định '
                                  'này có hiệu lực, ...(1)... phải gửi thông báo bằng văn bản tới '
                                  'các tổ | chức và cá nhân liên quan để thanh lý hợp đồng và hoàn '
                                  'tất các nghĩa vụ, trách, nhiệm giữa các bên.',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 71,
               'chunk_part': '1/1'},
  'content': 'Điều 2. Ngay khi nhận được Quyết định này, ...(1)... phải lập tức ngừng cung ứng các '
             'dịch vụ trưng gian thanh toán. Trong thời hạn 30 ngày kê từ ngày Quyết định này có '
             'hiệu lực, ...(1)... phải gửi thông báo bằng văn bản tới các tổ | chức và cá nhân '
             'liên quan để thanh lý hợp đồng và hoàn tất các nghĩa vụ, trách, nhiệm giữa các bên.'},
 {'id': 103,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 3',
               'article_title': 'Quyết định này có hiệu lực kế từ ngày ký.',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 3. Quyết định này có hiệu lực kế '
                                  'từ ngày ký.',
               'topics': 'Quy định chung',
               'is_core_fintech': False,
               'page': 71,
               'chunk_part': '1/1'},
  'content': 'Điều 3. Quyết định này có hiệu lực kế từ ngày ký.'},
 {'id': 104,
  'metadata': {'doc_code': 'Nghị định 52/2024/NĐ-CP',
               'doc_name': 'Nghị định quy định về thanh toán không dùng tiền mặt',
               'effective_date': '01/07/2024',
               'chapter': 'Chương VỊ HOẠT ĐỘNG GIÁM SÁT',
               'article': 'Điều 4',
               'article_title': 'Chánh Văn phòng, Vụ trưởng Vụ Thanh toán, Thủ trưởng các đơn vị '
                                'thuộc Ngân hàng Nhà nước Việt Nam, Người đại diện theo pháp luật '
                                'của.. (1)... chịu trách nhiệm thi hành Quyết định này.',
               'citation_header': 'Nghị định 52/2024/NĐ-CP - Điều 4. Chánh Văn phòng, Vụ trưởng Vụ '
                                  'Thanh toán, Thủ trưởng các đơn vị thuộc Ngân hàng Nhà nước Việt '
                                  'Nam, Người đại diện theo pháp luật của.. (1)... chịu trách '
                                  'nhiệm thi hành Quyết định này.',
               'topics': 'Ví điện tử, Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 71,
               'chunk_part': '1/1'},
  'content': 'Điều 4. Chánh Văn phòng, Vụ trưởng Vụ Thanh toán, Thủ trưởng các đơn vị thuộc Ngân '
             'hàng Nhà nước Việt Nam, Người đại diện theo pháp luật của.. (1)... chịu trách nhiệm '
             'thi hành Quyết định này. \n'
             'Nơi nhận: THÓNG ĐÓC - Như Điều 4; \n'
             '- Ban Lãnh đạo NHNN; \n'
             '- Cục CNTT, Vụ PC : \n'
             '- CQTTGSNH (để phối hợp); \n'
             '- Sở Giao dịch \n'
             '~ NHNN chỉ nhánh.. \n'
             '- Vụ Truyền thông (để công bố); \n'
             '- Lưu:. \n'
             'Ghi chú: (1) Tên Tổ chức bị thu hồi Giấy phép hoạt động cung ứng dịch vụ trung gian '
             'thanh toán.'},
 {'id': 105,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Mở đầu',
               'article_title': 'Căn cứ ban hành',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Phần mở đầu / Phụ lục',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 1,
               'chunk_part': '1/1'},
  'content': 'Người ký: CÔNG THÔNG TIN ĐIỆN TỬ CHÍNH PHỦ Email: thongtinchinhphu@)chinhphu.vn Cơ '
             'quan: VĂN PHÒNG CHÍNH PHÙ. \n'
             'GHINHPHILL.V NThời gian ký: 20.03.2024 11:00:01 +07:00 \n'
             '. BỘ TÀI CHÍNH: CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc Số: '
             '47/2024/TT-BTC Hà Nội, ngày 44 tháng Ó3 năm 2024 “GỐNG THÔNG TIN BIỆN TỪ CHÍNH PHÙ \n'
             '- THÔNG TƯ Hướng dẫn kiểm soát, thanh toán các khoản chỉ thường xuyên qua Kho bạc '
             'Nhà nước \n'
             'ý. „ GIỜ:........Š.............. ĐẾN (uy 70J212024.. \n'
             'Căn cứ Luật Ngân sách Nhà nước ngày 25 tháng 06 năm 2015, \n'
             'Căn cứ Nghị định số 163/2016/NĐ-CP ngày 21 tháng 12 năm 2016 của Chính phủ quy định '
             'chỉ tiệt thi hành một số điêu của Luật Ngân sách Nhà nước; \n'
             'Căn cứ Nghị định số 130/2005/NĐ-CP ngày 17 tháng 10 năm 2005 của Chính phủ quy định '
             'chế độ tự chủ, tự chịu trách nhiệm về sử dụng biên chế và kinh phí quản lý hành '
             'chính đổi với cơ quan nhà nước; \n'
             'Căn cứ Nghị định số 117/2013/NĐ-CP ngày 07 tháng 10 năm 2013 của Chính phủ sửa đổi, '
             'bố sung một số điều của Nghị định số 130/2005/NĐ-CP ngày 17 tháng 10 năm 2005 của '
             'Chính phú quy định chế độ tự chủ, tự chịu trách nhiệm \n'
             'về sử dụng biên chế và kinh phí quản lý hành chính đối với cơ quan nhà nước; \n'
             'Căn cứ Nghị định số 60/2021/NĐ-CP ngày 21 tháng 06 năm 2021 của Chính phủ quy định '
             'cơ chế tự chủ tài chính của đơn vị sự nghiệp công lập,. \n'
             'Căn cứ Nghị định số 120/2016/NĐ-CP ngày 23 tháng năm 2016 của Chính phủ quy định chỉ '
             'tiết và hướng dán thi hành một số điêu của Luật Phí và lệ phí, \n'
             'Căn cứ Nghị định số 82/2023/NĐ-CP ngày 28 tháng 11 năm 2023 của Chính phủ sửa đối, '
             'bổ sung một số điêu của Nghị định số 120/2016/NĐ-CP ngày \n'
             '-23 tháng 8 năm 2016 của Chính phủ quy định chỉ tiết và hướng dẫn thi hành một số '
             'điều của Luật Phí và lệ phí, \n'
             'Căn cứ Quyết định số 50/2017/QĐ-TTg ngày 31 tháng 12 năm 2017 của Thủ tướng Chính '
             'phủ quy định tiêu chuẩn, định mức sử dụng máy móc, thiết bị; \n'
             'Căn cứ Nghị định số 1 31⁄201 7/NĐ-CP ngày 26 tháng 12 năm 2017 của “Chính phủ quy '
             'định chỉ tiết một số điêu của Luật Quản lý, sử dụng tài sản công; \n'
             '-_ Căn cứ Nghị định số 85/2018/NĐ-CP ngày 30 tháng 5 năm 2018 của Chính phú quy định '
             'tiêu chuẩn, định mức sử dụng xe ô tô tại đơn vị lực lượng vũ trang nhân dân; \n'
             'Căn cứ Nghị định số 72/2023/NĐ-CP ngày 26 thắng 09 năm 2023 của Chính phú quy định '
             'tiêu chuẩn, định mức sử đụng xe Ô tô; \n'
             '`& ° \n'
             ' Căn cứ Nghị định số 32/2019/NĐ-CP ngày lị 0 tháng 4 năm 2019 của Chính phú quy định '
             'giao nhiệm vụ, đặt hàng hoặc đầu thắu cung cấp sản phẩm, dịch vụ công sử dụng ngân '
             'sách nhà nước từ nguôn kinh phí chỉ thường xuyên; \n'
             'Căn cứ Nghị định số 11/2020/NĐ-CP ngày 20 tháng 01 năm 2020 của Chính phủ quy định '
             'về thủ tục hành chính thuộc lĩnh vực Kho bạc Nhà nước; \n'
             'Căn cứ Nghị định số I 4/2023/NĐ-CP ngày 20 tháng 4 năm 2023 của Chính \n'
             '* \n'
             "' phủ quy định chức năng, nhiệm vụ, quyền hạn và cơ cầu tổ chức của Sộ Tài chính; "
             '" \n'
             'Theo đề nghị của Tổng Giám đốc Kho bạc Nhà nước; \n'
             'Bộ trưởng Bộ Tài chính ban hành Thông tư hướng dẫn kiểm soát, thanh toán các khoản '
             'chỉ thường xuyên qua Kho bạc Nhà nước.\n'
             '\n'
             'Chương I QUY ĐỊNH CHUNG'},
 {'id': 106,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 1',
               'article_title': 'Phạm vi điều chỉnh và đối tượng áp dụng .',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 1. Phạm vi điều chỉnh và đối '
                                  'tượng áp dụng .',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 1,
               'chunk_part': '1/1'},
  'content': 'Điều 1. Phạm vi điều chỉnh và đối tượng áp dụng . \n'
             '—.......`.. na... -....sannn\n'
             '\n'
             '1.1. Thông tư này hướng dẫn cơ chế kiểm soát, thanh toán các khoản chỉ thường xuyên '
             'qua Kho bạc Nhà nước từ các nguồn: |\n'
             '\n'
             'a) Nguồn Ngân sách nhà nước @NSNN); -\n'
             'b) Nguồn phí được để lại theo chế độ quy định và các nguồn từ khoản thu hợp pháp '
             'khác của cơ quan nhà nước;\n'
             '\n'
             'c) Nguồn thu từ dịch vụ khám bệnh, chữa bệnh, dịch + vụ y tế dự phòng, Học phí của '
             'đơn vị sự nghiệp công tự bảo đảm một phần chỉ thường xuyên (đơn vị \n'
             'cm m— nhóm 3} và đơn-v† sự nghiệp công do- nhà nước "bảo đảm chỉ thường xuyên (đơn '
             '——” \n'
             'vị nhóm 4) và nguồn thu phí theo pháp luật về phí, lệ phí của đơn Vị sự nghiệp công '
             'lập theo quy định tại khoản 2, khoản 3 Điều 23 Nghị định số 60/2021/NĐ- CP ngày 21 '
             'tháng 06 năm 2021 của Chính phủ quy định cơ chế tự chủ tài chính của đơn vị sự '
             'nghiệp công lập (Nghị định số 60/2021/NĐ-CP).\n'
             '\n'
             '1.2. Thông tư này không hướng dẫn đối với các khoản chỉ của cơ quan đại diện Việt '
             'Nam ở nước ngoài; các khoản chỉ có cơ chế hướng dẫn riêng theo quy \n'
             'định của cấp có thẩm quyền.\n'
             '2. Đối tượng áp dụng: \n'
             'Thông tư này áp dụng đối với cơ quan, đơn vị, tổ chức, cá nhân giao dịch với hệ '
             'thống Kho bạc Nhà nước; Kho bạc Nhà nước; Cơ quan tài chính các cấp.'},
 {'id': 107,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 2',
               'article_title': 'Nguyên tắc kiểm soát, thanh toán qua Kho bạc Nhà nước',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 2. Nguyên tắc kiểm soát, thanh '
                                  'toán qua Kho bạc Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 3,
               'chunk_part': '1/1'},
  'content': 'Điều 2. Nguyên tắc kiểm soát, thanh toán qua Kho bạc Nhà nước \n'
             '" . \n'
             '\n'
             '1. Kho bạc Nhà nước thực hiện kiểm soát, thanh toán theo quy định tại khoản 5 Điều '
             '56 Luật NSNN số 83/2015/QH13, cụ thể:\n'
             '\n'
             '1.1. Chí NSNN chỉ được thực biện khi đã có trong dự toán ngân sách được giao, trừ '
             'trường hợp quy định tại Điều 5] của Luật NSNN; đã được thủ trưởng đơn vị sử dụng '
             'ngân sách, chủ đầu tư hoặc người được ủy quyền quyết định chị, số dư tài khoản của '
             'đơn vị còn đủ để chỉ.\n'
             '\n'
             '1.2. Kho bạc Nhà nước kiểm tra tính hợp pháp của chứng từ chỉ và các hỗ sơ kèm theo '
             'chứng từ chỉ thuộc thủ tục hành chính gửi Kho bạc Nhà nước được quy định tại các '
             'Nghị định của Chính phủ. Trường hợp chứng từ chỉ và các hồ sơ kèm theo chứng từ chỉ '
             'đơn vị sử dụng ngân sách gửi Kho bạc Nhà nước cổ tình giả mạo, thay thế nội dung, '
             'đơn vị sử dụng ngân sách chịu trách nhiệm theo quy định của pháp luật. \n'
             'Đối với các hồ sơ không thuộc thủ tục hành chính gửi Kho bạc Nhà nước được quy định '
             'tại các Nghị định của Chính phủ, đơn vị sử dụng ngân sách chịu trách nhiệm kiểm soát '
             'nội dung thanh toán, hồ sơ thanh toán theo đúng quy định của pháp luật.\n'
             '\n'
             '1.3. Kho bạc Nhà nước kiểm soát định mức (mức chỉ) theo quy định tại các văn bản quy '
             'phạm pháp luật. Trường hợp các cơ quan, đơn vị đã được cấp có thấm quyền cho phép '
             'thực hiện theo cơ chế tự chủ thì kiểm soát đảm bảo theo đúng quy chế chỉ tiêu nội bộ '
             'và phù hợp với dự toán được giao tự chủ.\n'
             '\n'
             '2. Các khoản chỉ thực hiện theo nguyên tắc tạm ứng thanh toán theo quy định tại '
             'khoản 5 Điều 34 Nghị định số 163/2016/NĐ-CP ngày 21 tháng 12 năm 2016 của Chính phủ '
             'quy định chỉ tiết thi hành một số điều của Luật Ngân sách Nhà nước (Nghị định số '
             '163/2016/NĐ-CP).\n'
             '\n'
             '3. Các khoản chỉ tạm ứng và thanh toán bằng tiền mặt phải được thực hiện: đúng theo '
             'quy định tại Thông tư hướng dẫn quản lý thủ chỉ bằng tiền mặt qua hệ thống Kho bạc '
             'Nhà nước của Bộ Tài chính. ˆ .\n'
             '\n'
             '4. Trường hợp các khoản chỉ NSNN thực hiện bằng hình thức giao dịch điện tử qua '
             'Trang thông tin dịch vụ công của Kho bạc Nhà nước, việc kiểm soát, thanh toán của '
             'Kho bạc Nhà nước phải tuân thủ quy định về giao dịch điện tử trong hoạt động nghiệp '
             'vụ Kho bạc Nhà nước tại: Nghị định số 165/2018/NĐ- CP ngày 24 tháng 12 năm 2018 của '
             'Chính phủ về giao dịch điện tử trong hoạt động tài chính; Nghị định số 11/2020/NĐ-CP '
             'ngày 20 tháng 01 năm 2020 của Chính phủ quy định về thủ tục hành chính thuộc lĩnh '
             'vực Kho bạc Nhà nước (Nghị định số 11/2020/NĐ- -CP); Nghị định số 45/2020/NĐ-CP ngày '
             "08 tháng 4 năm '2020 của Chính phủ về thực hiện thủ tục hành chính trên môi trường "
             'điện tử; Thông tư số 87/2021/TT-BTC ngày 08 tháng 10 năm 2021 của Bộ Tài chính quy '
             'định về giao dịch điện tử trong hoạt động nghiệp vụ Kho bạc Nhà nước. \n'
             '` \n'
             ' 4 Ï | \n'
             '['},
 {'id': 108,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 3',
               'article_title': 'Hình thức kiểm soát, thanh toán các khoản chỉ thường xuyên qua '
                                'Kho bạc Nhà nước',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 3. Hình thức kiểm soát, thanh '
                                  'toán các khoản chỉ thường xuyên qua Kho bạc Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản',
               'is_core_fintech': True,
               'page': 3,
               'chunk_part': '1/1'},
  'content': 'Điều 3. Hình thức kiểm soát, thanh toán các khoản chỉ thường xuyên qua Kho bạc Nhà '
             'nước \n'
             'Kho bạc Nhà nước thực hiện kiểm soát, thanh toán các khoản chỉ thường ¡ \n'
             'xuyên từ NSNN qua Kho bạc Nhà nước theo các hình thức sau:\n'
             '1. Thanh toán trước, kiểm soát sau:\n'
             '\n'
             '1.1. Thanh toán trước, kiểm soát sau là hình thức thanh toán áp dụng đối - với từng '
             'lần thanh toán của các hợp đồng thanh toán nhiều lần, trừ lần thanh toán cuối cùng. '
             'Sau khi nhận đủ hồ sơ hợp lệ, hợp pháp, Kho bạc Nhà nước làm thủ tục thánh toán cho '
             'đối tượng thụ hưởng trong thời hạn 01 ngày làm VIỆC; đồng thời, gửi 01 chứng từ báo '
             'nợ cho đơn vị để xác nhận đã thực hiện thanh toán, gửi 01 chứng từ báo có cho đơn vị '
             '(nếu đơn vị thụ hưởng mở tài khoản tại Kho bạc Nhà nước).\n'
             '1.2: Trong thời bạn 01 ngày làm việc kể từ ngày thanh toán, Kho bạc Nhà nước thực '
             'hiện kiểm soát hồ sơ theo chế độ quy định.\n'
             '\n'
             '1.3. Trường hợp sau khi kiểm soát phát hiện khoản chỉ không đảm bảo đúng ....chê độ '
             'quy định, Kho bạc Nhà nước có văn bản thông báo kết quả kiểm soát chỉ — (theo Mẫu số '
             '01 kèm theo Thông tư này) gửi đơn vị sử dụng ngân sách; sau đó thực hiện xử lý thu '
             'hôi giảm trừ giá trị thanh toán vào lần thanh toán liền kề tiếp theo, trường hợp lần '
             'thanh toán liền kề tiệp theo không đủ khối lượng hoàn thành/dự toán để giảm trừ thì '
             'thực hiện theo quy định tại khoản 2, Điều 4 Thông tư này: ¬ si An ¬— - ¬ — 2. Kiểm '
             'soát trước, thanh toán sau:, Kiểm soát trước, thanh toán sau là hình thức thanh toán '
             'áp dụng đối với tất cả các khoản chỉ, trong đó, Kho bạc Nhà nước thực hiện kiểm '
             'soát, thanh toán theo đúng thời gian quy định tại Nghị định số 11/2020/NĐ-CP sau khi '
             'nhận đây _ đủ hỗ sơ hợp pháp, hợp lệ theo quy định. bà |'},
 {'id': 109,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 4',
               'article_title': 'Thu hồi giảm chỉ hoặc thu hồi nộp NSNN |',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 4. Thu hồi giảm chỉ hoặc thu '
                                  'hồi nộp NSNN |',
               'topics': 'Quy định chung',
               'is_core_fintech': False,
               'page': 5,
               'chunk_part': '1/1'},
  'content': 'Điều 4. Thu hồi giảm chỉ hoặc thu hồi nộp NSNN |\n'
             '1. Đơn vị sử dụng ngân sách gửi Kho bạc Nhà nước Giấy nộp trả kinh phí (theo mẫu '
             'C2-05a/NS) ban hành kèm theo Nghị định sô 11/2020/NĐ-CP để nộp trả kinh phí theo '
             'đúng quy định. Ị Trường hợp khoản chỉ sai chế độ, tiêu chuẩn, định mức chỉ của nhà '
             'nước theo kết luận, kiến nghị của các cơ quan nhà nước có thấm quyên hoặc đơn vị sử '
             'dụng ngân sách hoặc Kho bạc Nhà nước phát hiện sau khi kiểm soát, đơn vị sử dụng '
             'ngân sách chịu trách nhiệm thực hiện nộp trả NSNN. :\n'
             '2. Kho bạc Nhà nước thực hiện: Thu hồi giảm chỉ NSNN (trường hợp chưa quyết toán '
             'ngân sách) hoặc thu hôi nộp NSNN (trường hợp đã quyết toán ngân sách); thực hiện '
             'hạch toán kế toán theo đúng quy định. \n'
             '\n'
             'Chương H QUY ĐỊNH CỤ THẺ'},
 {'id': 110,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 5',
               'article_title': 'Nội dung kiểm soát chỉ qua Kho bạc Nhà nước',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua '
                                  'Kho bạc Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '1/21'},
  'content': 'Điều 5. Nội dung kiểm soát chỉ qua Kho bạc Nhà nước\n'
             '\n'
             '1. Kho bạc Nhà nước kiểm soát, thanh toán theo nguyên tắc quy định tại\n'
             'Điều 2 Thông tư này và kiểm soát các nội dung cụ thể như sau:\n'
             '\n'
             '1.1. Trường hợp giao dịch trực tiếp qua Kho bạc Nhà nước, dấu và chữ ký trên chứng '
             'từ chuyên tiền khớp đúng với mẫu dấu và mẫu chữ ký đăng ký giao dịch tại Kho bạc Nhà '
             'nước; Trường hợp thực hiện qua Trang thông tin dịch vụ . công của Kho bạc Nhà nước, '
             "việc ký số trên các chứng từ chuyển tiền ' phải đúng họ tên, chức danh các thành "
             'viên theo quyết định của cấp có thẩm quyền, đã thực hiện đăng ký với Kho bạc Nhà '
             'nước. \n'
             'Nội dung và số tiền, hạch toán kế toán tại chứng từ.chuyển tiền phủ hợp với các hồ '
             'sơ có liên quan kèm theo.\n'
             '\n'
             '1.2. Nội dung chi phải phù hợp với mã nội dung kinh tế theo quy định của\n'
             'Mục lục NSNN hiện hành (không bao gồm các khoản chỉ từ Tài khoản tiền gửi).\n'
             '\n'
             '1.3. Mức tạm ứng đảm bảo theo đúng quy định tại Điều 6 Thông tư này. Nội dung đề '
             'nghị thanh toán tạm ứng phải phù hợp với nội dung đề nghị tạm ứng. Trường hợp đơn vị '
             'chưa xác định được chính xác nội dung chị, thực hiện tạm ứng vào tiểu mục khác của '
             'mục tương ứng với khoản chỉ, Kho bạc Nhà nước thanh toán tạm ứng trong phạm vi các '
             'tiêu mục trong cùng nhóm mục tương ứng với nội dung đã tạm ứng.'},
 {'id': 111,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 5',
               'article_title': 'Nội dung kiểm soát chỉ qua Kho bạc Nhà nước',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua '
                                  'Kho bạc Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '2/21'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua Kho bạc Nhà nước '
             '(tiếp theo)]\n'
             '1.3. Mức tạm ứng đảm bảo theo đúng quy định tại Điều 6 Thông tư này. Nội dung đề '
             'nghị thanh toán tạm ứng phải phù hợp với nội dung đề nghị tạm ứng. Trường hợp đơn vị '
             'chưa xác định được chính xác nội dung chị, thực hiện tạm ứng vào tiểu mục khác của '
             'mục tương ứng với khoản chỉ, Kho bạc Nhà nước thanh toán tạm ứng trong phạm vi các '
             'tiêu mục trong cùng nhóm mục tương ứng với nội dung đã tạm ứng.\n'
             '\n'
             '1.4. Đối với các khoản chỉ phải gửi Hợp đồng đến Kho bạc Nhà nước (khoản chỉ có giá '
             'trị hợp đồng trên 50 triệu đồng), Kho bạc Nhà nước căn cứ hồ sơ đề nghị thanh toán '
             'của đơn vị sử dụng ngân sách, các điều khoản thanh toán, tạm ứng được quy định trong '
             'hợp đồng và tổng giá trị Hợp đồng để tạm ứng, thanh toán cho đối tượng thụ hưởng. \n'
             'Trường hợp hợp đồng có quy định phải bảo lãnh tạm ứng, Kho bạc Nhà nước kiểm soát '
             'đảm bảo thời gian hiệu lực của bảo lãnh tạm ứng hợp đồng phải được kéo dài cho đến '
             'khi đơn vị sử dụng ngân sách thu hồi hết số tiền tạm ứng, khi bảo lãnh tạm ứng hết '
             'thời hạn nhưng đơn vị sử dụng ngân sách chưa thanh toán hết số tiền tạm ứng, Kho bạc '
             'Nhà nước có trách nhiệm đôn đốc đơn VỊ SỬ dụng ngân sách phối hợp với nhà thầu làm '
             'thủ tục gia hạn bảo lãnh tạm ứng hoặc thanh toán tạm ứng theo quy định. \n'
             '.__ 1.5. Kiểm soát đối với Bảng xác định giá trị khối :lượng công việc hoàn thành '
             'theo Mẫu số 08a ban hành kèm theo Phụ lục II Nghị định số 11/2020/NĐ- CP:\n'
             '\n'
             'a) Đối với khoản chỉ mua sắm hàng hóa: \n'
             '*x xe \n'
             ' | \n'
             'Kho bạc Nhà nước kiểm soát nội dung công việc, đơn vị tính, số lượng, đơn giá đảm '
             'bảo theo đúng quy định của hợp đồng, không vượt giá trị hợp đồng. Riếng đối với chỉ '
             'mua thuốc, dịch truyền, hóa chất, vật tư y/ tế sử dụng trong quá trình điều trị khám '
             'chữa bệnh thực hiện theo quy định tại tiết b điểm 2.2 khoản 2 Điều này. |\n'
             '\n'
             'b) Đối với khoản chỉ địch vụ: Ị \n'
             'Đơn vị-sử dụng ngân sách căn cứ nội dung Hợp đồng, Biên bản: nghiệm thu để lập Bảng '
             'xác định giá trị khối lượng công việc hoàn thành gửi Kho bạc Nhà nước cho phủ hợp, '
             'đảm bảo thể hiện đúng nội dung công việc và giá trị thanh toán theo hợp đồng. để Kho '
             'bạc Nhà nước có cơ sở kiểm soát, thanh toán. Kho bạc Nhà nước kiểm soát nội dung '
             'công việc và giá trị thanh toán (đơn vị tính, số lượng, đơn giá nếu có) đảm bảo theo '
             'đúng quy định của hợp đồng, không vượt giá trị hợp đồng.'},
 {'id': 112,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 5',
               'article_title': 'Nội dung kiểm soát chỉ qua Kho bạc Nhà nước',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua '
                                  'Kho bạc Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '3/21'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua Kho bạc Nhà nước '
             '(tiếp theo)]\n'
             '1.6. Kiểm soát đối với Bảng kê nội dung thanh toán/tạm ứng theo Mẫu số 07 ban hành '
             'kẻm theo Phụ lục II Nghị định sô 11/2020/NĐ-CP: \n'
             'Đơn vị kê khai theo Mẫu số 07 ban hành kèm theo Phụ lục II Nghị định số ¬ \n'
             '_ 12020/NĐ-CP, trong đó lưu ý: Chứng từ chỉ tại đơn vị là các chứng từ được quy định '
             'theo hệ thống chứng từ tại Thông tư số 107/2017/TT-BTC ngày 10 tháng 10 năm 2017 của '
             'Bộ Tài chính hướng dẫn chế độ kế toán hành chính sự nghiệp. Kho bạc Nhà nước kiểm '
             'soát Bảng kê nội dưng thanh toán/tạm ứng đảm bảo mã nội \n'
             '-đung kinh tế khớp. -đúng với. Giấy rút dự toán, nội dung, thanh toán; tổng số tiền '
             'phù hợp với Giấy rút dự toán/Ủy nhiệm chỉ và không vượt định mức quy-định tại các '
             'văn bản quy phạm pháp luật hiện hành, Quy chế chỉ tiêu nội bộ của đơn vì.\n'
             '\n'
             '1.7. Đối với các khoản chỉ có yêu cầu bảo mật: i \n'
             'Đơn vị sử dụng ngân sách chịu trách nhiệm trong việc xác định nội dung chỉ thuộc yêu '
             'cầu bảo mật và việc kiểm soát nội dung thanh toán, hồ sơ thanh toán theo đúng quy '
             'định của pháp luật. Trên Giấy rút dự toán/Ủy nhiệm chỉ đơn vị sử dụng ngân sách ghi '
             'nội dung thanh toán và nội dung sau: "Khoản chỉ có yêu cầu bảo mật". Kho bạc Nhà '
             'nước kiểm soát tính hợp pháp, hợp lệ của Giấy rút dự toán/Ủy nhiệm chị để thanh toán '
             'theo đề nghị của đơn vị.'},
 {'id': 113,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 5',
               'article_title': 'Nội dung kiểm soát chỉ qua Kho bạc Nhà nước',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua '
                                  'Kho bạc Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '4/21'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua Kho bạc Nhà nước '
             '(tiếp theo)]\n'
             '1.8. Trường hợp phải thực hiện mua sắm theo phương thức tập trung: Kho bạc Nhà nước. '
             'kiểm soát đảm bảo: Có trong danh mục mua sắm tập trung được cấp có thâm quyền ban '
             'hành (theo quy định tại khoản 2 Điều 53 Luật Đấu thầu số 22/2023/QH15), đảm bảo hợp '
             'đồng phù hợp với Thỏa thuận khung (thỏa thuận khung được đơn vị mua sắm tập trung '
             'đăng tải theo quy định tại điểm a khoản 3\n'
             'Điều 77 Nghị định 15 1/2017/NĐ- CP ngày 26 tháng 12 năm 2017 của Chính phủ quy định '
             'chỉ tiết một số điều của Luật Quản lý, sử dụng tài sản công). \n'
             '` kứ Ị \n'
             '\n'
             '2. Ngoài các nội dung kiểm soát nêu tại khoản 1 Điều này, đối với một số nội dung '
             'chỉ cụ thể Kho bạc Nhà nước kiêm soát như sau:'},
 {'id': 114,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 5',
               'article_title': 'Nội dung kiểm soát chỉ qua Kho bạc Nhà nước',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua '
                                  'Kho bạc Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '5/21'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua Kho bạc Nhà nước '
             '(tiếp theo)]\n'
             '2.1. Đối với các khoản chỉ lương và phụ cấp theo lương (các khoản phụ cấp theo '
             'lương, theo hệ thống mục lục ngân sách hiện hành); tiền công lao động theo hợp đồng: '
             'tiền thu nhập tăng thêm, chỉ hỗ trợ, trợ cấp, phụ cấp khác, khoán, khen thưởng: \n'
             "' a) Kho bạc nhà nước kiểm soát đảm bảo không vượt chỉ tiêu biên chế được cấp có "
             'thẩm quyền phê duyệt; không vượt số lượng lao động hợp đồng, theo đúng - nguồn ˆ '
             'kinh phí thực hiện theo quy định tại Điều 7, Điều 8, Điều 9, Điều 12 Nghị định số 11 '
             '1/2022/NĐ-CP ngày 30 tháng 12 năm 2022 của Chính phủ về hợp đồng đối với một số loại '
             'công việc trong cơ quan hành chính và đơn vị sự nghiệp công. lập; không vượt số '
             'lượng tại Văn bản phê duyệt số lượng người hoạt động. không chuyên trách ở cấp xã '
             'của hội đồng nhân dân cấp tỉnh và ủy ban nhân dân câp huyện theo quy định tại Điều '
             '33 Nghị định số 33/2023/NĐ-CP ngày 10 tháng 6 năm 2023 của Chính phủ quy định về cán '
             'bộ, công chức cấp xã và người hoạt động không chuyên trách ở cấp xã, ở thôn, tô dân '
             'phố. Trong trường hợp đầu năm chưa có văn bản phê duyệt chỉ tiêu biên chế của cấp có '
             'thâm quyền thì Kho bạc Nhà nước thực hiện việc kiểm soát căn cứ vào văn bản giao chỉ '
             'tiêu biên chế của cấp có thắm quyền năm trước liền kể và văn bản đề nghị, cam kết '
             'của đơn vị sử dụng ngân sách. Trường hợp có sự thay đỗi so với tháng trước về chỉ '
             'tiêu, biên chế, đơn vị ghi rõ tại phần thuyết minh — Bảng thanh toán cho đối tượng '
             'thụ hưởng theo Mẫu số 09 Phụ lục II ban hành kèm theo Nghị định số 11/2020/NĐ-CP.\n'
             '\n'
             'b) Kiểm tra, đảm bảo khớp. đúng giữa chỉ tiết và tổng số; khớp đúng tông số tiền '
             'trên Bảng thanh toán cho đối tượng thụ hưởng với Giấy rút dự toánỦy nhiệm chỉ đã '
             'được thủ trưởng đơn vị ký duyệt.\n'
             '\n'
             'c) Đối với khoản chỉ thu nhập tăng thêm: \n'
             '- Đối với đơn vị sự nghiệp công lập do nhà nước đâm bảo chỉ thường xuyên (đơn vị '
             'nhóm 4): Kho bạc Nhà nước kiểm soát đảm bảo theo đúng Quy chế chị tiêu nội bộ của '
             'đơn vị và theo quy định tại khoản 2 Điều 22 Nghị định số 60/2021/NĐ-CP và khoản 3 '
             'Điều 10 Thông tư số 26/2022/TT- BTC ngày 16 tháng 9 năm 2022 của Bộ Tài chính hướng '
             'dẫn một số nội dung về cơ chế tự chủ tài chính của đơn vị sự nghiệp công lập; xử lý '
             'tài sản, tài chính khi tô chức lại, giải thể đơn vị sự nghiệp công lập (Thông tư sô '
             '56/2022/TT-BTC). \n'
             '- Đối với cơ quan nhà nước thực hiện chế độ tự chủ: Kho bạc Nhà nước kiểm soát đảm '
             'bảo theo đúng Quy chế chỉ tiêu nội bộ của đơn vị và theo quy định tại Thông tư liên '
             'tịch số 71/2014/TTLT-BTC-BNV ngày 30 tháng 5 năm 2014 của Liên Bộ Tải chính và Bộ '
             'Nội vụ quy định chế độ tự chủ, tự chịu trách nhiệm về sử dụng kinh phí quản lý hành '
             'chính đối với cơ quan nhà nước (Thông tư liên tịch số 71⁄2014/TTLT- BTC-BNV). \n'
             ' | \n'
             '|'},
 {'id': 115,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 5',
               'article_title': 'Nội dung kiểm soát chỉ qua Kho bạc Nhà nước',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua '
                                  'Kho bạc Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '6/21'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua Kho bạc Nhà nước '
             '(tiếp theo)]\n'
             '- Đối với cơ quan nhà nước thực hiện chế độ tự chủ: Kho bạc Nhà nước kiểm soát đảm '
             'bảo theo đúng Quy chế chỉ tiêu nội bộ của đơn vị và theo quy định tại Thông tư liên '
             'tịch số 71/2014/TTLT-BTC-BNV ngày 30 tháng 5 năm 2014 của Liên Bộ Tải chính và Bộ '
             'Nội vụ quy định chế độ tự chủ, tự chịu trách nhiệm về sử dụng kinh phí quản lý hành '
             'chính đối với cơ quan nhà nước (Thông tư liên tịch số 71⁄2014/TTLT- BTC-BNV). \n'
             ' | \n'
             '| \n'
             '- Trường hợp chỉ hỗ trợ, trợ cấp, phụ cấp khác (không bao gồm các khóản phụ cấp theo '
             'lương theo hệ thống mục lục ngân sách hiện hành), khoán, khen thưởng được quy định '
             'tại văn bản quy phạm pháp luật, quy chế chỉ tiêu nội bộ của đơn vị:\n'
             '\n'
             'd) Đối với chỉ hỗ trợ, trợ cấp, phụ cấp khác, khoán, khen thưởng: \n'
             'Kho bạc Nhà nước kiểm tra Chứng từ chuyên tiền; Bảng kê nội dung thành toán/tạm ứng '
             '(trường hợp chứng từ chuyển tiền.của đơn vị sử dụng NSNN không \n'
             'thế hiện được hết nội dung chỉ, số lượng, định mức, đơn giá thực tế đối với những \n'
             'khoản chỉ có quy định định mức tại Quy chế chí tiêu nội bộ, văn bản quy phạm pháp '
             'luật) và Bảng thanh toán cho đối tượng thụ hưởng đảm bảo không vượt định mức đã được '
             'quy định tại các văn bản quy phạm pháp luật, quy chế chỉ tiêu nội bộ của đơn vị. \n'
             'Riêng đối với chỉ trợ cấp, phụ cấp cho người có công với cách mạng lvà người trực '
             'tiếp tham gia kháng chiến do ngành Lao động - Thương bình vài Xã hội quản lý và trợ '
             'giúp xã hội đối với đối tượng bảo trợ xã hội thực hiện theo quy định tại tiết a điểm '
             '2.3 khoản 2 Điều u này. \n'
             'thẩm quyền: \n'
             'Kho bạc Nhà nước kiểm soát đảm bảo không vượt Dự toán được cấp thâm quyền giao; kiểm '
             'soát Chứng từ chuyển tiền; Bảng kê nội dung thanh toán/tạm \n'
             'đứng: (trường ượp chứng từ chuyển tiền cửa đơn vị sử dụng NSNN Không thê hiện \n'
             'được hết nội dung chỉ, số lượng, định mức, đơn giá thực | tế đối với những khoản chỉ '
             'có quy định định mức tại Quyết định của cấp có thầm quyển) và Bảng thanh toán cho '
             'đối tượng thụ hưởng đảm bảo không vượt định mức quy định tại quyết định của cấp có '
             'thắm quyền.'},
 {'id': 116,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 5',
               'article_title': 'Nội dung kiểm soát chỉ qua Kho bạc Nhà nước',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua '
                                  'Kho bạc Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '7/21'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua Kho bạc Nhà nước '
             '(tiếp theo)]\n'
             '2.2. Đối với chỉ mua sắm:\n'
             '\n'
             'a) Đối với chỉ mua sắm tài sản công là xe ô tô và phương tiện vận tải khác; máy móc, '
             'thiết bị: \n'
             '- Trường hợp không thực hiện mua sắm theo phương thức tập trung: \n'
             '+ Kho bạc Nhà nước kiểm soát đảm bảo không vượt đơn giá tối đa theo tiêu chuẩn, định '
             'mức quy định tại Nghị định số 72/2023/NĐ-CP ngày 26 tháng 09 năm 2023 của Chính phủ '
             'quy định tiêu chuẩn, định mức sử dụng xe ô tô (Nghị định số 72/2023/NĐ-CP); Nghị '
             'định số 85/2018/NĐ-CP ngày 30 tháng 5 năm 2018 của Chính phủ quy định tiệu chuẩn, '
             'định mức sử dụng xe ô tô tại đơn vị lực lượng vũ trang nhân dân (Nghị định số '
             '85/2018/NĐ-CP); Quyết định số \n'
             "50/2017/QĐ- -T'Tg ngày 3l tháng 12 năm 2017 của Thủ tướng Chính phủ quy 'định \n"
             'Ỳ \n'
             'tiêu chuẩn, định mức sử dụng máy móc, thiết bị (Quyết định số 50/2017/QĐ- TT); và '
             'các văn bản quy phạm pháp luật hướng dẫn có liên quan. #Ị \n'
             ' + Đối với tải sản chuyên dùng: Kho bạc Nhà nước kiểm soát đảm bảo có trong danh mục '
             'máy móc, thiết bị chuyên dùng (chủng loại) tại Văn bản quy định tiêu chuẩn, định mức '
             'sử dụng máy móc, thiết bị chuyên dùng của cấp có thấm quyền ban hành. . \n'
             '- Trường hợp phải thực hiện mua sắm theo phương thức tập trung: \n'
             'Kho bạc Nhà nước căn cứ hồ sơ để nghị thanh toán của đơn vị sử dụng ngân sách, các '
             'điều khoản được quy định trong hợp đồng, Biên bản nghiệm thu, bàn giao, tiếp nhận '
             'tài sản (theo Mẫu số 06/TSC-MSTT kèm theo Nghị định số 151/2017/NĐ-CP ngày 26 tháng '
             '12 năm 2017 của Chính phủ quy định chỉ tiết một số điều của Luật Quản lý, sử đụng '
             'tài sản công) để kiểm soát đảm bảo: Tên \n'
             '-tài sản, đơn vị tính, số lượng, giá mua theo đúng quy định của hợp đồng, không \n'
             'vượt giá trị hợp đồng và đơn giá tối đa theo tiêu chuẩn, định mức quy định tại Nghị '
             'định số 72/2023/NĐ-CP; Nghị định số 85/2018/NĐ-CP; Quyết định số 50/2017/QĐ-TTg; và '
             'các văn bản quy phạm pháp luật hướng dẫn có liên quan. \n'
             '+ Đối với tài sản chuyên dùng: Kho bạc Nhà nước kiểm soát đảm bảo có trong danh mục '
             'máy móc, thiết bị chuyên: dùng (chủng loại) tại Văn bản quy định tiêu chuẩn, định '
             'mức sử dụng máy móc, thiết bị chuyên dùng của cấp có thâm quyền ban hành.\n'
             '\n'
             'b) Đối với chỉ mua thuốc, địch truyền, hóa chất, vật tư y tế sử dụng trong quá trình '
             'điều trị khám chữa bệnh (không bao gồm mua sắm máy móc, thiết bị; xe ô tô và phương '
             'tiện vận tải khác được quy định tại tiết a điểm 2.2 khoản 2 Điều \n'
             'này):'},
 {'id': 117,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 5',
               'article_title': 'Nội dung kiểm soát chỉ qua Kho bạc Nhà nước',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua '
                                  'Kho bạc Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '8/21'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua Kho bạc Nhà nước '
             '(tiếp theo)]\n'
             'b) Đối với chỉ mua thuốc, địch truyền, hóa chất, vật tư y tế sử dụng trong quá trình '
             'điều trị khám chữa bệnh (không bao gồm mua sắm máy móc, thiết bị; xe ô tô và phương '
             'tiện vận tải khác được quy định tại tiết a điểm 2.2 khoản 2 Điều \n'
             'này): \n'
             'Kho bạc Nhà nước căn cứ hồ sơ đền nghị thanh toán của đơn vị sử dụng ngân sách, các '
             'điều khoản được quy định trong hợp đồng, thực hiện kiểm soát tông số tiền của các '
             'hóa đơn kê khai trên Bảng xác định giá trị khối lượng công việc hoàn thành (theo Mẫu '
             'số 08a ban hành kèm theo Phụ lục II Nghị định số 11/2020/NĐ- CP) đảm bảo không vượt '
             'giá trị hợp đồng. Đơn vị sử dụng ngân sách căn cứ Biên bản nghiệm thu để lập Bảng '
             'xác định giá trị khối lượng công việc hoàn thành, trong đó tại cột nội dung công '
             'việc (cột số 2) kê khai số hóa đơn và ngày, tháng, năm của hóa đơn, tại cột thành '
             'tiền (cột số 6) kê khai số tiền tương ứng với giá trị của từng hóa đơn. Đơn vị sử '
             'dụng ngân sách chịu trách nhiệm vẻ số lượng, chất lượng và đơn giá, chủng loại '
             'thuốc, dịch truyền, hóa chất, vật tư y tế sử dụng trong quá trình điều trị khám chữa '
             'bệnh đề nghị thanh toán, đảm bảo theo đúng quy định tại Hợp đồng: Thỏa thuận khung '
             '(trường hợp phải thực hiện mua sắm theo phương thức tập trung); Biên bản nghiệm thu '
             'giữa đơn vị sử dụng ngân sách và nhà cung cấp hàng hóa dịch vụ.'},
 {'id': 118,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 5',
               'article_title': 'Nội dung kiểm soát chỉ qua Kho bạc Nhà nước',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua '
                                  'Kho bạc Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '9/21'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua Kho bạc Nhà nước '
             '(tiếp theo)]\n'
             '2.3. Đối với kinh phí chỉ thường xuyên thực hiện chính sách, chế độ ưu đãi người có '
             'công với cách mạng, thân nhân của người có công với cách mạng và \n'
             "& ' « \n"
             ' người trực tiếp tham gia kháng chiến do ngành Lao động - Thương binh và Xã hội quản '
             'lý và trợ giúp xã hội đối với đối tượng bảo trợ xã hội:\n'
             '\n'
             'a) Đối với chỉ trợ cấp, phụ cấp cho người có công với cách mạng và người trực tiếp '
             'tham gia kháng chiến do ngành Lao động - Thương binh và Xã hội quản lý và trợ giúp '
             'xã hội đối với đối tượng bảo trợ xã hội: \n'
             '-~ Trường hợp đơn vị sử dụng ngân sách-trực tiếp thanh toán cho đối tượng '
             'thụuhưởngg: Sắc ¬ ` \n'
             '+ Trường hợp chỉ trả cho đối tượng thụ hưởng bằng tiền mặt: \n'
             'Hằng tháng, đơn vị sử dụng ngân sách lập Giấy rút dự toán (tạm ứng) [gửi Kho bạc Nhà '
             'nước, Kho bạc Nhà nước thực hiện tạm ứng trên cơ sở đề nghị của đơn vị. \n'
             'Sau khi chỉ trả cho đối tượng thụ hưởng, đơn vị sử dụng ngân sách lập Giấy : \n'
             'đề-nghị thanh toán tạm-ứng, -Bảng kê nội-dung thanh toán/tạm ứng đề đề nghị. \n'
             'thanh toán tạm ứng với Kho bạc Nhà nước. Kho bạc Nhà nước căn cứ Giấy đề - nghị '
             'thanh toán tạm ứng và Bảng kê nội dung thanh toán/tạm ứng do đơn vị gửi, \n'
             '_ thực hiện kiểm soát, thanh toán theo quy định tại khoản 1_Điều này và theo đúng. \n'
             'định mức quy định tại Nghị định số 20/2021/NĐ-CP ngày 15 tháng 3 năm 2021 của Chính '
             'phủ quy định chính sách trợ giúp xã hội đối với đối tượng bảo trợ xã \n'
             'hội (Nghị định số 20/2021/NĐ-CP) và Thông tư số 76/2021/TT-BTC ngày 15 tháng 9 năm '
             '2021 của Bộ Tài chính hướng dẫn khoản 1 và khoản 2 Điều 31 Ñghị định số '
             '20/2021/NĐ-CP-(Thông tư số 76/2021/TT-BTC); Thông tư số 44/2022/TT-BTC ngày 21 tháng '
             '7 năm 2022 của Bộ Tài chính quy định quản lý và sử dụng kinh phí chỉ thường xuyên '
             'thực hiện chính sách, chế độ ưu đãi người có công với cách mạng, thân nhân của người '
             'có công với cách mạng và người trực tiếp tham gia kháng chiến do ngành Lao động - '
             'Thương binh và Xã hội quản lý (Thông tư số 44/2022/TT-BTC). - : \n'
             'Đơn vị sử dụng ngân sách chịu trách nhiệm quản lý nguồn kinh phí và tổ chức thực '
             'hiện chỉ trả kịp thời, đúng chế độ, đúng đối tượng theo quy định. \n'
             '+ Trường hợp chỉ trả cho đối tượng thụ hưởng theo hình thức chuyển khoản: \n'
             'Kho bạc Nhà nước thực hiện kiểm soát Chứng từ chuyển tiền; Bảng kê nội dung thanh '
             'toán/tạm ứng (trường hợp chứng từ chuyển tiền của đơn Vị sử dụng NSNN không thể hiện '
             'được hết nội dung chỉ, số lượng, định mức, đơn giá thực tế đối với những khoản chỉ '
             'có quy định định mức tại các văn bản quy phạm pháp luật) và Bảng thanh toán cho đối '
             'tượng thụ hưởng theo quy định tại khoản 1 Điều này và theo đúng định mức quy định '
             'tại Nghị định số 20/2021/NĐ-CP, Thông tư sô 76/2021/TT-BTC và Thông tư số '
             '44/2022/TT-BTC. \n'
             '- Trường hợp thông qua tổ chức địch vụ chỉ trả. \n'
             '\n'
             ' H'},
 {'id': 119,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 5',
               'article_title': 'Nội dung kiểm soát chỉ qua Kho bạc Nhà nước',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua '
                                  'Kho bạc Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '10/21'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua Kho bạc Nhà nước '
             '(tiếp theo)]\n'
             '- Trường hợp thông qua tổ chức địch vụ chỉ trả. \n'
             '\n'
             ' H \n'
             '+ Hằng tháng, đơn vị sử dụng ngân sách lập Giấy rút dự toán (tạm ứng); Bảng kê chứng '
             'từ thanh toán/tạm ứng; Hợp đồng giữa đơn vị sử dụng ngân sách và tổ chức dịch vụ '
             'thanh toán gửi Kho bạc Nhà nước (Hợp đồng gửi lần đầu và gửi khi có bỗ sung, điều '
             'chỉnh), Kho bạc Nhà nước thực hiện kiểm soát theo quy định tại khoản 1 Điều này và '
             'theo đúng định mức quy định tại Nghị định số 20/2021/NĐ-CP, Thông tư số '
             '76/2021/TT-BTC và Thông tư số 44/2022/TT- BTC; thực hiện tạm ứng từ tài khoản dự '
             'toán của đơn vị sử dụng ngân sách sang tài khoản của tô chức địch vụ chỉ trả theo '
             'đúng quy định tại hợp đồng dịch vụ giữa hai bên. \n'
             '+ Trường hợp Tổ chức dịch vụ chỉ trả mở tài khoản tiền gửi tại Kho bạc Nhà nước, khi '
             'rút kinh phí từ tài khoản tiền gửi mở tại Kho bạc Nhà nước để thanh toán cho đối '
             'tượng thụ hưởng: Kho bạc Nhà nước kiểm soát tính hợp pháp, hợp lệ của Ủy nhiệm chỉ, '
             'trong phạm vi số dư tài khoản tiền gửi. \n'
             '+ Hằng tháng, đơn vị sử dụng ngân sách có trách nhiệm quyết toán với tổ chức dịch vụ '
             'chỉ trả về số tiền đã tạm ứng tại Kho bạc Nhà nước, đồng thời gửi Kho bạc Nhà nước '
             'Giấy đề nghị thanh toán tạm ứng và Bảng kê nội dung thanh toán/tạm ứng, Bảng kê kinh '
             'phí đã chỉ trả cho đối tượng theo Mẫu số 10 Phụ lục II kèm theo Nghị định số '
             '11/2020/NĐ-CP (trường hợp áp dụng đối với đối tượng bảo trợ xã hội thì đơn vị sử '
             'dụng ngân sách sửa tên biểu thành: Bảng kê kính phí đã chỉ trả cho đối tượng bảo trợ '
             'xã hội) để Kho bạc Nhà nước kiểm soát, thanh toán tạm ứng theo đúng định mức quy '
             'định tại Thông tư số 44/2022/TT-BTC, Thông tư số 76/2021/TT-BTC. \n'
             '+ Tổ chức dịch vụ chỉ trả có trách nhiệm quản lý kinh phí và tổ chức thực hiện chỉ '
             'trả đủ số tiền trợ cấp, phụ cấp đến đối tượng thụ hưởng theo đúng thời gian quy '
             'định, chịu trách nhiệm bồi hoàn số tiền mất, thất thoát.\n'
             '\n'
             'b) Đối với chỉ hỗ trợ cho các dự án cải tạo, sửa chữa, nâng cấp, bảo trì các công '
             'trình ghi công liệt sĩ; cơ sở vật chất cơ sở nuôi dưỡng, điều dưỡng, đón tiếp người '
             'có công với cách mạng theo quy định tại Thông tư số 44/2022/TT-BTC: \n'
             'Kho bạc Nhà nước căn cứ quyết định phân bỗ (hỗ trợ vốn) của cấp có thẩm quyền theo '
             'phân cấp của địa phương, Giấy rút dự toán của cơ quan Lao động - - Thương bình và Xã '
             'hội (hoặc cơ quan được Sở Lao động - Thương binh và Xã hội giao dự toán) thực hiện '
             'chuyển tiền sang tài khoản tiền gửi của chủ đầu tư mở tại Kho bạc Nhà nước. Khi chỉ '
             'từ tài khoản tiền gửi, Kho bạc Nhà nước kiểm soát theo quy định. -'},
 {'id': 120,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 5',
               'article_title': 'Nội dung kiểm soát chỉ qua Kho bạc Nhà nước',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua '
                                  'Kho bạc Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '11/21'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua Kho bạc Nhà nước '
             '(tiếp theo)]\n'
             '2.4. Đối với kinh phí giao nhiệm vụ, đặt hàng, đấu thầu cung cấp sản phẩm, dịch vụ '
             'công sử dụng NSNN từ nguồn kinh phí chỉ thường xuyên:\n'
             '\n'
             'a) Đối với kinh phí giao nhiệm vụ: \n'
             ' | “ \n'
             'Ị Kho bạc Nhà nước căn cứ Dự toán được cấp có thấm quyền giao, Quyết \n'
             'định của cơ quan quản lý cấp trên giao nhiệm vụ cung cấp dịch vụ sự nghiệp công sử '
             'dụng kinh phí NSNN cho đơn vị sự nghiệp công lập, Văn bản nghiệm thu nhiệm vụ được '
             'giao giữa cơ quan quản lý câp trên và đơn vị sự nghiệp công lập \n'
             'để kiểm soát, trong đó lưu ý ý: | \n'
             '- Nội dung được giao nhiệm vụ thuộc danh mục dịch vụ sự nghiệp công sử dụng kinh phí '
             'NSNN giao nhiệm vụ đặt hàng hoặc đấu thầu (heo Biểu 01 của : Phụ lục Ï ban hành kèm '
             'theo Nghị định số 32/2019/NB- -CP) hoặc Quyết định của cấp có thâm quyền về danh mục '
             'dịch vụ sự nghiệp công sử dụng kinh phí NSNN thực hiện theo phương thức giao nhiệm '
             'vụ. \n'
             '- Kinh phí giao nhiệm vụ kiểm soát như đối với nguồn kinh phí chi nhiệm vụ thường '
             'xuyên không giao tự chủ theo quy định tại tiết a điểm 5.1 khoản 5 Điều này và các '
             'quy định về tạm ứng, thanh toán; nghiệm thu nhiệm vụ được giao tại Quyết định của cơ '
             'quan quản lý cấp trên giao nhiệm vụ cung cấp dịch vụ sự nghiệp công sử dụng kinh phí '
             'NSNN cho đơn vị sự nghiệp công lập.\n'
             '\n'
             'b) Đối với kinh phí đặt hàng cho đơn vị sự nghiệp công lập trực thuộc: | \n'
             '- Kho bạc Nhà nước thực hiện kiểm soát đám bảo: Sản phẩm, dịch vụ được ˆ đặt hàng '
             'thuộc danh mục dịch vụ sự nghiệp công sử dụng kinh phí NSNN giao nhiệm vụ, đặt hàng '
             'hoặc đấu thầu và danh mục sản phẩm, dịch vụ công ích thực hiện đặt hàng hoặc đấu '
             'thầu (theo Phụ lục số I và Phụ lục số II ban hành kèm theo Nghị định số '
             '32/2019/NĐ-CP) hoặc Quyết định của cấp. có thẩm quyền về danh mục dịch vụ sự nghiệp '
             'công sử dụng NSNN thực hiện theo phương thức đặt hàng; có đơn giá, giá dịch vụ sự '
             'nghiệp công do cơ quan có thấm quyền ban hành làm cơ sở đề đặt hàng. \n'
             '- Căn cứ phương thức thanh toán quy định tại Quyết định của cơ quan quản . lý cấp '
             'trên đặt hàng đơn vị sự nghiệp công lập trực thuộc cung cấp dịch vụ sự nghiệp công '
             'sử dụng kinh phí NSNN theo Mẫu số 01 ban hành kèm theo Nghị định số 32/2019/NĐ-CP '
             '(Quyết định đặt hàng); Kho bạc Nhà nước thực hiện tạm ứng vào tài khoản tiền gửi của '
             'đơn vị sự nghiệp công lập mở tại Kho bạc Nhà nước theo đúng mức tạm ứng quy định tại '
             'Quyết định đặt hàng. Khi chỉ từ tài khoản tiền gửi của đơn vị sự nghiệp công lập, '
             'Kho bạc Nhà nước thực hiện kiểm soát theo đúng quy chế chỉ tiêu nội bộ và phù hợp '
             'với: Quyết định giao quyền tự chủ của cập có thẩm quyền; định mức (mức chỉ) theo quy '
             'định tại các văn bản quy phạm pháp luật. Ị'},
 {'id': 121,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 5',
               'article_title': 'Nội dung kiểm soát chỉ qua Kho bạc Nhà nước',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua '
                                  'Kho bạc Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '12/21'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua Kho bạc Nhà nước '
             '(tiếp theo)]\n'
             'Khi thanh toán tạm ứng, Kho bạc Nhà nước căn cứ quy định về thanh toán, nghiệm '
             'thu‹tại Quyết định đặt hàng và biên bản nghiệm thu đặt hàng giữa cơ quan cấp trên và '
             'đơn vị sự nghiệp công lập (theo Mẫu số 02 ban hành kèm theo Nghị định số '
             '32/2019/NĐ-CP) để kiểm soát theo đúng đơn giá, giá đặt hàng quy định tại các văn bản '
             'quy phạm pháp luật. Trường hợp đơn vị đã tạm ứng vượt số lượng, \n'
             ' khối lượng sản phẩm, dịch vụ công đặt hàng được nghiệm thu, đơn vị phải thu hồi và '
             'nộp lại số tiền đã tạm ứng vượt so với số lượng, khối lượng đã được nghiệm thu.\n'
             '\n'
             'e) Đối với kinh phí hợp đồng đặt hàng, đâu thầu cho nhà cung cấp dịch vụ sự nghiệp '
             'công khác; nhà sản xuất, cung ứng sản phẩm, dịch vụ công ích, Kho bạc Nhà nước kiểm '
             'soát: \n'
             '- Sản phẩm, dịch vụ đặt hàng thuộc danh mục Dịch vụ sự nghiệp công sử dụng kinh phí '
             'NSNN giao nhiệm vụ, đặt hàng hoặc đầu thầu và danh mục sản phẩm, dịch vụ công ích '
             'thực hiện đặt hàng hoặc đấu thầu (Phụ lục số I và Phụ lục số II ban hành kèm theo '
             'Nghị định số 32/2019/NĐ-CP) hoặc Quyết định của cấp có thấm quyền ban hành danh mục '
             'dịch vụ sự nghiệp công sử nử dụng NSMNN thực hiện thẻo phường thức đặt hàng; \n'
             '- Kho bạc Nhà nước căn cứ quy định tại hợp đồng đã được ký] kết giữa cơ quan đặt '
             'hàng với nhà cung cấp dịch vụ sự nghiệp công khác; nhà sản xuất, cung ứng sản phẩm, '
             'địch vụ công ích (theo Mẫu số 03 ban hành kèm theo Nghị định số 32/2019/NĐ-CP) và '
             'biên bản nghiệm thu, thanh lý hợp đồng (theo Mẫu số 04 ban hành kèm theo Nghị định '
             'số 32/2019/NĐ-CP) để kiểm soát theo đúng đơn giá, giá đặt hàng, mức trợ giá, giá '
             'tiêu thụ quy định tại các văn bản quy phạm pháp luật. \n'
             '.2„5. Đối với chỉ chương trình mục tiêu quốc gia sử dụng nguồn kinh phí sự nghiệp: \n'
             '__ Kho bạc Nhà nước căn cứ các hồ sơ kiểm soát chỉ theo quy định tại khoản 5 Điều 7 '
             'Nghị định số 11/2020/NĐ-CP để kiểm soát đảm bảo đúng theo quy định tại khoản 1 Điều '
             'này và định mức quy định tại Thông tư số 55/2023/TT-BTC ngày 15 tháng 8 năm 2023 của '
             'Bộ Tài chính quy định quản lý, sử dụng và quyết toán kinh phí sự nghiệp từ nguồn '
             'NSNN thực hiện các chương trình mục tiêu quốc gia giai đoạn 2021-2025.'},
 {'id': 122,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 5',
               'article_title': 'Nội dung kiểm soát chỉ qua Kho bạc Nhà nước',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua '
                                  'Kho bạc Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '13/21'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua Kho bạc Nhà nước '
             '(tiếp theo)]\n'
             '2.6. Đối với các khoản chỉ mà đơn vị sử dụng ngân sách ủy quyền cho Kho bạc Nhà nước '
             'thanh toán tự động theo định kỳ cho một số nhà cung cấp dịch vụ hoặc đơn vị được nhà '
             'cung cấp dịch vụ ủy quyền thu hộ (như điện, nước, viễn thông): : \n'
             'Kho bạc Nhà nước căn cứ vào văn bản ủy quyền thanh toán của đơn vị sử dụng ngân sách '
             'cho Kho bạc Nhà nước và số dư tài khoản dự toán, tài khoản tiền gửi của đơn vị để '
             'định kỳ tự động thanh toán cho nhà cung cấp dịch vụ -_ (điện/nước/viễn thông) theo '
             'đúng Bảng kê số tiền đơn vị sử dụng ngân sách phải thanh toán do các nhà cung cấp '
             'dịch vụ lập hoặc đơn vị được nhà cung cấp dịch vụ ủy quyền thu hộ lập gửi Kho bạc '
             'Nhà nước; đồng thời, sau khi thanh toán, Kho \n'
             ' [ 14 | | bạc Nhà nước gửi đơn vị 01 liên chứng từ giấy (chứng từ báo nợ) để xác '
             'nhận đã thực hiện thanh toán. Ị\n'
             '\n'
             '2.7. Đối với chỉ đào tạo, bồi dưỡng nhân lực ở nước ngoài bằng NSNN: \n'
             'Đơn VỊ SỬ dụng ngân sách lập Giấy rút dự toán ngân sách bằng ngoại tệ trên cơ sở '
             'tổng số tiền tại danh sách chuyển kinh phí đào tạo cho lưu học sinh theo từng '
             'nước/trường, từng loại ngoại tỆ và gửi đến Kho bạc Nhà nước kèm theo các \n'
             '. hồ sơ.quy định tại Điều 7 Nghị định số. 11/2020/NĐ-CP;- danh sách chuyền kinh phí '
             'đào tạo cho lưu học sinh phải đầy đủ các nội dung sau: Họ và tên lưu học sinh; tên '
             'nước lưu học sinh đang theo học; nội dung chỉ; số tiền bằng ngoại tệ cho từng lưu '
             'học sinh; tên tài khoản người hưởng (tên cơ sở đào tạo nước ngoài hoặc đơn vị cá '
             'nhân được hưởng); số tài khoản người hưởng: mã Swift code của ngân hàng người hưởng '
             'hoặc tên và địa chỉ đầy đủ của ngân hàng người hưởng; ngân hàng . trung gian (nếu '
             'có); phí chuyến tiền; khác (nếu có). \n'
             'Đơn vị sử dụng ngân sách chịu trách nhiệm về tính hợp pháp, hợp lệ vàitính chính xác '
             'của các nội dung trong danh sách chuyển kinh phí đào tạo cho lưu học sinh và các hồ '
             'sơ gửi đến Kho bạc Nhà nước theo đúng quy định. \n'
             'Khô bạc Nhà nước căn cử Dự toán . được giao, hỗ sơ đơn vị gửi đến và các quy định '
             'tại Thông tư liên tịch số 144/2007/TTLT-BTC-BGDĐT-BNG ngày 05 tháng 12 năm 2007 của '
             'Liên Bộ Tài chính, Bộ Giáo dục và Đào tạo, Bộ Ngoại giao hướng dẫn chế độ, cấp phát '
             'và quản lý kinh phí đào tạo lưu học sinh Việt Nam ở nước ngoài bằng nguồn vốn NSNN '
             '(Thông tư liên tịch số 144/2007/TTLT- -BTC-BGDĐT-BNG), Thông tư liên tịch sô '
             '206/2010/TTLT-BTC-BGDĐT-BNG ngày 15 tháng 12 năm 2010 của Liên Bộ Tài chính, Bộ Giáo '
             'dục và Đào tạo, Bộ Ngoại giao sửa đối, bỗ sung Thông tư liên tịch số 144/2007/TTLT- '
             'BTC- BGDĐT- BNG để kiểm soát và làm thủ tục đề nghị Ngân hàng chuyến tiền từ tài '
             'khoản thanh toán của Kho bạc Nhà nước đến từng đối tượng học sinh. Ị'},
 {'id': 123,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 5',
               'article_title': 'Nội dung kiểm soát chỉ qua Kho bạc Nhà nước',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua '
                                  'Kho bạc Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '14/21'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua Kho bạc Nhà nước '
             '(tiếp theo)]\n'
             '2.8. Đối với chỉ thực hiện nhiệm vụ khoa học và công nghệ sử dụng NSNN:. \n'
             'Kho bạc Nhà nước căn cứ hồ sơ theo quy định tại Nghị định số 11/2020/NĐ-CP và quy '
             'định về kiểm soát, thanh toán tại Thông tư liên tịch số 27/2015/TTLT-BKHCN-BTC ngày '
             '30 tháng 12 năm 2015 của Liên Bộ Khoa học và Công nghệ và Bộ Tài chính quy định '
             'khoán chỉ thực hiện nhiệm vụ khba học và công nghệ sử dụng ngân sách nhà nước; Thông '
             'tư số 55/2015/TTLT-BTC- ĐKHCN ngày 22 tháng 4 năm 2015 của Liên Bộ Khoa học và Công '
             'nghệ và Bộ Tài chính hướng dẫn định mức xây dựng, phân bổ dự toán và quyết toán kinh '
             'phí . đối với nhiệm vụ khoa học và công nghệ có sử dụng ngân sách nhà nước; Thông tư '
             'số 03/2023/TT-BTC ngày 10 tháng 1 năm 2023 của Bộ Tài chính quy định lập dự toán, '
             'quản lý sử dụng và quyết toán kinh phí ngân sách nhà nước thực hiện nhiệm vụ khoa '
             'học và công nghệ. \n'
             'I | \n'
             '-w “&'},
 {'id': 124,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 5',
               'article_title': 'Nội dung kiểm soát chỉ qua Kho bạc Nhà nước',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua '
                                  'Kho bạc Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '15/21'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua Kho bạc Nhà nước '
             '(tiếp theo)]\n'
             '2.9. Đối với các khoản chỉ đoàn ra: \n'
             'Kho. bạc Nhà nước căn cứ hồ sơ theo quy định tại Điều 7 Nghị định số 11/2020/NĐ-CP '
             'và tiêu chuẩn, định mức quy định tại Thông tư số 102/2012/TT- BTC ngày 21 tháng 6 '
             'năm 2012 của Bộ Tài chính quy định chế độ công tác phí \n'
             '.cho cán bộ, công chức nhà nước đi công tác ngắn hạn ở nước ngoài do ngân sách nhà '
             'nước bảo đảm kinh phí để thực hiện kiểm soát, thanh toán cho đơn vị.\n'
             '\n'
             '3. Đối với các khoản chỉ từ tài khoản tiền gửi tại Kho-bạc Nhà nước: \n'
             'Kho bạc Nhà nước căn cứ vào nguồn hình thành và nội dung chỉ để thực hiện kiểm soát, '
             'thanh toán từ tài khoản tiền gửi của đơn vị sử dụng ngân sách:'},
 {'id': 125,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 5',
               'article_title': 'Nội dung kiểm soát chỉ qua Kho bạc Nhà nước',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua '
                                  'Kho bạc Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '16/21'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua Kho bạc Nhà nước '
             '(tiếp theo)]\n'
             '3.1. Kho bạc Nhà nước thực hiện kiểm soát, thanh toán từ tài khoản tiền gửi đối với '
             'các nguồn hình thành như sau:\n'
             '\n'
             'a) Nguồn hình thành từ NSNN theo quy định: - . \n'
             'Kho bạc Nhà nước thực hiện kiểm soát, thanh toán theo quy định tại khoản 1, khoản 2 '
             'Điều này và theo quy định tại Thông tư số 55/2017/TT-BTC ngày 19 tháng 05 năm 2017 '
             'của Bộ Tài chính quy định chỉ tiết về quản lý, sử dụng NSNN đối với một số hoạt động '
             'thuộc lĩnh vực an ninh, trật tự, an toàn xã hội; Thông tư số 369/2017/T1 -BTC ngày '
             '11 tháng 4 năm 2017 của Bộ Tài chính hướng dẫn thực hiện cấp kinh phí thường xuyên '
             'từ NSNN để thực hiện nhiệm vụ chi quốc phòng của Bộ Quốc phòng; Thông tư số '
             '1539/2017/T1- BTC ngày 29 tháng 12 năm 2017 của Bộ Tài chính quy định chỉ tiết về '
             'quản lý, sử dụng NSNN đối với các cơ quan của Đảng Cộng sản Việt Nam (đối với lĩnh '
             'vực an nỉnh, trật tự, an toàn xã hội nhiệm vụ chi quốc phòng của Bộ Quốc phòng, các '
             'cơ quan của Đảng Cộng sản Việt Nam).\n'
             '\n'
             'b) Nguồn hình thành từ nguồn thu phí được để lại cho đơn vị sử dụng theo quy định '
             'của pháp luật về phí, lệ phí: \n'
             'Kho bạc Nhà nước thực hiện kiểm soát, thanh toán theo quy định tại khoản 1, khoản 2 '
             'Điều này và kiểm soát nội dung chi đúng theo quy định tại Điều 5 Nghị định số '
             '120/2016/NĐ-CP ngày 23 tháng 8 năm 2016 của Chính phủ quy định chỉ tiết và hướng dẫn '
             'thi hành một số điều của Luật Phí và lệ phí (Nghị định số 120/2016/NĐ-CP); Nghị định '
             'số 82/2023/NĐ-CP ngày 28 tháng 11 năm 2023 của Chính phủ sửa đổi, bổ sung một số '
             'điều của Nghị định số 120/2016/NĐ-CP và quy định của Bộ Tài chính về hướng dẫn thu, '
             'nộp quản lý và sử dụng từng loại phí. :\n'
             '\n'
             'c) Nguồn hình thành từ nguồn thu dịch vụ khám bệnh, chữa bệnh, dịch vụ y tế dự '
             'phòng, học phí theo mức giá quy định của cơ quan nhà nước có thâm quyền đối với đơn '
             'vị sự nghiệp công lập tự bảo đảm một phần chỉ thường xuyên (đơn vị nhóm 3) và đơn vị '
             'sự nghiệp công do nhà nước đảm bảo chỉ thường xuyên (đơn vị nhóm 4): é *° \n'
             ' | | \n'
             'Kho bạc Nhà nước thực hiện kiểm soát, thanh toán theo quy định tại khoản 1, khoản 2 '
             'Điều này. | \n'
             'đ) Nguồn hình thành từ nguồn kinh phí tiết kiệm được, được trích quỹ đối với cơ quan '
             'nhà nước: | \n'
             'Kho bạc Nhà nước thực hiện kiểm soát, thanh toán theo quy định tại khỏản 1, khoản 2 '
             'Điều này và quy chế chỉ tiêu nội bộ của đơn VỊ, các nội dung được phép chỉ từ các '
             'quỹ theo quy định tại Thông tư liên tịch-số 71/2014/TTLT- BTC- \n'
             '“BNV và các quy định của pháp luật hiện hành có liên quan. \n'
             'đ) Tiền gửi có nguồn hình thành từ các khoản thu hợp pháp khác theo quy định của '
             'pháp luật của cơ quan nhà nước thực hiện chê độ tự chủ, tự chịu trách nhiệm về sử '
             'dụng biên chế và kinh phí quản lý hành chính: \n'
             'Kho bạc Nhà nước thực hiện kiểm soát, thanh toán theo quy định tại khoản 1, khoản 2 '
             'Điều này.'},
 {'id': 126,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 5',
               'article_title': 'Nội dung kiểm soát chỉ qua Kho bạc Nhà nước',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua '
                                  'Kho bạc Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '17/21'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua Kho bạc Nhà nước '
             '(tiếp theo)]\n'
             '“BNV và các quy định của pháp luật hiện hành có liên quan. \n'
             'đ) Tiền gửi có nguồn hình thành từ các khoản thu hợp pháp khác theo quy định của '
             'pháp luật của cơ quan nhà nước thực hiện chê độ tự chủ, tự chịu trách nhiệm về sử '
             'dụng biên chế và kinh phí quản lý hành chính: \n'
             'Kho bạc Nhà nước thực hiện kiểm soát, thanh toán theo quy định tại khoản 1, khoản 2 '
             'Điều này. \n'
             'Riêng đổi với chỉ phí thẩm tra, phê duyệt quyết toán vốn đầu tư công dự á án hoàn '
             'thành: Thực hiện theo quy định tại khoản 2 Điều 46 Nghị định số \n'
             '—:——--99/2021/NĐ-CP-ngày-L1 tháng HH năm.-2021 của Chính phủ-quy định. về quản lý, : '
             'thanh toán, quyết toán dự án sử dụng vốn đầu tư công.'},
 {'id': 127,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 5',
               'article_title': 'Nội dung kiểm soát chỉ qua Kho bạc Nhà nước',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua '
                                  'Kho bạc Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '18/21'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua Kho bạc Nhà nước '
             '(tiếp theo)]\n'
             '3.2. Đối với tiền gửi có nguồn hình thành không thuộc điểm a khoản 3 Điều nảy, Kho '
             'bạc Nhà nước kiểm tra tính hợp pháp, hợp lệ của Ủy nhiệm chỉ, trong phạm vi số dư '
             'tài khoản tiền gửi. \n'
             '4, Đối với đơn vị sự nghiệp công. lập thực hiện chế độ tự chủ, tự chịu trách nhiệm '
             'theo quy định tại Nghị định số 60/2021/NĐ-CP:\n'
             '\n'
             '4.1. Các nguồn tài chính của đơn vị sự nghiệp công mở tài khoản tại Kho bạc Nhà nước '
             'để kiểm soát, thanh toán theo quy định tại khoản 2, khoản 3 Điều 23 Nghị định sô '
             '60/2021/NĐ-CP. \n'
             'Kho bạc Nhà nước thực hiện kiểm soát, thanh toán theo quy định tại khoản 1, khoản 2, '
             'điểm 3.1 khoản 3 Điều này trong đó lưu ý: ị\n'
             '\n'
             'a) Đối với kinh phí chỉ thường xuyên giao tự chủ: Kiểm soát đảm bảo theo \n'
             'đúng quy chế chỉ tiêu nội bộ và Quyết định giao quyền tự chủ của cấp có thâm quyền; '
             'chế độ, tiêu chuẩn, định mức do cơ quan nhà nước có thấm quyền quy định, cụ thể: \n'
             '- Đơn vị sự nghiệp công tự bảo đảm chỉ thường xuyên và chỉ đầu tư (đơn vị nhóm 1) và '
             'đơn vị sự nghiệp công tự bảo đảm chỉ thường xuyên (đơn vị nhóm 2): Thực hiện theo '
             'quy định tại Điêu 12 Nghị định sô-60/2021/NĐ-CP. \n'
             '- Đơn vị sự nghiệp công tự bảo đảm một phần chỉ thường xuyên (đơn vị nhóm 3): Thực '
             'hiện theo quy định tại Điều 16 Nghị định số 60/2021/NĐ-CP. \n'
             'te §ự- \n'
             '\n'
             ' - Đơn vị sự nghiệp công do nhà nước đảm bảo chỉ thường xuyên (đơn vị nhóm 4): Thực '
             'hiện theo quy định tại Điều 20 Nghị định số 60/2021/NĐ-CP.\n'
             '\n'
             'b) Đối với kinh phí chỉ thường xuyên không giao tự chủ: Kho bạc Nhà nước kiểm soát '
             'đảm bảo theo đúng tiêu chuẩn, định mức quy định tại các văn bản quy phạm pháp luật '
             'hiện hành đối với từng nguồn kinh phí theo quy định tại Điều 13,\n'
             'Điều 17, Điều 21 Nghị định số 60/2021/NĐ-CP và khoản 5 Điều 11 Thông tư số '
             '56/2022/TT-BTC.\n'
             '\n'
             '4.2. Đối với các nguồn tài chính của đơn vị sự nghiệp công được mở tài khoản tại '
             'ngân hàng thương mại theo quy định tại khoản 1 và khoản 4 Điều 23 Nghị định số '
             '60/2021/NĐ-CP, trường hợp đơn vị sự nghiệp công đề nghị mở tài khoản tiền gửi tại '
             'Kho bạc Nhà nước, Kho bạc Nhà nước không kiểm soát chỉ, thực hiện chỉ trả theo đề '
             'nghị của đơn vị sự nghiệp công. Đơn vị sự nghiệp công chịu trách nhiệm quản lý, sử '
             'dụng và chỉ tiêu theo đúng quy định của pháp luật.'},
 {'id': 128,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 5',
               'article_title': 'Nội dung kiểm soát chỉ qua Kho bạc Nhà nước',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua '
                                  'Kho bạc Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '19/21'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua Kho bạc Nhà nước '
             '(tiếp theo)]\n'
             '4.3. Đối với phân phối kết quả tài chính trong năm: \n'
             '- Đơn vị sự nghiệp công lập Giấy rút dự toán/Ủy nhiệm chỉ để để nghị Kho bạc Nhà '
             'nước trích quỹ hoặc tạm trích quỹ theo quy định tại Điều 10 Thông tư số '
             '56/2022/TT-BTC. Kho bạc Nhà nước căn cứ đề nghị trích lập quỹ của đơn vị sự nghiệp '
             'công thực hiện trích quỹ và chuyển tiền sang tài khoản tiền gửi tại ngân hàng thương '
             'mại hoặc tải khoản tiền gửi tại Kho bạc Nhà nước của đơn vị sự nghiệp công theo đề '
             'nghị của đơn vị. Kho bạc Nhà nước không kiểm soát việc sử dụng các quỹ. Đơn vị sự '
             'nghiệp công chịu trách nhiệm quản lý, sử dụng và chỉ tiêu theo đúng quy định của '
             'pháp luật. \n'
             "Riêng đối với đơn vị sự :' nghiệp công nhóm 4, đối với các khoản chỉ từ kinh phí "
             'tiết kiệm được, Kho bạc Nhà nước kiêm soát đảm bảo theo đúng quy chế chỉ tiêu nội bộ '
             'của đơn vị và theo quy định tại Điều 22 Nghị định số 60/2021/NĐ-CP, khoản 3 Điều 10 '
             'Thông tư số 56/2022/TT-BTC.\n'
             '\n'
             '5. Đối với cơ quan nhà nước:'},
 {'id': 129,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 5',
               'article_title': 'Nội dung kiểm soát chỉ qua Kho bạc Nhà nước',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua '
                                  'Kho bạc Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '20/21'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua Kho bạc Nhà nước '
             '(tiếp theo)]\n'
             '5. Đối với cơ quan nhà nước:\n'
             '\n'
             '5.1. Đối với cơ quan nhà nước thực hiện tự chủ, tự chịu trách nhiệm theo quy định '
             'tại Nghị định số 130/2005/NĐ-CP ngày 17 tháng 10 năm 2005 của Chính phủ quy định chế '
             'độ tự chủ, tự chịu trách nhiệm về sử dụng biên chế và kinh phí quản lý hành chính '
             'đối với cơ quan nhà nước và Nghị định số ] 17/2013/NĐ- CP ngày 07 tháng 10 năm 2013 '
             'của Chính phủ sửa đổi, bỗ sung một số điều của Nghị định số 130/2005/NĐ-CP: :\n'
             '\n'
             'a) Đối với kinh phí chỉ quản lý hành chính theo dự toán NSNN được cấp có thẩm quyền '
             'giao: Kho bạc Nhà nước thực hiện kiểm soát, thanh toán theo quy định tại khoản 1, '
             'khoản 2, điểm 3.1 khoản 3 Điều này, theo đúng quy chế chỉ tiêu nội bộ và chế độ, '
             'tiêu chuẩn, định mức do cơ quan nhà nước có thâm quyền quy định, trong đó lưu ý: \n'
             '*“ $C \n'
             ' I8 \n'
             '- Đối với kinh phí NSNN giao thực hiện chế độ tự chủ: \n'
             '+ Kho bạc Nhà nước kiểm soát kinh phí giao thực hiện chế độ tự chủ theo đúng quy '
             'định tại khoản 5, khoản 6 Điều 3 Thông tư liên tịch số 71/2014/TT LT- BTC-BNV. \n'
             '+Kinh phí chi thường xuyên được để lại cho đơn vị đối với nguồn thu phí theo pháp '
             'luật về phí, lệ phí (nếu có): \n'
             'Căn cứ dự toán chỉ từ nguồn thu phí do cơ quan có thầm quyền BlaO; chế độ, tiêu '
             'chuẩn, định mức chỉ NSNN do cơ quan nhà nước có thẩm quyền quy định, Thông tư liên '
             'tịch số 71/2014/TTLT-BTC-BNV, Kho bạc Nhà nước thực hiện kiểm soát, thanh toán cho '
             'đơn vị theo các nội dung chỉ quy định tại Nghị định số 120/2016/NĐ-CP. \n'
             '- Đối với kinh phí NSNN giao nhưng không thực hiện chế độ tự chủ: Kho bạc Nhà nước '
             'kiểm soát nội ¡ dụng chỉ theo đúng quy định tại Điều 4 Thông tư liên \n'
             'tịch số 71/2014/TTLT-BTC-BNV và tiêu chuẩn, định mức quy định tại các văn bản quy '
             'phạm pháp luật hiện hành. \n'
             'T—— ___b) Kiểm soát việc sử dụng kinh phí tiết kiệm được ¡ trongnăm __ |... \n'
             '- Kho bạc Nhà nước kiểm soát việc sử dụng kinh phí tiết kiệm được theo đúng các nội '
             'dung quy định tại điểm b khoản 7 Điều 3 Thông tư liên tịch số 71/2014/TT1LT-BTC-BNV. '
             '| \n'
             '- Đối với thu nhập tăng thêm, chỉ phúc đợi, chỉ khen thưởng từ nguồn. kinh phí tiết '
             'kiệm được, Kho bạc Nhà nước kiểm soát theo quy định tại khoản 3 Điều 5 Thông tư liên '
             'tịch số 71/2014/TTLT-BTC-BNV. | \n'
             '- Số kinh phí tiết kiệm được, cuối năm chưa sử dụng hết được chuyển vào quỹ dự phòng '
             'ô ổn định thu nhập theo quy định tại điểm b khoản 7 Điều 3 Thông tư liên tịch số '
             '71/2014/TTLT-BTC-BNV. Đối với Quỹ dự phòng để ổn định thu nhập: Đơn vị sử dụng ngân '
             'sách lập Giấy rút dự toán/Ủy nhiệm chi để đề nghị Kho bạc Nhà nước trích quỹ dự '
             'phòng để ổn định thu nhập. Kho bạc Nhà i nước căn cứ đề nghị trích lập quỹ của đơn '
             'vị thực hiện trích quỹ và chuyền tiền sang tài khoản tiền gửi quỹ của đơn vị. Kho '
             'bạc Nhà nước kiểm soát việc sử dụng quỹ \n'
             'theo quy định tại điểm 3.1 khoản 3 Điều này.'},
 {'id': 130,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương I QUY ĐỊNH CHUNG',
               'article': 'Điều 5',
               'article_title': 'Nội dung kiểm soát chỉ qua Kho bạc Nhà nước',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua '
                                  'Kho bạc Nhà nước',
               'topics': 'Hạn mức & Giao dịch, Mở & Đóng tài khoản, Bảo mật & Rủi ro',
               'is_core_fintech': True,
               'page': 6,
               'chunk_part': '21/21'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 5. Nội dung kiểm soát chỉ qua Kho bạc Nhà nước '
             '(tiếp theo)]\n'
             '5.2. Đối với cơ quan nhà nước thực hiện theo cơ chế tài chính đặc thù theo văn bản '
             'quy phạm pháp luật: \n'
             'Kho bạc Nhà nước thực hiện kiểm soát, thanh toán theo quy định tại Khoản 1, khoản 2, '
             'điểm 3.1 khoản 3 Điều này và các quy định về quản lý kinh phí hoạt \n'
             'động, về sử dụng kinh phí tiết kiệm được trong năm tại các văn bản quy phạm pháp '
             'luật về cơ chế tài chính đặc thù cho cơ quan nhà nước đó.\n'
             '\n'
             '6. Đối với các khoản chỉ theo hình thức lệnh chỉ tiền: tạ bu \n'
             '| \n'
             '\n'
             ' Kho bạc Nhà nước thực hiện theo quy định tại Điều 19 Thông tư số 342/2016/TT-BTC '
             'ngày 30 tháng 12 năm 2016 của Bộ Tài chính quy định chỉ tiết và hướng dẫn thi hành '
             'một số điều của Nghị định số 163/2016/NĐ-CP ngày 21 tháng 12 năm 2016 của Chính phủ '
             'quy định chỉ tiết thi hành một số điều của Luật NSNN.'},
 {'id': 131,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương II  - TRÁCH NHIỆM, QUYÈN HẠN CỦA CƠ QUAN TÀI CHÍNH, KHO BẠC NHÀ '
                          'NƯỚC VÀ CÁC ĐƠN VỊ SỬ DỤNG NGÂN SÁCH',
               'article': 'Điều 6',
               'article_title': 'Tạm ứng và thanh toán tạm ứng',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 6. Tạm ứng và thanh toán tạm '
                                  'ứng',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 20,
               'chunk_part': '1/6'},
  'content': 'Điều 6. Tạm ứng và thanh toán tạm ứng'},
 {'id': 132,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương II  - TRÁCH NHIỆM, QUYÈN HẠN CỦA CƠ QUAN TÀI CHÍNH, KHO BẠC NHÀ '
                          'NƯỚC VÀ CÁC ĐƠN VỊ SỬ DỤNG NGÂN SÁCH',
               'article': 'Điều 6',
               'article_title': 'Tạm ứng và thanh toán tạm ứng',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 6. Tạm ứng và thanh toán tạm '
                                  'ứng',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 20,
               'chunk_part': '2/6'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 6. Tạm ứng và thanh toán tạm ứng (tiếp theo)]\n'
             '1. Tạm ứng áp dụng đối với khoản chỉ NSNN của đơn vị sử dụng ngân sách chưa có đủ '
             'điều kiện thực hiện việc thanh toán trực tiệp từ Kho bạc Nhà nước.'},
 {'id': 133,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương II  - TRÁCH NHIỆM, QUYÈN HẠN CỦA CƠ QUAN TÀI CHÍNH, KHO BẠC NHÀ '
                          'NƯỚC VÀ CÁC ĐƠN VỊ SỬ DỤNG NGÂN SÁCH',
               'article': 'Điều 6',
               'article_title': 'Tạm ứng và thanh toán tạm ứng',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 6. Tạm ứng và thanh toán tạm '
                                  'ứng',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 20,
               'chunk_part': '3/6'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 6. Tạm ứng và thanh toán tạm ứng (tiếp theo)]\n'
             '2. Nội dung tạm ứng: Theo quy định của hợp đồng (đối với trường hợp thực hiện theo '
             'hợp đồng) và theo đề nghị của đơn vị sử dụng ngân sách (đối với trường hợp không có '
             'hợp đồng hoặc hợp đồng có giá trị không vượt quá 50 triệu đồng).\n'
             '\n'
             '3. Mức tạm ứng:\n'
             '\n'
             '3.1. Đối với những khoản chỉ có giá trị hợp đồng trên 50 triệu đồng, mức tạm ứng '
             'theo quy định tại hợp đồng đã ký kết của đơn vị sử dụng NSNN và nhà cung cấp hàng '
             'hóa, địch vụ. Đơn vị sử dụng ngân sách được tạm ứng vốn một lần hoặc nhiều lần cho '
             'một hợp đồng nhưng tôi đa không vượt quá 50% giá trị hợp đồng tại thời điểm ký kết '
             'và không vượt quá dự toán năm được cấp có thẩm quyền phê duyệt cho khoản chỉ đó, trừ '
             'trường hợp sau:\n'
             '\n'
             'a) Đối với.hàng hóa nhập khẩu, thiết bị chuyên dùng do đơn vị sử dụng ngân sách phải '
             'nhập khẩu trực tiếp từ nước ngoài (hoặc thông qua một đơn vị nhập. khẩu ủy thác) '
             'phải mở thư tín dụng (L/C) tại ngân hàng và trong hợp đồng _ nhà cung cấp yêu cầu '
             'phải tạm ứng lớn hơn nhưng không vượt quá dự toán năm được cấp có thẩm quyền phê '
             'duyệt cho khoản chỉ đó.\n'
             '\n'
             'b) Đối với các trường hợp đặc thù khác có hướng dẫn riêng của cơ quan nhà nước có '
             'thấm quyên, việc tạm ứng được thực hiện trong phạm vi dự toán được giao, theo hợp '
             'đồng ký kết giữa đơn vị sử dụng ngân sách và nhà cung cấp và theo quyết định của cấp '
             'có thẩm quyền.'},
 {'id': 134,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương II  - TRÁCH NHIỆM, QUYÈN HẠN CỦA CƠ QUAN TÀI CHÍNH, KHO BẠC NHÀ '
                          'NƯỚC VÀ CÁC ĐƠN VỊ SỬ DỤNG NGÂN SÁCH',
               'article': 'Điều 6',
               'article_title': 'Tạm ứng và thanh toán tạm ứng',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 6. Tạm ứng và thanh toán tạm '
                                  'ứng',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 20,
               'chunk_part': '4/6'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 6. Tạm ứng và thanh toán tạm ứng (tiếp theo)]\n'
             '3.2. Đối với những khoản chi không có hợp đồng và những khoản chỉ có giá trị hợp '
             'đồng không vượt quá 50 triệu đồng: Mức tạm ứng theo đề nghị của đơn vị sử dụng ngân '
             'sách và trong phạm vi dự toán được giao. Đơn vị sử dụng ngân sách chịu trách nhiệm '
             'về mức đề nghị tạm ứng theo đúng quy định.\n'
             '\n'
             '4. Thanh toán tạm ứng: Đơn vị sử dụng ngân sách phải thực hiện thanh toán tạm ứng '
             'với Kho bạc Nhà nước nơi giao dịch cụ thê như sau:\n'
             '\n'
             '4.1. Đối với các khoản chỉ không có hợp đồng hoặc khoản chỉ không phải gửi hợp đồng '
             'đến Kho bạc Nhà nước, đơn vị sử dụng. ngân sách phải thanh toán tạm ứng chậm nhất '
             'với Kho bạc Nhà nước vào ngày cuối cùng của tháng kế tiếp tháng tạm ứng (trừ khoản '
             'chi trợ cấp, phụ cấp theo quy định tại Thông tư số 44/2022/TT-BTC và Thông tư số '
             '76/2021/TT-BTC). \n'
             's. .. \n'
             ' | 20 | \n'
             '|\n'
             '\n'
             '4.2. Đối với các khoản chỉ phải gửi Hợp đồng đến Kho bạc Nhà nước (trường hợp hợp '
             'đồng có giá trị trên 50 triệu đồng), đơn vị sử dụng ngân sách thanh toán tạm ứng bắt '
             'đầu ngay từ lần thanh toán khối lượng hoàn thành đầu tiên của Hợp đồng, mức thanh '
             'toán tạm ứng từng lần do đơn vị sử dụng ngân sách thống nhất với nhà cung cấp hàng '
             'hóa, dịch vụ và quy định cụ thể trong Hợp đồng, đảm bảo thanh toán hết tạm ứng khi '
             'giá trị thanh toán (bao gồm cả tạm ứng và thanh toán khối lượiig hoàn thành) đạt 80% '
             'giá trị hợp đồng. ˆ'},
 {'id': 135,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương II  - TRÁCH NHIỆM, QUYÈN HẠN CỦA CƠ QUAN TÀI CHÍNH, KHO BẠC NHÀ '
                          'NƯỚC VÀ CÁC ĐƠN VỊ SỬ DỤNG NGÂN SÁCH',
               'article': 'Điều 6',
               'article_title': 'Tạm ứng và thanh toán tạm ứng',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 6. Tạm ứng và thanh toán tạm '
                                  'ứng',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 20,
               'chunk_part': '5/6'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 6. Tạm ứng và thanh toán tạm ứng (tiếp theo)]\n'
             '4.3. Khi thanh toán tạm. ứng, đơn vị sử dụng ngân sách có trách nhiệm gửi đến Kho '
             'bạc Nhà nước Giấy đề nghị thanh toán tạm ứng; Bảng kê nội dung thanh toán/tạm ú ứng '
             '(đối với những khoản chỉ không có hợp đồng hoặc những khoản chỉ có hợp đồng với giá '
             'trị không quá 50 triệu đồng), Các hồ sơ, chứng từ tương lứng có liên quan theo quy '
             'định để Kho bạc Nhà nước kiểm soát, thanh toán trừ các hồ sơ đã gửi khi tạm ứng). |\n'
             '\n'
             'a) Trường hợp đủ điều kiện quy. định, Kho bạc Nhà nước thực biện thanh toán tạm ứng '
             'cho đơn vị sử đụng ngân sách, cụ thể: | \n'
             '- Nếu số tiền đề nghị thanh toán lớn hơn số đã tạm ứng: Căn cứ vào Giấy \n'
             '— : _đề nghị thanh-toántạm-ứng của đơn v†(đã được: Kho bạc Nhà nước kiểm soát và- TT '
             'chấp nhận thanh toán), Kho bạc Nhà nước làm thủ tục chuyên. từ tạm Ứng sang thanh '
             'toán (số tiền đã tạm ứng); đồng thời, đơn vị lập thêm Giấy rút dự toánỦy nhiệm chỉ '
             'gửi Kho bạc Nhà nước đề thanh toán cho đơn vị sô chênh lệch giữa số tiền Kho bạc Nhà '
             'nước chấp nhận thanh toán và số tiền đã tạm ứng. - Nếu số tiền đề nghị thanh toán '
             'nhỏ hơn số tiền đã. tạm ứng: căn cứ hồ sơ đề nghị thanh toán tạm ứng của đơn vị (đã '
             'được Kho bạc Nhà nước kiểm soát và chấp nhận thanh toán), Kho bạc Nhà nước làm thủ '
             'tục chuyển từ tạm ứng sang thanh toán (bằng số tiền Kho bạc Nhà nước chấp nhận thanh '
             'toán tạm ứng), số tiền \n'
             'chênh lệch sẽ được theo đõi để thu hồi hoặc thanh toán vào lần thanh toán sau. \n'
             '- Nếu số tiền đề nghị thanh toán bằng số tiền đã tạm ứng: căn cứ hồ sơ đề nghị thanh '
             'toán tạm ứng của đơn vị (đã được Kho bạc Nhà nước kiểm sDất và chấp nhận thanh '
             'toán), Kho bạc Nhà nước làm thủ tục chuyển từ tạm ứng sang thanh toán (bằng số tiền '
             'Kho bạc Nhà nước chấp nhận thanh toán tạm ứng).\n'
             '\n'
             'b) Tất cả các khoản đã tạm ứng (kể cả tạm ứng bằng tiền mặt và bằng chuyển khoản) để '
             'chỉ theo dự toán NSNN đến hết thời gian chỉnh lý quyết toán chưa đủ hồ sơ, thủ tục '
             'thanh toán được xử lý theo quy định tại Điều 42 Nghị định số 163/2016/NĐ-CP và Điều '
             '26 Thông tư sô 342/2016/TT-BTC. |\n'
             '\n'
             '5. Bảo lãnh tạm ứng: \n'
             'Đối với các hợp đồng có thỏa thuận về bảo lãnh tạm ứng trong điều khoản của hợp '
             "đồng: ' \n"
             '*ứ vn h ~ \n'
             '\n'
             '\n'
             '5.1. Trước khi Kho bạc Nhà nước thực hiện việc tạm ứng hợp đồng cho đơn vị sử dụng '
             'ngân sách để tạm ứng vốn cho nhà thầu hoặc nhà cung cấp, đơn vị sử dụng ngân sách '
             'gửi đến Kho bạc Nhà nước bảo lãnh tạm ứng của nhà thầu hoặc nhà cung cấp đã có hiệu '
             'lực với giá trị không nhỏ hơn khoản tiền tạm ứng.\n'
             '\n'
             '5.2. Giá trị của bảo lãnh tạm ứng hợp đồng sẽ được giảm trừ tương ứng với giá trị '
             'tiền tạm ứng đã thu hồi qua mỗi lần thanh toán giữa đơn vị sử dụng ngân sách và nhà '
             'thầu hoặc nhà cung cấp. Đơn vị sử dụng. ngân sách đảm bảo và chịu trách nhiệm về giá '
             'trị của bảo lãnh tạm ứng của nhà thầu hoặc nhà cùng cấp không nhỏ hơn số dư tiền tạm '
             'ứng còn lại.'},
 {'id': 136,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương II  - TRÁCH NHIỆM, QUYÈN HẠN CỦA CƠ QUAN TÀI CHÍNH, KHO BẠC NHÀ '
                          'NƯỚC VÀ CÁC ĐƠN VỊ SỬ DỤNG NGÂN SÁCH',
               'article': 'Điều 6',
               'article_title': 'Tạm ứng và thanh toán tạm ứng',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 6. Tạm ứng và thanh toán tạm '
                                  'ứng',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 20,
               'chunk_part': '6/6'},
  'content': '[Thông tư 17/2024/TT-NHNN - Điều 6. Tạm ứng và thanh toán tạm ứng (tiếp theo)]\n'
             '5.2. Giá trị của bảo lãnh tạm ứng hợp đồng sẽ được giảm trừ tương ứng với giá trị '
             'tiền tạm ứng đã thu hồi qua mỗi lần thanh toán giữa đơn vị sử dụng ngân sách và nhà '
             'thầu hoặc nhà cung cấp. Đơn vị sử dụng. ngân sách đảm bảo và chịu trách nhiệm về giá '
             'trị của bảo lãnh tạm ứng của nhà thầu hoặc nhà cùng cấp không nhỏ hơn số dư tiền tạm '
             'ứng còn lại.\n'
             '\n'
             '5.3. Thời gian có hiệu lực của bảo lãnh tạm ứng hợp đồng phải được kéo dài cho đến '
             'khi đơn vị sử dụng ngân sách thanh toán hết số tiền tạm ứng và phải được quy định cụ '
             'thể trong hợp đồng và trong văn bản bảo lãnh tạm ứng. Trường hợp bảo lãnh tạm ứng '
             'hết thời hạn nhưng đơn vị sử dụng ngân sách chưa thanh toán hết số tiền tạm ứng, đơn '
             'vị sử dụng ngân sách có trách nhiệm làm thủ tục gia hạn bảo lãnh tạm ứng và gửi Kho '
             'bạc Nhà nước đề làm căn cứ kiểm soát chỉ theo quy định. Đơn vị sử dụng ngân sách '
             'chịu trách nhiệm theo dõi về thời gian hiệu \n'
             'lực của bảo lãnh tạm ứng.\n'
             'Chương II \n'
             'TRÁCH NHIỆM, QUYÈN HẠN CỦA CƠ QUAN TÀI CHÍNH, KHO BẠC NHÀ NƯỚC VÀ CÁC ĐƠN VỊ SỬ DỤNG '
             'NGÂN SÁCH'},
 {'id': 137,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương II  - TRÁCH NHIỆM, QUYÈN HẠN CỦA CƠ QUAN TÀI CHÍNH, KHO BẠC NHÀ '
                          'NƯỚC VÀ CÁC ĐƠN VỊ SỬ DỤNG NGÂN SÁCH',
               'article': 'Điều 7',
               'article_title': 'Trách nhiệm quyền hạn của Bộ Tài chính, Sở Tài chính tỉnh, thành '
                                'phố trực thuộc trung ương, Phòng Tài chính - Kế hoạch quận, '
                                'huyện, thị xã, thành phố thuộc tỉnh (sau đây gọi chung là cơ quan '
                                'tài chính)',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 7. Trách nhiệm quyền hạn của Bộ '
                                  'Tài chính, Sở Tài chính tỉnh, thành phố trực thuộc trung ương, '
                                  'Phòng Tài chính - Kế hoạch quận, huyện, thị xã, thành phố thuộc '
                                  'tỉnh (sau đây gọi chung là cơ quan tài chính)',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 22,
               'chunk_part': '1/1'},
  'content': 'Điều 7. Trách nhiệm quyền hạn của Bộ Tài chính, Sở Tài chính tỉnh, thành phố trực '
             'thuộc trung ương, Phòng Tài chính - Kế hoạch quận, huyện, thị xã, thành phố thuộc '
             'tỉnh (sau đây gọi chung là cơ quan tài chính)\n'
             '\n'
             '1. Đảm bảo tồn quỹ NSNN các cấp để đáp ứng các nhu câu chỉ của NSNN theo quy định '
             'của Luật NSNN, Nghị định số 163/2016/NĐ-CP và các văn bản hướng dẫn Luật. Trường hợp '
             'tồn quỹ ngân sách các cấp không đáp ứng đủ nhu cầu chỉ, cơ quan tài chính được quyên '
             'yêu cầu (bằng văn bản) gửi Kho bạc Nhà nước tạm dừng thanh toán một số khoản chỉ về '
             'mua sắm, sửa chữa theo từng nhiệm vụ cụ thể để đảm bảo cân đối quỹ NSNN, nhưng không '
             'ảnh hưởng đến việc tổ chức thực hiện nhiệm vụ chính được giao của đơn vị.\n'
             '\n'
             '2. Kiểm tra, giám sát việc thực hiện chi tiêu và sử dụng ngân sách ở các đơn vị sử '
             'dụng ngân sách, trường hợp phát hiện các khoản chỉ vượt nguồn cho phép, không đúng '
             'chế độ quy định hoặc đơn vị không chấp hành chế độ báo cáo, thì có quyền yêu cầu Kho '
             'bạc Nhà nước tạm dừng thanh toán.\n'
             '\n'
             '3. Chịu trách nhiệm nhập dự toán chỉ ngân sách vào hệ thống thông tin quản lý ngân '
             'sách và kho bạc (sau đây gọi tắt là TABMIS) và phê đuyệt TABMIS theo quy định về '
             'hướng dẫn quản lý điều hành NSNN trong điều kiện áp dụng hệ thống TABMIS. \n'
             'Xi *e \n'
             ' quá hạn theo quy định tại Điều 6 Thông tư này. \n'
             '| 22 | \n'
             '|\n'
             '\n'
             '4. Đối với những khoản chỉ do cơ quan tài chính quyết định chỉ bằng hỉnh thức “lệnh '
             'chỉ tiền”: Cơ quan tài chính chịu trách nhiệm kiểm tra, kiểm soát theo đúng quy định '
             'tại Điều !9 Thông tư số 342/2016/TT-BTC. |'},
 {'id': 138,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương II  - TRÁCH NHIỆM, QUYÈN HẠN CỦA CƠ QUAN TÀI CHÍNH, KHO BẠC NHÀ '
                          'NƯỚC VÀ CÁC ĐƠN VỊ SỬ DỤNG NGÂN SÁCH',
               'article': 'Điều 8',
               'article_title': 'Trách nhiệm, quyền hạn của Kho bạc Nhà nước |',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 8. Trách nhiệm, quyền hạn của '
                                  'Kho bạc Nhà nước |',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 22,
               'chunk_part': '1/1'},
  'content': 'Điều 8. Trách nhiệm, quyền hạn của Kho bạc Nhà nước |\n'
             '\n'
             '1. Chấp hành đúng các quy định tại khoản 3 Điều 15 Nghị “định, số 11/2020/NĐ-CP, '
             'trường hợp bắt khả kháng do sự có hệ thống ứng dụng công nghệ \n'
             '" -thông. tin hoặc các nguyên nhân khách quan khác, thì được kéo dài. thời hạn. '
             'giải. \n'
             'quyết thủ tục hành chính, nhưng phải có thông báo rõ thời hạn giải quyết thủ ltục '
             'hành chính cuối cùng cho đơn vị. - |\n'
             '\n'
             '2. Tham gia với cơ quan tài chính, cơ quan quản lý nhà nước có thâm quyền trong việc '
             'kiểm tra tỉnh hình sử đựng ngân sách; xác nhận số thực chỉ, số tạm ứng, số dư kinh '
             'phí cuối năm ngân sách của các đơn vị sử dụng ngân sách tại Kho: ibac Nhà nước.\n'
             '\n'
             '3. Thực hiện kiểm. soát, thanh toán theo quy định tại Điều 2 và Điều 5 Trồng \n'
             'tư này. Trường hợp không đảm bảo theo quy định, Kho bạc Nhà nước thực hiện từ chối '
             'thanh toán đồng thời gửi Thông báo bằng văn bản đến đơn vị (theo Mẫu \n'
             '_ số 02 ban hành kèm theo Thông tử näy) và chịu trách nhiệm về quyết định! của \n'
             'mnình. Kho bạc Nhà nước không chịu trách nhiệm về chứng từ chỉ và các hồ sơ kèm theo '
             'chứng từ chi không thuộc thủ tục hành chính gửi Khơ bạc nhà nước được quy định tại '
             'các Nghị định của Chính phủ. .\n'
             '\n'
             '4. Kho bạc Nhà nước có trách nhiệm tạm dừng thanh toán theo yêu cầu của \n'
             "'€ơ quan tài chính (bằng văn bản) đối với: các ° trường hợp quy định tại khoản 2\n"
             '\n'
             'Điều 7 Thông tư này. ¬ ——..\n'
             '\n'
             '2. Thường xuyên đôn đốc các đơn vị sử dụng ng ngân sách thực hiện đúng quy định về '
             'tạm ứng và thu hồi vốn tạm ứng sau khi phát hiện các khoản tạm ứng đã\n'
             '\n'
             '6. Tổ chức công tác kiểm soát, thanh toán theo quy trình. nghiệp vụ, thanh toán kịp '
             'thời, đầy đủ, thuận tiện cho đơn vị sử dụng ngân sách, đảm bảo đơn giản thủ tục hành '
             'chính và quản lý chặt chế vốn NSNN.'},
 {'id': 139,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương IV . ĐIÊU KHOẢN THỊ HÀNH',
               'article': 'Điều 9',
               'article_title': 'Trách nhiệm, quyền hạn của đơn vị sử dụng ngân sách',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 9. Trách nhiệm, quyền hạn của '
                                  'đơn vị sử dụng ngân sách',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 23,
               'chunk_part': '1/1'},
  'content': 'Điều 9. Trách nhiệm, quyền hạn của đơn vị sử dụng ngân sách\n'
             '\n'
             '1. Tuân thủ đầy đủ các quy định của Luật Ngân sách nhà nước, Luật Đầu tư công, Luật '
             'Đấu thầu, Luật Xây dựng và các quy định khác của pháp luật có liên quan trong quá. '
             'trinh quản lý, sử dụng kinh phí NSNN cấp và quy định tại ¡ Điều 17 Nghị định số '
             '11/2020/NĐ-CP.\n'
             '\n'
             '2. Đơn vị sử dụng ngân sách chịu trách nhiệm về tính chính xác, tính hợp pháp theo '
             'quy định của pháp luật đối với chứng từ chuyển tiền và các hồ SƠ kèm theo chứng từ '
             'chuyển tiền thuộc thủ tục hành chính gửi Kho bạc nhà nước được \n'
             'kẽ # \n'
             '\n'
             ' quy định tại các Nghị định của Chính phủ; đồng thời chịu trách nhiệm nghiệm thu '
             'khối lượng mua sắm, dịch vụ theo quy định của pháp luật.\n'
             '\n'
             '3. Đối với chứng từ chuyển tiền và các mẫu biểu ban hành kèm theo thuộc các thủ tục '
             'hành chính quy định tại các Nghị định của Chính phủ: Đơn vị Sử dụng ngân sách chịu '
             'trách nhiệm kê khai đảm bảo khớp đúng giữa chỉ tiết và tổng số; Đối với các khoản '
             'chỉ được kê khai trên Bảng thanh toán cho đối tượng thụ hưởng, đơn vị chịu trách '
             'nhiệm vẻ tính chính xác của các chỉ tiêu: Tên đối tượng thụ hưởng, tài khoản ngân '
             'hàng, xác định số tiền thực nhận cho từng đối tượng thụ hưởng sau khi đã trích trừ '
             'các khoản phải khấu trừ vào lương, đảm bảo đúng mức lương, phụ cấp, các khoản phải '
             'khấu trừ vào lương theo đúng quy định.\n'
             '\n'
             '4. Đối với chỉ mua sắm:\n'
             '\n'
             '4.1. Trường hợp không thực hiện mua sắm theo phương thức tập trung: Đơn vị sử dụng '
             'ngân sách chịu trách nhiệm về chất lượng, khối lượng, chủng loại, đơn giá đã thực '
             'hiện mua sắm, đảm bảo theo đúng tiêu chuẩn, định mức quy định tại các văn bản quy '
             'phạm pháp luật, văn bản ban hành tiêu chuẩn, định mức sử dụng máy móc, thiết bị '
             'chuyên dùng do cấp có thẩm quyền ban hành (đối với máy móc, thiết bị, ô tô chuyên '
             'dùng).\n'
             '\n'
             '4.2. Trường hợp tài sản, hàng hóa thuộc danh mục mua sắm tập trung: Đơn vị mua sắm '
             'tập trung và đơn vị sử dụng tài sản chịu. trách nhiệm mua sắm tập trung theo đúng '
             'danh mục và ì quy trình, thủ tục mua sắm theo đúng quy định của pháp luật; chịu '
             'trách nhiệm về chất lượng, khối lượng, chúng loại, đơn giá đã thực . hiện mua sắm, '
             'đảm bảo theo đúng tiêu chuẩn, định mức quy định tại các văn bản quy phạm pháp luật, '
             'văn bản ban hành tiêu chuẩn, định mức sử dụng máy móc, thiết bị chuyên dùng do cấp '
             'có thấm quyền ban hành (đối với máy móc, thiết bị, ô tô chuyên dùng) và đảm bảo hợp '
             'đồng mua sắm tài sản phù hợp với thỏa thuận khung (trường hợp mua sắm tập trưng theo '
             'cách thức ký thỏa thuận khung).\n'
             '\n'
             '5. Đối với kinh phí giao nhiệm vụ, đặt hàng, đấu thầu cung cấp sản phẩm, dịch vụ '
             'công sử dụng NSNN, đơn vị sử dụng ngân sách chịu trách nhiệm thực hiện theo đúng quy '
             'định tại Nghị định số 32/2019/NĐ-CP và quy định của các pháp luật hiện hành khác.\n'
             '\n'
             '6. Chịu trách nhiệm đảm bảo tính pháp lý, đầy đủ, chính xác và trung thực đối với hồ '
             'sơ, chứng từ điện tử và chữ ký số trên hồ sơ kiểm soát chỉ gửi Kho bạc Nhà nước qua '
             'Trang thông tin dịch vụ công trực tuyến của Kho bạc Nhà nước theo đúng quy định của '
             'Luật Giao dịch điện tử và các văn bản quy phạm pháp luật hiện hành. \n'
             '\n'
             'Chương IV . ĐIÊU KHOẢN THỊ HÀNH'},
 {'id': 140,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương IV . ĐIÊU KHOẢN THỊ HÀNH',
               'article': 'Điều 10',
               'article_title': 'Hiệu lực thi hành',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 10. Hiệu lực thi hành',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 23,
               'chunk_part': '1/1'},
  'content': 'Điều 10. Hiệu lực thi hành\n'
             '\n'
             '1. Thông tư này có hiệu lực từ ngày04 thángÓ 5 năm 2024.\n'
             '\n'
             '2. Trong quá trình thực hiện, nếu các văn bản quy phạm pháp luật được dẫn chiếu để '
             'áp dụng tại Thông tư này được sửa đổi, bỗ sung hoặc thay thế bằng văn bản mới thì sẽ '
             'áp dụng theo các văn bản sửa đỗi, bố sung hoặc thay thế đó.\n'
             '\n'
             '3. Thông tư này thay thế Thông tư số 62/2020/TT-BTC ngày 22 tháng 6 năm 2020 của Bộ '
             'Tài chính hướng dẫn kiểm soát, thanh toán các khoản chỉ thường xuyên từ Ngân sách '
             'Nhà nước qua Kho bạc Nhà nước.'},
 {'id': 141,
  'metadata': {'doc_code': 'Thông tư 17/2024/TT-NHNN',
               'doc_name': 'Thông tư quy định việc mở và sử dụng tài khoản thanh toán tại tổ chức '
                           'cung ứng dịch vụ thanh toán',
               'effective_date': '01/07/2024',
               'chapter': 'Chương IV . ĐIÊU KHOẢN THỊ HÀNH',
               'article': 'Điều 11',
               'article_title': 'Tổ chức thực hiện',
               'citation_header': 'Thông tư 17/2024/TT-NHNN - Điều 11. Tổ chức thực hiện',
               'topics': 'Hạn mức & Giao dịch',
               'is_core_fintech': True,
               'page': 25,
               'chunk_part': '1/1'},
  'content': 'Điều 11. Tổ chức thực hiện \n'
             'Các Bộ, cơ quan ngang Bộ, cơ quan thuộc Chính phủ, cơ quan khác ở trung ương, Ủy ban '
             'nhân dân các tỉnh, thành phô trực thuộc trung ương, Kho bạc Nhà nước có trách nhiệm '
             'hướng dẫn các đơn vị trực thuộc và cấp đưới tổ chức thự hiện, nếu có phát sinh vướng '
             'mắc, đề nghị Cơ quan, đơn vị, tổ chức, cá nhân phản ánh kịp thời về Bộ Tài chính để '
             'xem xét, phối hợp giải quyết./. \n'
             'Nơi nhận: KT. họ TRƯỞNG - Ban Bí thư Trung ương Đảng; : \n'
             '~ Thủ tướng, các Phó Thủ tướng Chính phủ; \n'
             '- Văn phòng Trung ương Đảng và các Ban của Đảng; \n'
             '- Văn phòng Tổng Bí thư; \n'
             '- Văn phòng Quốc hội, Hội đồng Dân tộc, Ủy ban Tài chính, Ngân sách; \n'
             '- Văn phòng Chủ tịch nước; \n'
             '- Văn phòng Chính phủ; \n'
             '~ Viện Kiểm sát nhân dân tối cao; \n'
             '- Tòa án nhân dân tối cao; \n'
             '- Kiểm toán nhà nước; \n'
             '- Các Bộ, cơ quan ngang Bộ, cơ quan thuộc Chính phủ; \n'
             '- HĐND, UBND các tỉnh, thành phố trực thuộc TW; \n'
             '- Cơ quan Trung ương của các Hội, Đoàn thể; \n'
             '- Sở Tài chính, Kho bạc Nhà nước các tỉnh, thành phố trực thuộc TW; \n'
             '- Công báo, Cổng Thông tin điện tử của Chính phủ; \n'
             '- Cục Kiểm tra văn bản quy phạm pháp luật, Bộ Tư pháp; - Công Thông tin điện tử của '
             'Bộ Tài chính; \n'
             '- Các đơn vị thuộc Bộ Tài chính; \n'
             '- Lưu: VT, KBNN (240 bản). ““ \n'
             '\n'
             ' Mẫu số 01 KHO BẠC NHÀ NƯỚC CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM \n'
             'KHO BẠC NHÀ NƯỚC..... Độc lập - Tự do - Hạnh phúc Số: /TB.Kn,,ọ— — Quy , ngày ... '
             'thẳng ... năm... THÔNG BÁO \n'
             'Về việc kết quã kiểm soát chỉ sau khi thực hiện theo hình thức “thanh toán trước, '
             'kiểm soát sau” \n'
             'Sau khi kiểm soát, Kho bạc Nhà nước.......... thông báo kết quả kiểm soát \n'
             '_chỉ như sau: \n'
             'Kho bạc Nhà nước.............. đã thanh toán số tiền............. (số tiền bằng chữ) '
             'cho khoản chỉ........ tại chứng từ SỐ........ ngày ..... tháng ..... năm ..... theo '
             'hình thức “thanh toán trước, kiểm soát sau”. Sau khi kiểm soát sau, Kho bạc Nhà '
             'nước...... phát hiện khoản chỉ chưa đúng quy định, Kho bạc Nhà nước.... đề nghị: \n'
             'p Giảm trừ giá trị thanh toán vào lần thanh toán liền kề tiếp, số tiền: ..... (số '
             'tiền bằng chữ........ ). \n'
             'Lý dO:....................... Hee, "....,:(4... 0 Thu hồi nộp NSNN số '
             'tiền:.......(số tiền bằng chữ........ }. \n'
             'ID. ..................... — Kho bạc Nhà nước ................--.- 2c ss se s52 xin '
             'thông báo cho đơn vị \n'
             'biết để có biện pháp xử lý./. \n'
             'Nơi nhận: GIÁM ĐÓC - Đơn vị...; (Ký, ghi rõ họ tên, đóng đấu) \n'
             '- Lưu: VT, viết tắt đơn vị soạn thảo (...bản). \n'
             ' Mẫu số 02 KHO BẠC NHÀ NƯỚC CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM \n'
             'KHO BẠC NHÀ NƯỚC..... Độc lập - Tự do - Hạnh phúc Số: /TB-KB... ...., Ngàp ... thẳng '
             '... năm... \n'
             'THÔNG BÁO Về việc từ chối thanh toán - Sau khi kiểm soát chí, Kho bạc Nhà nước '
             '............... thông báo kết quả kiêm soát chỉ như sau: : \n'
             'Kho bạc Nhà nước từ chối khoản chỉ.................... " tại chứng từ \n'
             'sỐ.....ngày.....tháng....năm..... - \n'
             'k0 0-11... ............. \n'
             '" \n'
             'Số tiền bằng chữ:......... an YAannnn \n'
             'Lý do: \n'
             'la \n'
             'Kho bạc Nhà nước ............................... xin thông báo cho đơn vị biết đê có '
             'biện pháp xử lý./. \n'
             'Nơi nhận: GIÁMĐÓC _ - Đơn vị...; (Ký. ghi rõ họ tên, đóng dâu) \n'
             '- Lưu: VT, viết tắt đơn vị soạn thảo (... bản).'}]


if __name__ == "__main__":
    print(f"Tổng số chunk: {len(CHUNKS)}")

Tổng số chunk: 141


In [ ]:
!pip install -q langchain-google-genai langchain-chroma chromadb

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [ ]:
import os
import time
from google.colab import userdata
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

# 1. Lấy API Key
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

# 2. Chuyển đổi CHUNKS sang chuẩn Document
formatted_docs = []
for i, item in enumerate(CHUNKS):
    if isinstance(item, str):
        formatted_docs.append(Document(page_content=item, metadata={"chunk_id": i}))
    elif isinstance(item, dict):
        content = item.get("page_content") or item.get("text") or item.get("content") or str(item)
        meta = item.get("metadata", {"chunk_id": i})
        formatted_docs.append(Document(page_content=content, metadata=meta))
    else:
        formatted_docs.append(item)

# 3. Khởi tạo Embedding
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# Dùng tên thư mục mới hoàn toàn để tránh bị khóa file SQLite cũ
DB_DIR = "./chroma_db_final"
vector_db = None

# 4. Nạp từng lô 10 chunks kèm cơ chế tự động nghỉ khi chạm trần API (Auto-retry)
batch_size = 10
total_docs = len(formatted_docs)
i = 0

while i < total_docs:
    batch = formatted_docs[i : i + batch_size]
    try:
        if vector_db is None:
            # Lô đầu tiên: Khởi tạo DB mới trực tiếp từ lô chunks đầu
            vector_db = Chroma.from_documents(
                documents=batch,
                embedding=embeddings,
                persist_directory=DB_DIR
            )
        else:
            # Các lô tiếp theo: Nạp nối tiếp vào DB đã tạo
            vector_db.add_documents(batch)

        i += batch_size
        print(f"✅ Đã nhúng xong {min(i, total_docs)}/{total_docs} chunks...")
        time.sleep(8)
    except Exception as e:
        if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
            print("⏳ Chạm giới hạn 100 requests/phút của Google. Tự động nghỉ 25 giây rồi chạy tiếp...")
            time.sleep(25)
        else:
            raise e

print(f"🎉 HOÀN TẤT 100%! Toàn bộ dữ liệu đã được lưu an toàn vào '{DB_DIR}'!")

✅ Đã nhúng xong 10/141 chunks...
✅ Đã nhúng xong 20/141 chunks...
✅ Đã nhúng xong 30/141 chunks...
✅ Đã nhúng xong 40/141 chunks...
✅ Đã nhúng xong 50/141 chunks...
✅ Đã nhúng xong 60/141 chunks...
✅ Đã nhúng xong 70/141 chunks...
✅ Đã nhúng xong 80/141 chunks...
✅ Đã nhúng xong 90/141 chunks...
✅ Đã nhúng xong 100/141 chunks...
✅ Đã nhúng xong 110/141 chunks...
✅ Đã nhúng xong 120/141 chunks...
✅ Đã nhúng xong 130/141 chunks...
⏳ Chạm giới hạn 100 requests/phút của Google. Tự động nghỉ 25 giây rồi chạy tiếp...
✅ Đã nhúng xong 140/141 chunks...
✅ Đã nhúng xong 141/141 chunks...
🎉 HOÀN TẤT 100%! Toàn bộ dữ liệu đã được lưu an toàn vào './chroma_db_final'!


In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

# 1. Đảm bảo API Key đã được nạp
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

def get_legal_retriever(db_dir="./chroma_db_final", k=4):
    """
    Khởi tạo bộ tìm kiếm (Retriever) từ ChromaDB đã lưu.
    - k: Số lượng đoạn luật (chunks) liên quan nhất muốn lấy ra.
    """
    embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

    vector_db = Chroma(
        persist_directory=db_dir,
        embedding_function=embeddings
    )

    # Dùng search_type="mmr" để lấy kết quả đa dạng, không bị trùng ý giữa các chunks
    retriever = vector_db.as_retriever(
        search_type="mmr",
        search_kwargs={
            "k": k,          # Trả về 4 đoạn tốt nhất
            "fetch_k": 12    # Quét 12 đoạn ứng viên trước rồi lọc lại 4 đoạn đa dạng nhất
        }
    )
    return retriever

def search_legal_context(query: str, retriever=None):
    """
    Hàm thực thi tìm kiếm theo câu hỏi của người dùng và trả về:
    - formatted_context: Chuỗi văn bản gộp sẵn để nhét vào Prompt cho Gemini.
    - raw_docs: Danh sách các Document gốc để hiển thị nguồn & băm SHA-256 trên UI.
    """
    if retriever is None:
        retriever = get_legal_retriever()

    raw_docs = retriever.invoke(query)

    context_parts = []
    for i, doc in enumerate(raw_docs, start=1):
        # Lấy thông tin nguồn từ metadata (nếu Thành viên A & B có gắn)
        source_name = doc.metadata.get("source", "Văn bản pháp luật NHNN")
        page_num = doc.metadata.get("page", doc.metadata.get("chunk_id", "N/A"))

        block = f"[Nguồn {i} | Tài liệu: {source_name} | Vị trí: {page_num}]\n{doc.page_content}"
        context_parts.append(block)

    formatted_context = "\n\n---\n\n".join(context_parts)
    return formatted_context, raw_docs

In [ ]:
import re
from typing import List, Dict, Any
from langchain_core.documents import Document

# ==============================================================================
# 1. SYSTEM PROMPT CHUẨN PHÁP CHẾ SME FINTECH (STRICT GROUNDING & 3 STATES)
# ==============================================================================
SYSTEM_PROMPT_FINTECH_LEGAL = """Bạn là **Chuyên viên Pháp chế Ngân hàng & Fintech (Fintech Legal Virtual Advisor)** chuyên tư vấn tuân thủ pháp luật cho Ban Giám đốc và Đội ngũ Sản phẩm (PO/BA) của các Startup/SME Fintech tại Việt Nam.

NHIỆM VỤ CỐT LÕI:
Đối soát tính hợp pháp của các tính năng, luồng nghiệp vụ (đặc biệt là Ví điện tử, eKYC, mở tài khoản thanh toán, hạn mức giao dịch) dựa DUY NHẤT trên các trích đoạn văn bản pháp luật được cung cấp trong phần `[DỮ LIỆU PHÁP LÝ CUNG CẤP]`.

NGUYÊN TẮC BẮT BUỘC (ZERO-HALLUCINATION POLICY):
1. **Tuyệt đối không suy diễn hay bịa đặt:** Chỉ sử dụng thông tin có trong `[DỮ LIỆU PHÁP LÝ CUNG CẤP]`. Không tự ý lấy kiến thức bên ngoài hoặc các văn bản hết hiệu lực nếu không có trong ngữ cảnh được cấp.
2. **Trích dẫn chính xác từng Điều, Khoản:** Mọi nhận định pháp lý BẮT BUỘC phải chỉ rõ **Tên văn bản + Điều X + Khoản Y (nếu có)** làm căn cứ.
3. **Không đủ dữ liệu thì dừng lại:** Nếu `[DỮ LIỆU PHÁP LÝ CUNG CẤP]` không chứa quy định trực tiếp trả lời cho tình huống của người dùng, hoặc tình huống thiếu thông số định lượng (ví dụ: không rõ giá trị giao dịch là bao nhiêu triệu đồng, đối tượng là cá nhân hay tổ chức, đã gặp mặt trực tiếp hay chưa), BẮT BUỘC phân loại là `CẦN XÁC MINH THÊM`.

QUY TẮC PHÂN LOẠI 3 TRẠNG THÁI ĐẦU RA:
Bạn phải chọn DUY NHẤT 1 trong 3 trạng thái sau ở ngay dòng đầu tiên của câu trả lời:
- **`[TRẠNG THÁI: HỢP LỆ]`**: Khi hành vi/tính năng dự kiến của SME Fintech tuân thủ đầy đủ các điều kiện, quy trình, hạn mức được quy định rõ trong tài liệu cung cấp.
- **`[TRẠNG THÁI: VI PHẠM]`**: Khi hành vi/tính năng dự kiến làm trái điều cấm, vượt hạn mức tối đa cho phép, hoặc bỏ qua bước xác thực bắt buộc được quy định rõ trong tài liệu cung cấp.
- **`[TRẠNG THÁI: CẦN XÁC MINH THÊM]`**: Khi tài liệu cung cấp không đề cập đến tình huống này, HOẶC câu hỏi của người dùng chưa cung cấp đủ điều kiện/số liệu để kết luận chắc chắn Hợp lệ hay Vi phạm.

CẤU TRÚC TRẢ LỜI BẮT BUỘC (TUÂN THỦ ĐÚNG ĐỊNH DẠNG MARKDOWN SAU):
**[TRẠNG THÁI: <HỢP LỆ | VI PHẠM | CẦN XÁC MINH THÊM>]**

### 1. Kết luận nhanh cho Đội ngũ Sản phẩm (PO/BA)
(Tóm tắt ngắn gọn 2-3 câu trả lời trực diện: Tính năng này có được phép triển khai hay không, hoặc cần bổ sung thông tin gì).

### 2. Căn cứ pháp lý đối soát
- **Văn bản áp dụng:** (Ghi rõ tên văn bản, số hiệu, Điều, Khoản, Điểm cụ thể).
- **Nội dung quy định cốt lõi:** (Trích xuất chính xác điều kiện/hạn mức/quy trình từ tài liệu).

### 3. Phân tích rủi ro & Khuyến nghị điều chỉnh nghiệp vụ
- (Giải thích vì sao tính năng khớp hoặc chạm trần quy định).
- (Đề xuất hành động cụ thể cho đội Dev/PO/BA: ví dụ cần thêm bước xác thực sinh trắc học, giới hạn hạn mức giao dịch, hoặc liệt kê các câu hỏi cần SME làm rõ nếu ở trạng thái CẦN XÁC MINH THÊM).
"""

# ==============================================================================
# 2. AI HARNESS: ĐỊNH DẠNG NGỮ CẢNH & KIỂM SOÁT ĐẦU RA (GUARDRAILS)
# ==============================================================================
def format_retrieved_context(retrieved_docs: List[Document]) -> str:
    """
    Chuyển danh sách Document từ ChromaDB (TV C) thành chuỗi ngữ cảnh có gắn nhãn nguồn rõ ràng
    để AI không nhầm lẫn giữa các văn bản luật.
    """
    if not retrieved_docs:
        return "KHÔNG TÌM THẤY ĐOẠN LUẬT NÀO PHÙ HỢP TRONG CƠ SỞ DỮ LIỆU."

    formatted_blocks = []
    for idx, doc in enumerate(retrieved_docs, start=1):
        meta = doc.metadata
        header = (
            f"--- TRÍCH ĐOẠN #{idx} | Văn bản: {meta.get('doc_code', 'N/A')} | "
            f"Vị trí: {meta.get('article', 'N/A')} ({meta.get('article_title', '')}) | "
            f"Trang PDF: {meta.get('page', 'N/A')} ---"
        )
        formatted_blocks.append(f"{header}\n{doc.page_content}")

    return "\n\n".join(formatted_blocks)


def build_full_prompt(user_question: str, retrieved_docs: List[Document]) -> str:
    """
    Ghép System Prompt + Ngữ cảnh Luật + Câu hỏi của SME thành Prompt hoàn chỉnh
    để Thành viên C gửi thẳng tới Gemini API.
    """
    context_str = format_retrieved_context(retrieved_docs)

    full_prompt = f"""{SYSTEM_PROMPT_FINTECH_LEGAL}

==============================================================================
[DỮ LIỆU PHÁP LÝ CUNG CẤP]
{context_str}
==============================================================================

[CÂU HỎI / KỊCH BẢN NGHIỆP VỤ CỦA SME FINTECH]
{user_question}

Hãy phân tích và xuất kết quả tuân thủ đúng 3 trạng thái và cấu trúc bắt buộc ở trên:"""
    return full_prompt


def validate_and_parse_ai_output(raw_response: str) -> Dict[str, Any]:
    """
    AI Harness hậu kiểm đầu ra của Gemini:
    - Tự động nhận diện 1 trong 3 trạng thái chuẩn để TV D render màu sắc trên UI.
    - Kiểm tra xem AI có quên trích dẫn Điều/Khoản hay không (cảnh báo nếu thiếu).
    """
    status = "CẦN XÁC MINH THÊM"
    badge_color = "orange"

    if re.search(r'\[TRẠNG THÁI:\s*HỢP LỆ\]', raw_response, re.IGNORECASE):
        status = "HỢP LỆ"
        badge_color = "green"
    elif re.search(r'\[TRẠNG THÁI:\s*VI PHẠM\]', raw_response, re.IGNORECASE):
        status = "VI PHẠM"
        badge_color = "red"
    elif re.search(r'\[TRẠNG THÁI:\s*CẦN XÁC MINH THÊM\]', raw_response, re.IGNORECASE):
        status = "CẦN XÁC MINH THÊM"
        badge_color = "orange"
    else:
        # Fallback an toàn nếu AI quên gắn nhãn ở dòng đầu
        raw_response = "**[TRẠNG THÁI: CẦN XÁC MINH THÊM]**\n\n" + raw_response

    # Kiểm tra AI có trích dẫn số "Điều" hay không
    has_citation = bool(re.search(r'Điều\s+\d+', raw_response))
    if not has_citation and status in ["HỢP LỆ", "VI PHẠM"]:
        raw_response += (
            "\n\n> ⚠️ **Cảnh báo kiểm soát AI Harness:** Câu trả lời chưa nêu rõ số Điều luật cụ thể. "
            "Vui lòng đối chiếu lại văn bản thô bên dưới trước khi áp dụng."
        )

    return {
        "status": status,               # "HỢP LỆ" | "VI PHẠM" | "CẦN XÁC MINH THÊM"
        "badge_color": badge_color,     # "green" | "red" | "orange" (cho UI của TV D)
        "has_legal_citation": has_citation,
        "formatted_answer": raw_response
    }

In [ ]:
import time
import uuid
import hashlib
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# ==============================================================================
# 1. LOGIC VÍ WEB3 (x402 PAYMENT) & MÃ BĂM ĐỐI SOÁT SHA-256
# ==============================================================================
def init_wallet(initial_balance: int = 10) -> dict:
    """Tạo địa chỉ ví Web3 giả lập và cấp số dư token CMP ban đầu."""
    seed = f"{uuid.uuid4()}-{time.time()}".encode("utf-8")
    return {
        "address": "0x" + hashlib.sha256(seed).hexdigest()[:40],
        "balance": initial_balance,
        "currency": "CMP"
    }

def charge_x402_token(wallet: dict, query: str, cost: int = 1) -> dict:
    """Trừ 1 token mỗi lượt hỏi và sinh mã giao dịch tx_hash."""
    if wallet["balance"] < cost:
        return {"success": False, "error": "402 Payment Required: Số dư CMP không đủ!"}

    wallet["balance"] -= cost
    tx_hash = "0x" + hashlib.sha256(f"{wallet['address']}|{query}|{time.time()}".encode()).hexdigest()
    return {
        "success": True,
        "tx_hash": tx_hash,
        "deducted": cost,
        "remaining_balance": wallet["balance"]
    }

def compute_sha256_audit(query: str, answer: str, tx_hash: str) -> str:
    """Tính mã băm SHA-256 niêm phong bằng chứng tư vấn pháp lý."""
    payload = f"TX:{tx_hash}|Q:{query}|A:{answer}".encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


# ==============================================================================
# 2. LẮP RÁP RAG CHAIN HOÀN CHỈNH (NỐI HÀM CỦA TV B & TV C + GEMINI)
# ==============================================================================
# Khởi tạo Retriever từ hàm của bạn và LLM Gemini
legal_retriever = get_legal_retriever(db_dir="./chroma_db_final", k=4)
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0.1)

# Hàm cầu nối: Nhận câu hỏi -> Tìm luật bằng Retriever -> Ghép Prompt chuẩn của TV B
def prepare_prompt_step(user_question: str) -> str:
    retrieved_docs = legal_retriever.invoke(user_question)
    return build_full_prompt(user_question, retrieved_docs)

# Lắp thành RAG Chain hoàn chỉnh bằng LCEL (LangChain Expression Language)
rag_chain = (
    RunnableLambda(prepare_prompt_step)
    | llm
    | StrOutputParser()
    | RunnableLambda(validate_and_parse_ai_output)
)


# ==============================================================================
# 3. HÀM CHẠY TRỌN VẸN 1 PHIÊN HỎI ĐÁP (DÙNG CHO CẢ TEST & UI THÀNH VIÊN D)
# ==============================================================================
def run_compliance_query(user_question: str, wallet: dict) -> dict:
    # Bước 1: Kiểm tra & trừ 1 Token x402
    payment = charge_x402_token(wallet, user_question, cost=1)
    if not payment["success"]:
        return {"success": False, "error": payment["error"]}

    # Bước 2: Lấy nguồn luật thô để hiển thị trên Expander của UI
    raw_docs = legal_retriever.invoke(user_question)

    # Bước 3: Chạy RAG Chain (Prompt TV B -> Gemini -> Hậu kiểm AI Harness)
    ai_result = rag_chain.invoke(user_question)

    # Bước 4: Tính mã băm SHA-256 đối soát
    audit_hash = compute_sha256_audit(
        query=user_question,
        answer=ai_result["formatted_answer"],
        tx_hash=payment["tx_hash"]
    )

    return {
        "success": True,
        "status": ai_result["status"],               # "HỢP LỆ" | "VI PHẠM" | "CẦN XÁC MINH THÊM"
        "badge_color": ai_result["badge_color"],     # "green" | "red" | "orange"
        "answer": ai_result["formatted_answer"],     # Nội dung Markdown đã format
        "sources": raw_docs,                         # Danh sách Document gốc cho UI
        "tx_hash": payment["tx_hash"],               # Mã giao dịch ví x402
        "remaining_balance": payment["remaining_balance"],
        "sha256_hash": audit_hash                    # Mã băm đối soát SHA-256
    }

In [ ]:
!pip install -q streamlit
# -*- coding: utf-8 -*-
"""
app.py - UI Streamlit (Thành viên D) - Phần 3: Task 3.1, 3.2, 3.3

Chạy:  streamlit run app.py

Yêu cầu: file rag_backend.py chứa code của TV B + TV C (xem hướng dẫn trong phần trả lời):
    init_wallet, run_compliance_query, compute_sha256_audit
Nếu chưa có rag_backend.py, app tự chạy ở chế độ DEMO (dữ liệu giả, cùng cấu trúc trả về)
để D vẫn test được giao diện độc lập.
"""
import hashlib
import time
import uuid
from types import SimpleNamespace

import streamlit as st

# ------------------------------------------------------------------------------
# CẤU HÌNH x402 (khớp với backend của TV C: ban đầu 10 CMP, mỗi lượt hỏi trừ 1 CMP)
# ------------------------------------------------------------------------------
INITIAL_BALANCE = 10
COST_PER_QUERY = 1          # chỉ để hiển thị & kiểm tra trước; việc TRỪ TIỀN do backend C thực hiện
TOPUP_AMOUNT = 10

# ------------------------------------------------------------------------------
# KẾT NỐI BACKEND TV C (fallback DEMO nếu chưa có)
# ------------------------------------------------------------------------------
try:
    from rag_backend import init_wallet, run_compliance_query, compute_sha256_audit
    DEMO_MODE = False
except ImportError:
    DEMO_MODE = True

    def init_wallet(initial_balance: int = 10) -> dict:
        seed = f"{uuid.uuid4()}-{time.time()}".encode("utf-8")
        return {"address": "0x" + hashlib.sha256(seed).hexdigest()[:40],
                "balance": initial_balance, "currency": "CMP"}

    def compute_sha256_audit(query: str, answer: str, tx_hash: str) -> str:
        return hashlib.sha256(f"TX:{tx_hash}|Q:{query}|A:{answer}".encode("utf-8")).hexdigest()

    def run_compliance_query(user_question: str, wallet: dict) -> dict:
        if wallet["balance"] < COST_PER_QUERY:
            return {"success": False, "error": "402 Payment Required: Số dư CMP không đủ!"}
        wallet["balance"] -= COST_PER_QUERY
        time.sleep(1)
        tx_hash = "0x" + hashlib.sha256(
            f"{wallet['address']}|{user_question}|{time.time()}".encode()).hexdigest()
        answer = ("**[TRẠNG THÁI: CẦN XÁC MINH THÊM]**\n\n(CHẾ ĐỘ DEMO - chưa kết nối RAG thật)\n\n"
                  "Theo Nghị định 52/2024/NĐ-CP Điều 8, nghiêm cấm mở/duy trì ví điện tử nặc danh, mạo danh.")
        doc = SimpleNamespace(
            page_content="Điều 8. Các hành vi bị cấm\n5. Mở hoặc duy trì tài khoản thanh toán, ví điện tử nặc danh, mạo danh...",
            metadata={"doc_code": "Nghị định 52/2024/NĐ-CP", "article": "Điều 8",
                      "article_title": "Các hành vi bị cấm", "page": 7})
        return {"success": True, "status": "CẦN XÁC MINH THÊM", "badge_color": "orange",
                "answer": answer, "sources": [doc], "tx_hash": tx_hash,
                "remaining_balance": wallet["balance"],
                "sha256_hash": compute_sha256_audit(user_question, answer, tx_hash)}


# ------------------------------------------------------------------------------
# TASK 3.1 - CẤU HÌNH TRANG
# ------------------------------------------------------------------------------
st.set_page_config(
    page_title="Hệ thống Tư vấn Pháp luật Tài chính - BKFintech",
    page_icon="⚖️",
    layout="wide",
)

# ------------------------------------------------------------------------------
# TASK 3.2 - KHỞI TẠO SESSION STATE
# ------------------------------------------------------------------------------
if "messages" not in st.session_state:
    st.session_state.messages = []
if "wallet" not in st.session_state:
    # Ví Web3 giả lập do backend C tạo (có địa chỉ ví + số dư). Lưu nguyên dict trong session
    # để run_compliance_query() trừ token trực tiếp trên cùng một ví.
    st.session_state.wallet = init_wallet(INITIAL_BALANCE)

wallet = st.session_state.wallet


# ------------------------------------------------------------------------------
# TASK 3.3 - HIỂN THỊ TRÍCH DẪN LUẬT & BLOCKCHAIN PROOF
# ------------------------------------------------------------------------------
def serialize_sources(docs) -> list:
    """Chuyển Document của LangChain thành dict thuần để lưu an toàn trong session_state."""
    out = []
    for d in docs or []:
        m = getattr(d, "metadata", {}) or {}
        out.append({
            "doc_code": m.get("doc_code", m.get("source", "N/A")),
            "article": m.get("article", "N/A"),
            "article_title": m.get("article_title", ""),
            "page": m.get("page", "N/A"),
            "content": d.page_content,
        })
    return out


def render_status(status: str):
    """Hiển thị 1 trong 3 trạng thái chuẩn của AI Harness (TV B)."""
    if status == "HỢP LỆ":
        st.success(f"✅ {status}")
    elif status == "VI PHẠM":
        st.error(f"⛔ {status}")
    else:
        st.warning(f"⚠️ {status}")


def render_proof(proof: dict):
    """Expander: văn bản luật thô + SHA-256 + TxHash + đối soát tính toàn vẹn."""
    with st.expander("🔍 Chi tiết Trích dẫn Luật & Blockchain Proof (Đối soát)"):
        st.markdown("**📖 Văn bản pháp luật thô trích xuất từ ChromaDB:**")
        if not proof["sources"]:
            st.info("Không tìm thấy đoạn luật phù hợp trong cơ sở dữ liệu.")
        for i, s in enumerate(proof["sources"], start=1):
            title = f" ({s['article_title']})" if s["article_title"] else ""
            st.markdown(f"**#{i} · {s['doc_code']} · {s['article']}{title} · Trang PDF: {s['page']}**")
            st.text(s["content"])
            st.divider()

        st.markdown("**🔐 Mã đối soát:**")
        st.code(f"SHA-256 Hash: {proof['sha256_hash']}", language="text")
        st.code(f"TxHash (Blockchain Proof): {proof['tx_hash']}", language="text")

        # Đối soát: tính lại SHA-256 từ (tx_hash, câu hỏi, câu trả lời) và so với mã backend đã niêm phong
        recomputed = compute_sha256_audit(proof["query"], proof["answer"], proof["tx_hash"])
        if recomputed == proof["sha256_hash"]:
            st.success("✅ Đối soát thành công: nội dung tư vấn khớp với mã SHA-256 đã niêm phong.")
        else:
            st.error("❌ Đối soát thất bại: nội dung đã bị thay đổi so với mã SHA-256 gốc!")


def render_message(msg: dict):
    with st.chat_message(msg["role"]):
        if msg.get("status"):
            render_status(msg["status"])
        if msg.get("is_error"):
            st.error(msg["content"])
        else:
            st.markdown(msg["content"])
        if msg["role"] == "assistant" and msg.get("proof"):
            render_proof(msg["proof"])


# ------------------------------------------------------------------------------
# TASK 3.1 - SIDEBAR: VÍ TOKEN
# ------------------------------------------------------------------------------
with st.sidebar:
    st.markdown("## 💳 x402 Flow & Wallet")
    st.caption("Địa chỉ ví (Web3 giả lập)")
    st.code(wallet["address"], language="text")
    st.metric(label="Số dư Token hiện tại", value=f"{wallet['balance']} {wallet.get('currency', 'CMP')}")
    st.caption(f"Phí mỗi lượt truy vấn: **{COST_PER_QUERY} CMP** · "
               f"Còn **{wallet['balance'] // COST_PER_QUERY}** lượt hỏi")

    if wallet["balance"] < COST_PER_QUERY:
        st.error("402 Payment Required: ví đã hết token, vui lòng nạp thêm.")

    if st.button(f"🔄 Nạp thêm {TOPUP_AMOUNT} CMP (Test)"):
        wallet["balance"] += TOPUP_AMOUNT
        st.rerun()

    if st.button("🗑️ Xoá lịch sử chat"):
        st.session_state.messages = []
        st.rerun()

    st.markdown("---")
    if DEMO_MODE:
        st.warning("Đang chạy **chế độ DEMO** (không tìm thấy `rag_backend.py`).")
    st.info(
        "**Phần 3: Thiết kế UI (Thành viên D)**\n\n"
        "- **3.1:** Bố cục Streamlit & Ví Token.\n"
        "- **3.2:** `session_state` & x402 Flow (gọi RAG Backend của TV C).\n"
        "- **3.3:** Trích dẫn luật thô, SHA-256 & TxHash đối soát."
    )

# ------------------------------------------------------------------------------
# TASK 3.1 - GIAO DIỆN CHÍNH
# ------------------------------------------------------------------------------
st.title("🏛️ AI Fintech & Legal Assistant - BKFintech Hackday")
st.caption("Hệ thống hỏi đáp pháp luật tài chính tích hợp RAG, x402 Flow và Xác thực Blockchain Proof")

for message in st.session_state.messages:
    render_message(message)

# ------------------------------------------------------------------------------
# TASK 3.2 - x402 FLOW: KIỂM TRA SỐ DƯ -> GỌI RAG BACKEND (TV C) -> LƯU STATE
# ------------------------------------------------------------------------------
if prompt := st.chat_input("Nhập câu hỏi về Luật TCTD, Nghị định 52, Thông tư 17..."):
    # Bước 1: kiểm tra số dư trước (HTTP 402 Payment Required)
    if wallet["balance"] < COST_PER_QUERY:
        st.error(f"❌ 402 Payment Required: cần tối thiểu {COST_PER_QUERY} CMP để truy vấn. "
                 "Vui lòng nạp thêm ở thanh bên.")
    else:
        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("user"):
            st.markdown(prompt)

        # Bước 2: gọi backend C. Backend tự trừ token + sinh tx_hash + sha256 (D KHÔNG trừ lần nữa)
        balance_before = wallet["balance"]
        with st.chat_message("assistant"):
            with st.spinner("Đang truy vấn VectorDB (ChromaDB) & xác thực đối soát giao dịch..."):
                try:
                    result = run_compliance_query(prompt, wallet)
                except Exception as e:  # lỗi API Gemini / ChromaDB...
                    wallet["balance"] = balance_before   # hoàn token nếu truy vấn lỗi
                    result = {"success": False,
                              "error": f"Lỗi hệ thống RAG: {e}. Token đã được hoàn lại."}

        # Bước 3: lưu kết quả vào session_state
        if result.get("success"):
            st.session_state.messages.append({
                "role": "assistant",
                "content": result["answer"],
                "status": result["status"],
                "proof": {
                    "query": prompt,
                    "answer": result["answer"],
                    "sources": serialize_sources(result["sources"]),
                    "sha256_hash": result["sha256_hash"],
                    "tx_hash": result["tx_hash"],
                },
            })
        else:
            st.session_state.messages.append({
                "role": "assistant",
                "content": f"⚠️ {result.get('error', 'Có lỗi xảy ra.')}",
                "is_error": True,
            })

        # Cập nhật lại sidebar (số dư mới) và render tin nhắn mới từ session_state
        st.rerun()

2026-11-19 13:52:27.605 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-19 13:52:27.607 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-19 13:52:27.611 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2026-11-19 13:52:27.616 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-19 13:52:27.618 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-19 13:52:27.623 WARNING streamlit.runtime.scriptrunner_utils.script_run_c